In [ ]:
# E23 CPU kit: the three GPU notebooks against a fake vLLM server, real /kaggle/input mounts, Linux semantics
import json, os, re, shutil, subprocess, sys, tarfile, time, traceback
from pathlib import Path
T0 = time.time()
W, T = Path("/kaggle/working"), Path("/tmp/e23kit")
T.mkdir(parents=True, exist_ok=True)
NBF = '# ==================================================================== CELL 1 — config, inputs, lab module\nimport os, sys, json, time, re, shutil, subprocess, hashlib, tarfile, threading, traceback\nfrom pathlib import Path\nT_START = time.time()\nMODE = os.environ.get("Q27X_E23_MODE", "FULL").upper()        # SMOKE | TARGET | FULL (baked per notebook)\nPART = os.environ.get("Q27X_E23_PART", "ALL").upper()        # SMOKE / TARGET: A (Merlin) | B (stock) | ALL\nassert MODE in ("SMOKE", "TARGET", "FULL") and PART in ("A", "B", "ALL"), (MODE, PART)\nRERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}\nFAKE = os.environ.get("E23_FAKE_SERVER")                          # local CPU harness only\n# rev 3h: TARGET = the targeted GPU validation of the three GPT SMOKE-audit fixes before the 690-min FULL\nBUDGET = int(os.environ.get("E23_BUDGET_SEC") or {"SMOKE": {"A": 105, "B": 135, "ALL": 240}[PART] * 60,\n                                                  "TARGET": {"A": 120, "B": 120, "ALL": 240}[PART] * 60, "FULL": 690 * 60}[MODE])\nHARD_STOP = T_START + BUDGET\nMAX_LEN = 131072                                                  # user 2026-10-29: thinking up to 120k\nOUT = Path(os.environ.get("E23_OUT", "/kaggle/working/e23"))\nOUT.mkdir(parents=True, exist_ok=True)\nLOGS = OUT / "logs"\nLOGS.mkdir(exist_ok=True)\nTMP = Path(os.environ.get("E23_TMP", "/tmp"))\nINP = Path(os.environ.get("E23_INPUT", "/kaggle/input"))\nR = {"mode": MODE, "part": PART, "budget_min": BUDGET // 60, "max_len": MAX_LEN, "rev": "3h",\n     "start_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "setup": {}, "sections": {}, "errors": [], "order": []}\n_SAVE_LOCK = threading.Lock()\n\n\ndef save():\n    with _SAVE_LOCK:\n        tmp = OUT / "e23_results.json.tmp"\n        tmp.write_text(json.dumps(R, indent=1, default=str))\n        tmp.replace(OUT / "e23_results.json")\n\n\ndef left():\n    return HARD_STOP - time.time()\n\n\ndef log(*a):\n    print(f"[{(time.time() - T_START) / 60:6.1f} min]", *a, flush=True)\n\n\ndef find(pattern, root=INP, must=True):\n    hits = sorted(Path(root).rglob(pattern)) if Path(root).exists() else []\n    if not hits and must:\n        raise FileNotFoundError(f"{pattern} not under {root}: attach the input")\n    return hits[0] if hits else None\n\n\nif RERUN:      # rev 3: a competition rerun writes the minimal valid submission and skips cells 2-6 (no exception)\n    tasks = json.load(open(find("arc-agi_test_challenges.json")))\n    sub = {k: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in v["test"]] for k, v in tasks.items()}\n    Path("/kaggle/working/submission.json").write_text(json.dumps(sub))\n    print("E23 lab notebook: competition rerun -> fallback submission.json only")\nelse:\n    LAB_SRC = \'"""E23 lab, rev 3: port-parameterized server lifecycle and measurement for the six-lever notebook (stdlib only).\\n\\nBase: the Exp 14 VMS harness (VServer / VGen: token ids from return_token_ids, [DONE] + finish_reason required, socket\\nshutdown on stop, top-1/top-2 margins, garbage checks on every window). Adopted from the consolidated v3 notebook:\\nsame-weight greedy oracle classified by the oracle\\\'s logprob margin at the first differing token, a steady decode window\\nthat starts only after prefill settled, a GPU-release gate, fail-closed statuses, checkpoint identity, a runtime\\n--help flag check, KV + GPU memory samples.\\nrev 3 (validation agents 2026-10-29, measurement + robustness):\\n  - port check by connect (a TIME_WAIT socket of the previous server is not "busy"), with a 90 s wait;\\n  - kill(): SIGKILL the whole session after SIGTERM, even when the leader already exited;\\n  - steady(): idle gate at entry, admission cap from the prefill to do (700 tok/s assumed), settle needs no preemption\\n    and every running stream gaining tokens; in the window: stalled streams, a changing running/waiting set (KV-bound),\\n    preemptions and prefills make it INCONCLUSIVE (numbers kept); garbage under load fails it; steps from the\\n    vllm:iteration_tokens_total histogram count; replicas measured in one shared window (barrier);\\n  - profiled_load(): settle as steady(), bounded profile (delay / max iterations), with_stack off, frontend ignored;\\n  - trace_summary(): categories on the kernel base name (attention before GDN: "fla" is inside "flashinfer"), per-forward\\n    graph / kernel launch counts inside the gpu_model_runner forward ranges (VLLM_CUSTOM_SCOPES_FOR_PROFILING=1) and the\\n    CUPTI correlation fallback -> FULL / PIECEWISE / EAGER / UNKNOWN; startup cudagraph lines parsed explicitly;\\n  - teacher_forced(): prompt logprobs of the last N positions of a long prompt (NVFP4 vs FP8 KV at 40k / 90k);\\n  - logprobs keyed by token id (return_tokens_as_token_ids); stop() race closed; compare() flags garbage only when the\\n    oracle itself is clean.\\n"""\\nimport gzip\\nimport hashlib\\nimport json\\nimport os\\nimport re\\nimport signal\\nimport socket\\nimport statistics\\nimport subprocess\\nimport threading\\nimport time\\nimport urllib.request as U\\nfrom pathlib import Path\\n\\nFAIL_MARGIN = 0.5          # first differing token: oracle top-1 minus top-2 logprob at or above this = real divergence\\nLIVE = set()               # every running Server, for the hard-stop watchdog\\n\\n\\n# ------------------------------------------------------------------ server lifecycle\\ndef port_listening(port):\\n    """True when something accepts connections on 127.0.0.1:port (TIME_WAIT leftovers do not)."""\\n    s = socket.socket()\\n    s.settimeout(1.0)\\n    try:\\n        s.connect(("127.0.0.1", int(port)))\\n        return True\\n    except OSError:\\n        return False\\n    finally:\\n        s.close()\\n\\n\\nclass Server:\\n    def __init__(self, name, argv, env, port, log_dir, gpus="0,1,2,3"):\\n        self.name, self.argv, self.env, self.port = name, [str(a) for a in argv], dict(env), int(port)\\n        self.gpus = str(gpus)\\n        self.log_path = Path(log_dir) / f"server_{name}.log"\\n        self.proc, self.t0, self.ready_sec = None, None, None\\n\\n    @property\\n    def url(self):\\n        return f"http://127.0.0.1:{self.port}"\\n\\n    def start(self):\\n        end = time.time() + 90\\n        while port_listening(self.port):\\n            if time.time() > end:\\n                raise RuntimeError(f"PORT_BUSY {self.port} (a listener is still there)")\\n            time.sleep(3)\\n        self.log_path.parent.mkdir(parents=True, exist_ok=True)\\n        self._fh = open(self.log_path, "w", encoding="utf-8")\\n        self.t0 = time.perf_counter()\\n        self.proc = subprocess.Popen(self.argv, env=self.env, stdout=self._fh, stderr=subprocess.STDOUT,\\n                                     start_new_session=True, cwd="/tmp" if os.path.isdir("/tmp") else None)\\n        LIVE.add(self)\\n        return self\\n\\n    def alive(self):\\n        return self.proc is not None and self.proc.poll() is None\\n\\n    def log(self):\\n        try:\\n            return self.log_path.read_text(encoding="utf-8", errors="replace")\\n        except Exception:\\n            return ""\\n\\n    def rss_mb(self):\\n        """Host RAM of the server\\\'s whole session (engine + workers), MB; None off Linux."""\\n        try:\\n            r = subprocess.run(["ps", "-o", "rss=", "--sid", str(self.proc.pid)], capture_output=True, text=True, timeout=20)\\n            return round(sum(int(x) for x in r.stdout.split() if x.isdigit()) / 1024, 1)\\n        except Exception:\\n            return None\\n\\n    def wait_ready(self, timeout):\\n        while time.perf_counter() - self.t0 < timeout:\\n            if not self.alive():\\n                return False, f"exited rc={self.proc.returncode}"\\n            try:\\n                U.urlopen(self.url + "/health", timeout=10).read()\\n                self.ready_sec = round(time.perf_counter() - self.t0, 1)\\n                return True, f"ready in {self.ready_sec}s"\\n            except Exception:\\n                time.sleep(3)\\n        return False, "startup timeout"\\n\\n    def kill(self):\\n        if self.proc is None:\\n            return\\n        pg = self.proc.pid\\n        try:\\n            os.killpg(pg, signal.SIGTERM) if hasattr(os, "killpg") else self.proc.terminate()\\n        except Exception:\\n            pass\\n        try:\\n            self.proc.wait(timeout=40)\\n        except Exception:\\n            pass\\n        # rev 3: the leader may exit while EngineCore / workers of the session still hold GPUs: always SIGKILL the group\\n        try:\\n            os.killpg(pg, signal.SIGKILL) if hasattr(os, "killpg") else self.proc.kill()\\n        except Exception:\\n            pass\\n        try:\\n            self.proc.wait(timeout=30)\\n        except Exception:\\n            pass\\n        try:\\n            self._fh.close()\\n        except Exception:\\n            pass\\n        LIVE.discard(self)\\n\\n    def facts(self):\\n        """What vLLM logged at startup: KV pool, max concurrency, selected kernels, graph mode, backend, KV dtype."""\\n        t = self.log()\\n        kv = [int(x.replace(",", "")) for x in re.findall(r"GPU KV cache size: ([\\\\d,]+) tokens", t)]\\n        mc = re.findall(r"Maximum concurrency for [\\\\d,]+ tokens per request: ([\\\\d.]+)x", t)\\n        kern = sorted(set(re.findall(r"Selected (\\\\w+Kernel) for", t)))\\n        errs = [l[:300] for l in t.splitlines() if re.search(r"Traceback|CUDA error|illegal memory|out of memory|"\\n                                                               r"RuntimeError|AssertionError", l)][:12]\\n        pick = lambda pat, n=8: [l.strip()[:240] for l in t.splitlines() if re.search(pat, l, re.I)][:n]\\n        # rev 3 (measurement H5): the decisive startup evidence for the graph mode actually used with spec decode\\n        downgrade = pick(r"not supported with spec-decode.*setting cudagraph_mode|setting cudagraph_mode\\\\s*=|"\\n                         r"[Oo]verriding cudagraph_mode to", 4)       # rev 3e: + the 0.32 wording\\n        capture = sorted(set(re.findall(r"Capturing CUDA graphs \\\\(([^)]+)\\\\)", t)))\\n        configured = sorted(set(re.findall(r"CUDAGraphMode\\\\.(\\\\w+)", t)))\\n        return {"kv_tokens": min(kv) if kv else None, "kv_tokens_all": kv, "max_concurrency": float(mc[-1]) if mc else None,\\n                "kernels": kern, "error_lines": errs, "ready_sec": self.ready_sec,\\n                "graph_capture_modes": capture, "graph_downgrade_lines": downgrade, "cudagraph_modes_logged": configured,\\n                "backend_lines": pick(r"attention backend|Using \\\\w+ backend|backend=|AttentionBackend"),\\n                "kv_dtype_lines": pick(r"kv_cache_dtype|kv cache dtype|nvfp4"),\\n                "host_embed_lines": pick(r"host[- _]?(resident|embed)|HOST_EMBED|embed.*(cpu|host)", 6)}\\n\\n\\ndef gpu_pids(gpus=None):\\n    """Compute PIDs on all GPUs, or only on the GPU indices in `gpus` (a bench on another GPU must not fail the release\\n    gate of a server that used GPU 0 only). None when nvidia-smi is unavailable."""\\n    try:\\n        a = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,gpu_uuid", "--format=csv,noheader,nounits"],\\n                           capture_output=True, text=True, timeout=30)\\n        g = subprocess.run(["nvidia-smi", "--query-gpu=index,uuid", "--format=csv,noheader,nounits"],\\n                           capture_output=True, text=True, timeout=30)\\n    except Exception:\\n        return None\\n    if a.returncode or g.returncode:\\n        return None\\n    idx = {u.strip(): int(i) for i, u in (l.split(",", 1) for l in g.stdout.strip().splitlines() if "," in l)}\\n    want = None if gpus is None else {int(x) for x in str(gpus).split(",") if str(x).strip() != ""}\\n    out = set()\\n    for l in a.stdout.strip().splitlines():\\n        if "," not in l:\\n            continue\\n        pid, uuid = (x.strip() for x in l.split(",", 1))\\n        if pid.isdigit() and (want is None or idx.get(uuid) in want):\\n            out.add(int(pid))\\n    return sorted(out)\\n\\n\\ndef gpu_mem():\\n    try:\\n        r = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"],\\n                           capture_output=True, text=True, timeout=30)\\n        return [[int(v) for v in l.split(",")] for l in r.stdout.strip().splitlines() if l.strip()]\\n    except Exception:\\n        return None\\n\\n\\ndef wait_gpu_release(timeout=180, fake=False, gpus=None, free_mib=1024):\\n    """True when the GPUs in `gpus` (all when None) hold no compute PID AND use < free_mib MiB each (rev 3: compute-app\\n    PIDs are often invisible inside Kaggle containers; memory.used is the proven E18 gate); None without nvidia-smi."""\\n    if fake:\\n        return True\\n    want = None if gpus is None else {int(x) for x in str(gpus).split(",") if str(x).strip() != ""}\\n    end = time.time() + timeout\\n    while time.time() < end:\\n        p, mem = gpu_pids(gpus), gpu_mem()\\n        if p is None or mem is None:\\n            return None\\n        used = [m[1] for m in mem if want is None or m[0] in want]\\n        if not p and all(u < free_mib for u in used):\\n            return True\\n        time.sleep(3)\\n    return False\\n\\n\\ndef identity(model_dir):\\n    """Checkpoint identity: sha256 of config / tokenizer / index files (an oracle is only compared with the same weights)."""\\n    root = Path(model_dir)\\n    files = [p for n in ("config.json", "tokenizer.json", "tokenizer_config.json", "generation_config.json")\\n             for p in [root / n] if p.is_file()] + sorted(root.glob("*.safetensors.index.json"))\\n    h = hashlib.sha256()\\n    for p in files:\\n        h.update(p.name.encode())\\n        h.update(hashlib.sha256(p.read_bytes()).digest())\\n    return {"dir": str(root), "files": [p.name for p in files], "sha256": h.hexdigest()[:16]}\\n\\n\\n_HELP = {}\\n\\n\\ndef help_check(key, argv_prefix, env, flags, timeout=300):\\n    """The engine\\\'s own `serve --help=all` (run once per engine `key`, cached) must list every flag of THIS launch\\n    (rev 3: every launch is checked, not only the first); returns the missing ones."""\\n    if key not in _HELP:\\n        try:\\n            r = subprocess.run(list(argv_prefix) + ["--help=all"], capture_output=True, text=True, timeout=timeout, env=env)\\n            _HELP[key] = r.stdout + r.stderr\\n        except Exception as e:\\n            _HELP[key] = "ERROR " + repr(e)[:200]\\n    txt = _HELP[key]\\n    if len(txt) < 2000:\\n        return {"ran": False, "error": "help output too short or failed", "tail": txt[-400:]}\\n    want = sorted({f for f in flags if isinstance(f, str) and f.startswith("--")})\\n    return {"ran": True, "missing": [f for f in want if f not in txt]}\\n\\n\\ndef start_watchdog(hard_stop_epoch, on_fire):\\n    """A real hard stop: at `hard_stop_epoch` every live Server is killed and on_fire() is called."""\\n    def run():\\n        while time.time() < hard_stop_epoch:\\n            time.sleep(5)\\n        try:\\n            on_fire()\\n        finally:\\n            for s in list(LIVE):\\n                try:\\n                    s.kill()\\n                except Exception:\\n                    pass\\n    t = threading.Thread(target=run, daemon=True, name="e23-watchdog")\\n    t.start()\\n    return t\\n\\n\\ndef prefetch(paths, threads=16, chunk=16 << 20):\\n    """Read files once with many readers (page cache warm-up of a cold /kaggle/input mount); returns bytes and seconds."""\\n    files = [p for root in paths for p in sorted(Path(root).rglob("*")) if p.is_file()]\\n    t0, done, lock = time.time(), [0], threading.Lock()\\n\\n    def worker(fs):\\n        for p in fs:\\n            try:\\n                with open(p, "rb") as f:\\n                    while True:\\n                        b = f.read(chunk)\\n                        if not b:\\n                            break\\n                        with lock:\\n                            done[0] += len(b)\\n            except Exception:\\n                pass\\n    big = sorted(files, key=lambda p: -p.stat().st_size)\\n    parts = [big[i::threads] for i in range(threads)]\\n    ths = [threading.Thread(target=worker, args=(x,), daemon=True) for x in parts if x]\\n    for t in ths:\\n        t.start()\\n    for t in ths:\\n        t.join()\\n    dt = time.time() - t0\\n    return {"files": len(files), "bytes": done[0], "sec": round(dt, 1), "MBps": round(done[0] / 1e6 / max(dt, 1e-3), 1)}\\n\\n\\n# ------------------------------------------------------------------ metrics\\ndef prom(port):\\n    txt = U.urlopen(f"http://127.0.0.1:{port}/metrics", timeout=15).read().decode("utf-8", "replace")\\n    out = {}\\n    for line in txt.splitlines():\\n        if not line or line.startswith("#"):\\n            continue\\n        m = re.match(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\\\\{[^}]*\\\\})?\\\\s+(\\\\S+)", line)\\n        if not m:\\n            continue\\n        try:\\n            v = float(m.group(3))\\n        except ValueError:\\n            continue\\n        if v != v:\\n            continue\\n        out[m.group(1)] = out.get(m.group(1), 0.0) + v\\n        out[m.group(1) + "@max"] = max(out.get(m.group(1) + "@max", v), v)\\n    return out\\n\\n\\nGEN, PROMPT, RUN, WAIT = ("vllm:generation_tokens_total", "vllm:prompt_tokens_total", "vllm:num_requests_running",\\n                          "vllm:num_requests_waiting")\\nPREEMPT, KVU, ITER = "vllm:num_preemptions_total", "vllm:kv_cache_usage_perc", "vllm:iteration_tokens_total_count"\\nACC, DRF, DTOK = ("vllm:spec_decode_num_accepted_tokens_total", "vllm:spec_decode_num_drafts_total",\\n                  "vllm:spec_decode_num_draft_tokens_total")\\n\\n\\n# ------------------------------------------------------------------ one streaming request (Exp 14 VGen, port-aware)\\nclass Stream(threading.Thread):\\n    def __init__(self, port, ids, max_new, temperature=1.0, seed=None, logprobs=0, ignore_eos=True, model="q27"):\\n        super().__init__(daemon=True)\\n        self.port = int(port)\\n        self.body = {"model": model, "prompt": list(ids), "max_tokens": int(max_new), "temperature": temperature,\\n                     "top_p": 0.95 if temperature else 1.0, "stream": True, "ignore_eos": ignore_eos, "return_token_ids": True}\\n        if temperature:\\n            self.body["top_k"] = 20\\n        if seed is not None:\\n            self.body["seed"] = int(seed)\\n        if logprobs:\\n            self.body["logprobs"] = int(logprobs)\\n            self.body["return_tokens_as_token_ids"] = True      # rev 3: logprob keys by id (no string collisions)\\n        self.text, self.tokens, self.margins = [], [], []\\n        self.err, self.done_seen, self.finish_reason = None, False, None\\n        self.t_start = self.t_first = self.t_end = None\\n        self._resp, self._halt = None, False\\n\\n    def run(self):\\n        self.t_start = time.perf_counter()\\n        try:\\n            req = U.Request(f"http://127.0.0.1:{self.port}/v1/completions", data=json.dumps(self.body).encode(),\\n                            method="POST", headers={"Content-Type": "application/json"})\\n            self._resp = U.urlopen(req, timeout=7200)\\n            for raw in self._resp:\\n                if self._halt:\\n                    break\\n                line = raw.strip()\\n                if not line.startswith(b"data:"):\\n                    continue\\n                payload = line[5:].strip()\\n                if payload == b"[DONE]":\\n                    self.done_seen = True\\n                    break\\n                d = json.loads(payload)\\n                if "error" in d or d.get("object") == "error":\\n                    self.err = ("stream error: " + json.dumps(d.get("error") or d.get("message") or d))[:300]\\n                    break\\n                for c in d.get("choices") or []:\\n                    self.text.append(c.get("text") or "")\\n                    ids_ = c.get("token_ids")\\n                    lp = c.get("logprobs") or {}\\n                    if ids_ is None:\\n                        if c.get("text") or lp.get("tokens"):\\n                            self.err = "MISSING_STREAM_TOKEN_IDS"\\n                    elif any(not isinstance(v, int) or isinstance(v, bool) for v in ids_):\\n                        self.err = "INVALID_STREAM_TOKEN_IDS"\\n                    else:\\n                        if ids_ and self.t_first is None:\\n                            self.t_first = time.perf_counter()\\n                        self.tokens.extend(ids_)\\n                    for top in lp.get("top_logprobs") or []:\\n                        vals = sorted((v for v in (top or {}).values() if isinstance(v, (int, float))), reverse=True)\\n                        self.margins.append(round(vals[0] - vals[1], 4) if len(vals) >= 2 else None)\\n                    if c.get("finish_reason") is not None:\\n                        self.finish_reason = c["finish_reason"]\\n                if self.err:\\n                    break\\n        except Exception as e:\\n            if not self._halt:\\n                self.err = repr(e)[:300]\\n        finally:                         # rev 3: a stop() before the response existed still closes the socket here\\n            try:\\n                if self._resp is not None:\\n                    self._resp.close()\\n            except Exception:\\n                pass\\n        self.t_end = time.perf_counter()\\n        if not self._halt and not self.err and not (self.done_seen and self.finish_reason):\\n            self.err = f"INCOMPLETE_SSE done={self.done_seen} finish={self.finish_reason}"\\n\\n    def stop(self):\\n        self._halt = True\\n        try:\\n            self._resp.fp.raw._sock.shutdown(socket.SHUT_RDWR)\\n        except Exception:\\n            pass\\n        try:\\n            if self._resp is not None:\\n                self._resp.close()\\n        except Exception:\\n            pass\\n\\n    def row(self):\\n        return {"text": "".join(self.text), "tokens": list(self.tokens), "margins": list(self.margins), "error": self.err,\\n                "done": self.done_seen, "finish_reason": self.finish_reason}\\n\\n\\n# ------------------------------------------------------------------ correctness\\ndef greedy(port, prompts, n_tok=128, logprobs=2, timeout=900, model="q27"):\\n    """Greedy rows for the same-weight oracle (one request at a time: batch-size effects stay out of the comparison)."""\\n    rows = []\\n    for ids in prompts:\\n        g = Stream(port, ids, n_tok, temperature=0.0, logprobs=logprobs, ignore_eos=True, model=model)\\n        g.start()\\n        g.join(timeout)\\n        if g.is_alive():\\n            g.stop()\\n            g.join(10)\\n            r = g.row()\\n            r["error"] = r["error"] or "GREEDY_TIMEOUT"\\n        else:\\n            r = g.row()\\n        rows.append(r)\\n    return rows\\n\\n\\ndef degenerate(text):\\n    """Objective garbage checks: empty, a run of one char, \\\'!!!!\\\' (the E3 / E13 garbage), very low variety."""\\n    t = text or ""\\n    if not t.strip():\\n        return "EMPTY"\\n    if re.search(r"(\\\\S)\\\\1{200,}", t, re.S):      # rev 3: grid rows / separators are not garbage\\n        return "REPEATED_CHAR"\\n    if t.count("!") > 0.5 * max(1, len(t)):\\n        return "BANG_GARBAGE"\\n    words = re.findall(r"[A-Za-z]{3,}", t)            # rev 3c: words only; ARC grid dumps (digits) are not garbage\\n    if len(words) > 60 and len(set(words)) < 0.08 * len(words):\\n        return "LOW_VARIETY"\\n    return None\\n\\n\\ndef compare(ref, cand):\\n    """Same-weight oracle: at the first differing token the ORACLE\\\'s top-1 - top-2 logprob margin decides:\\n    >= FAIL_MARGIN -> FAIL_HIGH_MARGIN, below -> NEAR_TIE; identical -> EXACT; missing data -> INCONCLUSIVE."""\\n    if not ref or not cand or len(ref) != len(cand):\\n        return {"status": "INCONCLUSIVE_PAIRS_MISSING"}\\n    per, worst = [], "EXACT"\\n    rank = {"EXACT": 0, "NEAR_TIE": 1, "INCONCLUSIVE": 2, "FAIL_HIGH_MARGIN": 3}\\n    for a, b in zip(ref, cand):\\n        if a.get("error") or b.get("error") or not a.get("tokens") or not b.get("tokens"):\\n            st, d, m = "INCONCLUSIVE", None, None\\n        else:\\n            x, y = a["tokens"], b["tokens"]\\n            d = next((i for i in range(min(len(x), len(y))) if x[i] != y[i]), None)\\n            if d is None:\\n                st, m = ("EXACT" if len(x) == len(y) else "INCONCLUSIVE"), None\\n            else:\\n                mm = a.get("margins") or []\\n                m = mm[d] if d < len(mm) else None\\n                st = "INCONCLUSIVE" if m is None else ("FAIL_HIGH_MARGIN" if m >= FAIL_MARGIN else "NEAR_TIE")\\n        g = degenerate(b.get("text")) if (b.get("tokens") and not b.get("error")) else None\\n        if g and degenerate(a.get("text")) is None:           # rev 3c: a request error is not corruption\\n            st = "FAIL_HIGH_MARGIN"\\n        per.append({"first_diff": d, "oracle_margin": m, "status": st, "garbage": g})\\n        worst = st if rank[st] > rank[worst] else worst\\n    return {"status": worst, "rows": per}\\n\\n\\ndef chat_probes(port, model="q27", cases=(("Reply with exactly one number: 7 * 6 =", "42"),\\n                                          ("Reply with exactly one number: 2 + 2 =", "4"),\\n                                          ("Reply with exactly one number: 9 - 3 =", "6")), timeout=300):\\n    """Arithmetic through /v1/chat/completions with thinking off. vLLM 0.32 returns the answer in message.content and\\n    the thinking in message.reasoning (NOT reasoning_content: the consolidated notebook\\\'s blocker)."""\\n    out = []\\n    for q, want in cases:\\n        body = {"model": model, "messages": [{"role": "user", "content": q}], "temperature": 0, "max_tokens": 32,\\n                "chat_template_kwargs": {"enable_thinking": False}}\\n        try:\\n            r = json.loads(U.urlopen(U.Request(f"http://127.0.0.1:{port}/v1/chat/completions", data=json.dumps(body).encode(),\\n                                               headers={"Content-Type": "application/json"}), timeout=timeout).read())\\n            msg = r["choices"][0]["message"]\\n            got = (msg.get("content") or "").strip()\\n            out.append({"q": q, "want": want, "content": got[:80], "reasoning_chars": len(msg.get("reasoning") or ""),\\n                        "ok": re.search(r"(?<!\\\\d)" + want + r"(?!\\\\d)", got) is not None})\\n        except Exception as e:\\n            out.append({"q": q, "want": want, "error": repr(e)[:200], "ok": False})\\n    return {"ok": all(x["ok"] for x in out), "cases": out}\\n\\n\\ndef template_probe(port, messages, want_ids, model="q27", timeout=120):\\n    """rev 3h (three GPT SMOKE audits): what the SERVER\\\'s chat path renders for `messages` (its own template and default\\n    chat-template kwargs, no per-request kwargs) against the production xhigh render `want_ids`. MATCH / MISMATCH /\\n    ERROR; the load requests themselves use pre-rendered ids, so this shows chat-path equivalence only."""\\n    body = {"model": model, "messages": messages, "add_generation_prompt": True}\\n    try:\\n        r = json.loads(U.urlopen(U.Request(f"http://127.0.0.1:{port}/tokenize", data=json.dumps(body).encode(),\\n                                           headers={"Content-Type": "application/json"}), timeout=timeout).read())\\n        got = list(r.get("tokens") or [])\\n    except Exception as e:\\n        return {"status": "ERROR", "error": repr(e)[:200]}\\n    want = list(want_ids)\\n    first = next((i for i, (a, b) in enumerate(zip(got, want)) if a != b), None if len(got) == len(want) else min(len(got), len(want)))\\n    return {"status": "MATCH" if got == want else "MISMATCH", "server_tokens": len(got), "want_tokens": len(want),\\n            "first_diff": first}\\n\\n\\ndef teacher_forced(port, ids, last=256, model="q27", timeout=1800):\\n    """rev 3 (measurement M3): prompt logprobs of the last `last` positions of a long prompt (one prefill, 1 token out),\\n    for a lossy-KV comparison (NVFP4 vs FP8 KV) at the context lengths that matter. Returns per-position\\n    [logprob of the actual token, its rank, the engine\\\'s own rank-1 token id] or an error. rev 3h (three GPT SMOKE\\n    audits): the rank-1 token id is kept, so top-1 agreement compares the two engines\\\' predictions."""\\n    body = {"model": model, "prompt": list(ids), "max_tokens": 1, "temperature": 0, "prompt_logprobs": 1,\\n            "return_tokens_as_token_ids": True}\\n    try:\\n        r = json.loads(U.urlopen(U.Request(f"http://127.0.0.1:{port}/v1/completions", data=json.dumps(body).encode(),\\n                                           headers={"Content-Type": "application/json"}), timeout=timeout).read())\\n        pl = r["choices"][0].get("prompt_logprobs") or []\\n    except Exception as e:\\n        return {"error": repr(e)[:300]}\\n    out = []\\n    for pos in range(max(1, len(ids) - last), len(ids)):\\n        d = pl[pos] if pos < len(pl) else None\\n        if not isinstance(d, dict):\\n            out.append(None)\\n            continue\\n        e = d.get(str(ids[pos])) or d.get(f"token_id:{ids[pos]}")     # rev 3c: no fallback to another token\\n        top1 = [_tok_id(k) for k, v in d.items() if isinstance(v, dict) and v.get("rank") == 1]\\n        out.append([e.get("logprob"), e.get("rank"), top1[0] if len(top1) == 1 else None] if isinstance(e, dict) else None)\\n    return {"positions": out, "n": len(out)}\\n\\n\\ndef _tok_id(k):\\n    """A prompt-logprob key (\\\'123\\\' or \\\'token_id:123\\\') as an int id; anything else stays a tagged string."""\\n    m = re.fullmatch(r"(?:token_id:)?(-?\\\\d+)", str(k))\\n    return int(m.group(1)) if m else "str:" + str(k)\\n\\n\\ndef compare_tf(ref, cand):\\n    """Teacher-forced comparison: mean / max |delta logprob| of the actual tokens and rank-1 agreement. rev 3d (four\\n    GPT audits): MEASURED only when EVERY expected position is valid on both sides (1 of 256 used to be MEASURED);\\n    MEASURED is a measurement, not a PASS (the caller applies the A/A-derived tolerance)."""\\n    if not ref or not cand or ref.get("error") or cand.get("error"):\\n        return {"status": "INCONCLUSIVE"}\\n    a_, b_ = ref.get("positions") or [], cand.get("positions") or []\\n    if not a_ or len(a_) != len(b_):\\n        return {"status": "INCONCLUSIVE_INCOMPLETE_TF", "ref_n": len(a_), "cand_n": len(b_)}\\n    if any(not a or not b or a[0] is None or b[0] is None for a, b in zip(a_, b_)):\\n        return {"status": "INCONCLUSIVE_INCOMPLETE_TF", "valid": sum(1 for a, b in zip(a_, b_) if a and b and a[0] is not None\\n                                                                     and b[0] is not None), "expected": len(a_)}\\n    pairs = list(zip(a_, b_))\\n    if any(len(a) < 3 or len(b) < 3 or a[2] is None or b[2] is None for a, b in pairs):\\n        return {"status": "INCONCLUSIVE_TOP1_MISSING", "expected": len(pairs),\\n                "with_top1": sum(1 for a, b in pairs if len(a) >= 3 and len(b) >= 3 and a[2] is not None and b[2] is not None)}\\n    d = sorted(abs(a[0] - b[0]) for a, b in pairs)\\n    # rev 3h (three GPT SMOKE audits): top-1 agreement = both engines predict the same rank-1 token id. rev 3d-3g counted\\n    # "the forced token is rank 1 on both sides or on neither", which reads 1.0 for ranks 2 vs 3.\\n    r1 = sum(1 for a, b in pairs if a[2] == b[2]) / len(pairs)\\n    both = sum(1 for a, b in pairs if (a[1] == 1) == (b[1] == 1)) / len(pairs)\\n    p99 = d[min(len(d) - 1, int(round(0.99 * (len(d) - 1))))]          # rev 3e: robust to one outlier position\\n    return {"status": "MEASURED", "n": len(pairs), "mean_abs_dlogprob": round(statistics.mean(d), 4),\\n            "p99_abs_dlogprob": round(p99, 3), "max_abs_dlogprob": round(d[-1], 3), "rank1_agreement": round(r1, 4),\\n            "forced_token_rank1_both_or_neither": round(both, 4)}\\n\\n\\ndef tf_complete(rows):\\n    """rev 3e: a teacher-forced row set counts as a reference only when every position of every band is valid\\n    (rev 3h: including the rank-1 token id)."""\\n    return bool(rows) and all(isinstance(r, dict) and not r.get("error") and r.get("positions") and\\n                              all(p and p[0] is not None and len(p) >= 3 and p[2] is not None for p in r["positions"])\\n                              for r in rows.values())\\n\\n\\n# ------------------------------------------------------------------ steady decode\\ndef steady(port, prompts, level, secs=60.0, admit=None, max_new=8192, temperature=1.0, seed=1234, model="q27",\\n           sample=2.0, prefill_tok_s=700.0, gpu_sample=True, barrier=None):\\n    """`level` distinct prompts, then a decode window that starts only once prefill has settled. Status:\\n    STEADY / CAPACITY_LIMITED (fewer running than asked: KV) / INCONCLUSIVE_* / FAIL_*; numbers are kept on an\\n    INCONCLUSIVE window so a KV census survives; never a made-up number. `barrier` (steady_multi) starts every replica\\\'s\\n    window at the same moment."""\\n    plen = len(prompts[0]) if prompts else 0\\n    admit = admit or max(420.0, 120.0 + level * plen / prefill_tok_s)\\n    r = {"level": level, "status": "INCONCLUSIVE_NOT_STARTED", "port": port, "prompt_tokens": plen, "max_new": max_new,\\n         "admit_limit_sec": round(admit)}\\n    if not wait_idle(port, 60):\\n        r["status"] = "INCONCLUSIVE_NOT_IDLE_AT_START"\\n        if barrier:\\n            barrier.abort()\\n        return r\\n    try:\\n        p0 = prom(port)\\n    except Exception as e:\\n        r.update(status="INCONCLUSIVE_METRICS_UNAVAILABLE", error=repr(e)[:200])\\n        if barrier:\\n            barrier.abort()\\n        return r\\n    if GEN not in p0 or PROMPT not in p0:\\n        r["status"] = "INCONCLUSIVE_COUNTERS_MISSING"\\n        if barrier:\\n            barrier.abort()\\n        return r\\n    ss = [Stream(port, prompts[i % len(prompts)], max_new, temperature, seed + i, model=model) for i in range(level)]\\n    for g in ss:\\n        g.start()\\n    try:\\n        t_adm, best, settled, prev, samples, flat_since = time.perf_counter(), 0, 0, None, [], None\\n        last_tok = [0] * len(ss)\\n        while time.perf_counter() - t_adm < admit:\\n            time.sleep(sample)\\n            if all(not g.is_alive() for g in ss):          # rev 3: checked before the metrics read (a dead server)\\n                r.update(status="INCONCLUSIVE_STREAMS_ENDED_IN_ADMISSION", errors=[g.err for g in ss if g.err][:3])\\n                if barrier:\\n                    barrier.abort()                       # rev 3c: the other replicas must not wait for this one\\n                return r\\n            try:\\n                m = prom(port)\\n            except Exception:\\n                continue\\n            run, wait, pt, pre = int(m.get(RUN, 0)), int(m.get(WAIT, 0)), m.get(PROMPT), m.get(PREEMPT, 0)\\n            now_tok = [len(g.tokens) for g in ss]\\n            first = sum(g.t_first is not None for g in ss)\\n            gaining = sum(1 for a, b in zip(last_tok, now_tok) if b > a)\\n            last_tok = now_tok\\n            best = max(best, run)\\n            samples.append([round(time.perf_counter() - t_adm, 1), run, wait, first, gaining])\\n            # rev 3: no preemption in the sample, every running request emitted and is gaining tokens\\n            calm = run > 0 and prev is not None and pt == prev[0] and pre == prev[1] and first >= run and gaining >= run\\n            if calm and run >= level and wait == 0:\\n                settled += 1\\n            elif calm and run < level and wait > 0:       # KV-bound: the rest wait; accept a plateau held 30 s\\n                flat_since = flat_since or time.perf_counter()\\n                settled = settled + 1 if time.perf_counter() - flat_since > 30 else 0\\n            else:\\n                settled, flat_since = 0, None\\n            prev = (pt, pre)\\n            if settled >= 3:\\n                break\\n        r["admission"] = {"sec": round(time.perf_counter() - t_adm, 1), "best_running": best, "tail": samples[-6:],\\n                          "prefill_tok_s": round((prom(port).get(PROMPT, 0) - p0.get(PROMPT, 0)) /\\n                                                 max(1e-3, time.perf_counter() - t_adm), 1)}\\n        if settled < 3:\\n            r["status"] = "INCONCLUSIVE_NO_SETTLE"\\n            if barrier:\\n                barrier.abort()\\n            return r\\n        if barrier:\\n            try:\\n                barrier.wait(timeout=max(60.0, admit))\\n            except threading.BrokenBarrierError:\\n                r["status"] = "INCONCLUSIVE_REPLICA_BARRIER"\\n                return r\\n        m0, t0 = prom(port), time.perf_counter()\\n        runs, waits, kvs, mids, gmem, stalls = [], [], [], [], [], [0] * len(ss)\\n        last_tok, track, misses = [len(g.tokens) for g in ss], None, 0\\n        while time.perf_counter() - t0 < secs:\\n            time.sleep(min(sample, max(0.1, secs - (time.perf_counter() - t0))))\\n            try:\\n                m = prom(port)\\n            except Exception:                              # rev 3c: one failed metrics read is not a crash\\n                misses += 1\\n                if misses >= 5:\\n                    break\\n                continue\\n            mids.append(m)\\n            runs.append(m.get(RUN, 0))\\n            waits.append(m.get(WAIT, 0))\\n            if KVU in m:\\n                kvs.append(m[KVU])\\n            now_tok = [len(g.tokens) for g in ss]\\n            if track is None:                              # rev 3c: only streams decoding at window start can stall\\n                track = {i for i in range(len(ss)) if now_tok[i] > last_tok[i]}\\n            for i, g in enumerate(ss):\\n                if i in track and g.is_alive():\\n                    stalls[i] = stalls[i] + 1 if now_tok[i] <= last_tok[i] else 0\\n            last_tok = now_tok\\n            if any(x >= 3 for x in stalls):\\n                break\\n            if gpu_sample and len(mids) % 5 == 1:\\n                g_ = gpu_mem()\\n                if g_:\\n                    gmem.append(g_)\\n        m1, t1 = prom(port), time.perf_counter()\\n        gen = m1.get(GEN, 0) - m0.get(GEN, 0)\\n        dt = t1 - t0\\n        acc, drf, dtk, it = (m1.get(k, 0) - m0.get(k, 0) for k in (ACC, DRF, DTOK, ITER))\\n        rm = statistics.mean(runs) if runs else 0\\n        steps = it if it else ((drf / rm) if (drf and rm) else ((gen / rm) if rm else 0))\\n        r.update(tok_s=round(gen / dt, 1) if dt else None, gen_tokens=int(gen), window_sec=round(dt, 1),\\n                 running_mean=round(rm, 2), running_min=min(runs or [0]), running_max=max(runs or [0]),\\n                 waiting_min=min(waits or [0]), waiting_max=max(waits or [0]),\\n                 tok_s_per_stream=round(gen / dt / rm, 2) if rm and dt else None,\\n                 accept_per_draft=round(acc / drf, 3) if drf else None,\\n                 mean_acceptance_length=round(1 + acc / drf, 3) if drf else None,\\n                 accept_rate=round(acc / dtk, 3) if dtk else None,\\n                 steps=int(steps), steps_source="iteration_histogram" if it else "estimate",\\n                 step_ms=round(dt * 1000 / steps, 1) if steps else None,\\n                 tokens_per_step=round(gen / steps, 2) if steps else None,\\n                 kv_usage_mean=round(statistics.mean(kvs), 3) if kvs else None, kv_usage_max=max(kvs) if kvs else None,\\n                 gpu_mem_peak_mib=[max(row[i][1] for row in gmem if len(row) > i) for i in range(max(map(len, gmem)))] if gmem else None,\\n                 preemptions=m1.get(PREEMPT, 0) - m0.get(PREEMPT, 0), t0=t0, t1=t1)\\n        if misses >= 5:\\n            r["status"] = "INCONCLUSIVE_METRICS_LOST"\\n        elif any(x >= 3 for x in stalls):\\n            r["status"] = "INCONCLUSIVE_STALLED_STREAM"\\n        elif sum(g.is_alive() for g in ss) < min(level, best):\\n            r["status"] = "INCONCLUSIVE_STREAMS_ENDED_IN_WINDOW"\\n        elif any(x.get(PROMPT) != m0.get(PROMPT) for x in mids + [m1]):\\n            r["status"] = "INCONCLUSIVE_PREFILL_IN_WINDOW"\\n        elif r["preemptions"] > 0:\\n            r["status"] = "INCONCLUSIVE_PREEMPTION"\\n        elif gen <= 0:\\n            r["status"] = "FAIL_NO_DECODE"\\n        elif min(runs or [0]) < level and (len(set(runs)) > 1 or len(set(waits)) > 1):\\n            r["status"] = "INCONCLUSIVE_RUNNING_SET_CHANGED"      # KV-bound window must hold one fixed set\\n        elif min(runs or [0]) < 0.8 * max(runs or [1]):\\n            r["status"] = "INCONCLUSIVE_RUNNING_UNSTABLE"\\n        else:\\n            r["status"] = ("STEADY" if min(runs or [0]) == max(runs or [0]) == level else\\n                           "CAPACITY_LIMITED" if max(runs or [0]) < level else "INCONCLUSIVE_RUNNING_ABOVE_LEVEL")\\n        return r\\n    finally:\\n        errs = sorted({g.err for g in ss if g.err})[:3]           # rev 3: snapshot before stop() sets _halt\\n        for g in ss:\\n            g.stop()\\n        for g in ss:\\n            g.join(10)\\n        r["client_leak"] = any(g.is_alive() for g in ss)\\n        r["stream_errors"] = errs\\n        gb = [degenerate("".join(g.text)[-4000:]) if g.tokens else None for g in ss]\\n        r["garbage"] = {"streams": len(ss), "with_output": sum(1 for g in ss if g.tokens),\\n                        "garbage": [x for x in gb if x], "sample_tail": "".join(ss[0].text)[-200:] if ss and ss[0].tokens else ""}\\n        if r["garbage"]["garbage"] and r.get("status") in ("STEADY", "CAPACITY_LIMITED"):\\n            r["status"] = "FAIL_GARBAGE_UNDER_LOAD"\\n        r["idle_after"] = wait_idle(port)\\n\\n\\ndef steady_multi(jobs, secs=60.0, **kw):\\n    """Several servers at once (TP1 replicas): jobs = [(port, prompts, level)]; one shared window (barrier). The aggregate\\n    counts only when every window is STEADY / CAPACITY_LIMITED and the windows overlap >= 80 % of the shortest."""\\n    out = [None] * len(jobs)\\n    bar = threading.Barrier(len(jobs))\\n\\n    def one(i, j):\\n        try:\\n            out[i] = steady(j[0], j[1], j[2], secs=secs, barrier=bar, **kw)\\n        except Exception as e:          # rev 3: a dead replica is recorded, not printed to stderr only\\n            out[i] = {"status": "ERROR_REPLICA", "error": f"{type(e).__name__}: {e}"[:300], "port": j[0]}\\n            try:\\n                bar.abort()\\n            except Exception:\\n                pass\\n    th = [threading.Thread(target=one, args=(i, j), daemon=True) for i, j in enumerate(jobs)]\\n    for t in th:\\n        t.start()\\n    for t in th:\\n        t.join()\\n    good = all(o and o.get("status") in ("STEADY", "CAPACITY_LIMITED") for o in out)\\n    ov = (min(o["t1"] for o in out) - max(o["t0"] for o in out)) if good else 0\\n    aligned = good and ov > 0.8 * min(o["window_sec"] for o in out)\\n    return {"replicas": out, "aligned": aligned, "overlap_sec": round(ov, 1),\\n            "tok_s": round(sum(o["tok_s"] for o in out), 1) if aligned else None,\\n            "running_mean": round(sum(o["running_mean"] for o in out), 2) if aligned else None,\\n            "status": "AGGREGATE_STEADY" if aligned else "INCONCLUSIVE_REPLICA_WINDOWS"}\\n\\n\\ndef wait_idle(port, wait=90.0):\\n    end, hits = time.perf_counter() + wait, 0\\n    while time.perf_counter() < end:\\n        try:\\n            m = prom(port)\\n            hits = hits + 1 if m.get(RUN, 0) == 0 and m.get(WAIT, 0) == 0 else 0\\n        except Exception:\\n            hits = 0\\n        if hits >= 3:\\n            return True\\n        time.sleep(2)\\n    return False\\n\\n\\n# ------------------------------------------------------------------ profiler (FULL-graph proof + per-category time)\\n# rev 3: matched on the kernel BASE name, attention before GDN ("fla" is a substring of "flashinfer"), real 0.32 GDN names\\nCATS = [("nccl", ("nccl", "allreduce", "all_reduce", "allgather", "all_gather")),\\n        ("attention", ("flashinfer", "flash_fwd", "unified_attention", "paged", "attn", "attention", "batchdecode", "batchprefill",\\n                   "reshape_and_cache")),\\n        ("gdn_linear_attn", ("gdn", "delta_rule", "chunk_", "recurrent", "causal_conv", "l2norm", "solve_tril", "merge_16x16",\\n                             "recompute_w_u", "kkt", "cumsum", "post_conv", "layer_norm_gated", "gated_delta")),\\n        ("gemm_marlin", ("marlin",)),\\n        ("gemm_other", ("gemm", "cutlass", "xmma", "cublas", "sm89_", "ampere_", "sm80_")),\\n        ("sampling", ("sampl", "topk", "top_k", "argmax", "softmax", "rejection"))]\\n\\n\\ndef _cat(name):\\n    # rev 3h: "void (anonymous namespace)::gdn_decode_post_conv_mtp_kernel<...>(...)" used to reduce to "void " -> other\\n    s = re.sub(r"\\\\(anonymous namespace\\\\)::", "", str(name))\\n    s = re.sub(r"^\\\\s*void\\\\s+", "", s)\\n    base = s.split("<")[0].split("(")[0].lower()\\n    return next((k for k, pats in CATS if any(p in base for p in pats)), "other")\\n\\n\\nCOMPUTE_CATS = ("nccl", "attention", "gdn_linear_attn", "gemm_marlin", "gemm_other")\\n# rev 3h: vLLM 0.32 marks each model execution as execute_context_N(N)_generation_N(N) (no gpu_model_runner range)\\nSCOPE_RE = re.compile(r"^execute_context_\\\\d+\\\\(\\\\d+\\\\)_generation_\\\\d+\\\\(\\\\d+\\\\)$")\\n\\n\\ndef _is_scope(e):\\n    n = str(e.get("name", ""))\\n    return SCOPE_RE.match(n) is not None or ("forward" in n.lower() and "gpu_model_runner" in n.lower())\\n\\n\\ndef _scope_kind(g, compute_eager, unresolved):\\n    """FULL = graph-launched with no eager model compute (attention / GDN / GEMM / NCCL) inside the execution scope;\\n    eager input-preparation kernels are counted apart and allowed. Unresolved eager launches never read FULL."""\\n    if g == 0:\\n        return "EAGER"\\n    if compute_eager:\\n        return "PIECEWISE"\\n    if unresolved:\\n        return "PIECEWISE" if g > 1 else "MIXED"\\n    return "FULL"\\n\\n\\ndef trace_summary(pdir):\\n    files = sorted(Path(pdir).rglob("*.pt.trace.json*"), key=lambda p: p.stat().st_mtime)\\n    if not files:\\n        return {"error": "NO_TRACE_FILE", "listing": [str(p) for p in Path(pdir).rglob("*")][:20]}\\n    f = next((p for p in files if re.search(r"rank0\\\\.|rank[-_]?0[._]|tp0_", p.name)), files[0])\\n    with (gzip.open if f.name.endswith(".gz") else open)(f, "rt", encoding="utf-8") as fh:\\n        tr = json.load(fh)\\n    ev = tr.get("traceEvents", [])\\n    kern = [e for e in ev if e.get("ph") == "X" and str(e.get("cat", "")).lower() in ("kernel", "gpu_kernel")]\\n    rt = [e for e in ev if e.get("ph") == "X" and str(e.get("cat", "")).lower() in ("cuda_runtime", "cuda_driver")]\\n    is_graph = lambda e: "GraphLaunch" in str(e.get("name", ""))\\n    is_kl = lambda e: re.search(r"LaunchKernel|cuLaunchKernel", str(e.get("name", ""))) is not None\\n    # per-scope counts inside the execution annotations (rev 3h: execute_context_*_generation_*; the older\\n    # gpu_model_runner forward ranges still count). rev 3c: user_annotation only, not the gpu_user_annotation copies.\\n    fw = sorted([e for e in ev if e.get("ph") == "X" and str(e.get("cat", "")) == "user_annotation" and _is_scope(e)],\\n                key=lambda e: e.get("ts", 0))\\n    kname = {(e.get("args") or {}).get("correlation"): str(e.get("name", "")) for e in kern\\n             if (e.get("args") or {}).get("correlation") is not None}\\n    per_fw, aux = [], {}\\n    for a in fw[:400]:\\n        lo, hi, tid = a.get("ts", 0), a.get("ts", 0) + a.get("dur", 0), a.get("tid")\\n        inside = [e for e in rt if e.get("tid") == tid and lo <= e.get("ts", -1) <= hi]\\n        kls = [e for e in inside if is_kl(e)]\\n        ce = un = 0\\n        for e in kls:\\n            kn = kname.get((e.get("args") or {}).get("correlation"))\\n            if kn is None:\\n                un += 1\\n            elif _cat(kn) in COMPUTE_CATS:\\n                ce += 1\\n            else:\\n                b = re.sub(r"^\\\\s*void\\\\s+", "", re.sub(r"\\\\(anonymous namespace\\\\)::", "", kn)).split("<")[0].split("(")[0][:60]\\n                aux[b] = aux.get(b, 0) + 1\\n        per_fw.append([sum(1 for e in inside if is_graph(e)), len(kls), ce, un])\\n    corr = {(e.get("args") or {}).get("correlation"): str(e.get("name", "")) for e in rt\\n            if (e.get("args") or {}).get("correlation") is not None}\\n    per, tot, names = {k: 0.0 for k, _ in CATS + [("other", ())]}, 0.0, {}\\n    src = {k: {"graph": 0, "eager": 0, "unknown": 0} for k in per}\\n    for e in kern:\\n        d = float(e.get("dur", 0.0))\\n        tot += d\\n        names[e.get("name", "")] = names.get(e.get("name", ""), 0.0) + d\\n        k = _cat(e.get("name", ""))\\n        per[k] += d\\n        launcher = corr.get((e.get("args") or {}).get("correlation"))\\n        src[k]["graph" if launcher and "GraphLaunch" in launcher else ("eager" if launcher else "unknown")] += 1\\n    tot_k = sum(v["graph"] + v["eager"] for v in src.values())\\n    eager_share = round(sum(v["eager"] for v in src.values()) / tot_k, 3) if tot_k else None\\n    fw_g = statistics.median([x[0] for x in per_fw]) if per_fw else None\\n    fw_k = statistics.median([x[1] for x in per_fw]) if per_fw else None\\n    kinds_l = [_scope_kind(g, ce, un) for g, k, ce, un in per_fw]\\n    if per_fw:                   # rev 3f (GPT rev 3e audit: 2 FULL + 1 EAGER forwards read FULL by the median)\\n        kinds = set(kinds_l)\\n        verdict = kinds.pop() if len(kinds) == 1 else "MIXED"       # every scope must agree\\n        basis = "per_scope_annotation (all scopes; eager input-prep kernels allowed, eager model compute is not)"\\n    elif tot_k:                  # rev 3d (GPT-2 reproduced FULL with 0 forwards): CUPTI alone never proves FULL\\n        att = src["attention"]\\n        hint = ("FULL" if att["eager"] == 0 and (eager_share or 0) < 0.05 else\\n                "PIECEWISE" if att["eager"] > 0 and any(v["graph"] for v in src.values()) else\\n                "EAGER" if not any(v["graph"] for v in src.values()) else "MIXED")\\n        verdict, basis = "UNPROVEN", f"no forward annotations (cupti hint: {hint})"\\n    else:\\n        verdict, basis = "UNKNOWN", "none"\\n    return {"file": f.name, "files": len(files), "kernels": len(kern), "kernel_us_sum": round(tot),\\n            "graph_launches": sum(1 for e in rt if is_graph(e)), "kernel_launches": sum(1 for e in rt if is_kl(e)),\\n            "forwards": len(per_fw), "forward_kinds": {k_: kinds_l.count(k_) for k_ in ("FULL", "PIECEWISE", "EAGER", "MIXED")},\\n            "per_forward_median_graph_launches": fw_g, "per_forward_median_kernel_launches": fw_k,\\n            "per_forward_median_compute_eager": statistics.median([x[2] for x in per_fw]) if per_fw else None,\\n            "per_forward_unresolved_eager": sum(x[3] for x in per_fw),\\n            "scope_names": sorted({str(a.get("name", "")) for a in fw})[:5],\\n            "aux_eager_kernels_in_scopes": dict(sorted(aux.items(), key=lambda x: -x[1])[:20]),\\n            "launch_source": src, "eager_kernel_share": eager_share, "graph_verdict": verdict, "verdict_basis": basis,\\n            "by_category_pct": {k: round(100 * v / tot, 1) for k, v in per.items()} if tot else {},\\n            "top_kernels_us": dict(sorted(((k, round(v)) for k, v in names.items()), key=lambda x: -x[1])[:25]),\\n            "note": "kernel time sums overlapping streams; NCCL time includes rank-skew waiting"}\\n\\n\\ndef profile(port, pdir, secs=6.0, delay_iters=20, max_iters=40):\\n    """POST /start_profile (bounded by iterations; falls back to a bare POST), decode, POST /stop_profile, summary."""\\n    rec = {}\\n    try:\\n        U.urlopen(U.Request(f"http://127.0.0.1:{port}/start_profile", method="POST",\\n                            data=json.dumps({"delay_iterations": delay_iters, "max_iterations": max_iters}).encode(),\\n                            headers={"Content-Type": "application/json"}), timeout=900).read()\\n        rec["start_profile"] = "ok_bounded"\\n    except Exception as e:\\n        rec["start_profile_bounded_error"] = repr(e)[:200]\\n        try:\\n            U.urlopen(U.Request(f"http://127.0.0.1:{port}/start_profile", data=b"", method="POST"), timeout=900).read()\\n            rec["start_profile"] = "ok"\\n        except Exception as e2:\\n            rec["start_profile"] = repr(e2)[:200]\\n    time.sleep(secs)\\n    try:\\n        U.urlopen(U.Request(f"http://127.0.0.1:{port}/stop_profile", data=b"", method="POST"), timeout=900).read()\\n        rec["stop_profile"] = "ok"\\n    except Exception as e:\\n        rec["stop_profile"] = repr(e)[:200]\\n    time.sleep(20)\\n    try:\\n        rec["trace"] = trace_summary(pdir)\\n    except Exception as e:\\n        rec["trace"] = {"error": f"TRACE_PARSE {type(e).__name__}: {e}"[:300]}\\n    return rec\\n\\n\\ndef profiled_load(port, prompts, level, pdir, secs=6.0, max_new=8192, model="q27", prefill_tok_s=700.0, spec=False):\\n    """`level` decoding streams (own window: the profiler slows decode), profile() only after prefill settled exactly\\n    as in steady() (prompt counter + preemptions flat, every running request emitted and gaining tokens). rev 3d: no\\n    profile when unsettled; the profile window must be decode-only (prompt counter flat) and, with `spec`, show\\n    speculative drafts (else the trace cannot speak for multi-token verification)."""\\n    plen = len(prompts[0]) if prompts else 0\\n    settle = max(300.0, 120.0 + level * plen / prefill_tok_s)\\n    ss = [Stream(port, prompts[i % len(prompts)], max_new, 1.0, 4321 + i, model=model) for i in range(level)]\\n    for g in ss:\\n        g.start()\\n    rec, t0, flat, ok, prev, last_tok = {"level": level}, time.perf_counter(), None, 0, None, [0] * level\\n    try:\\n        while time.perf_counter() - t0 < settle:\\n            time.sleep(2)\\n            try:\\n                m = prom(port)\\n            except Exception:\\n                continue\\n            run, wait, pt, pre = int(m.get(RUN, 0)), int(m.get(WAIT, 0)), m.get(PROMPT), m.get(PREEMPT, 0)\\n            now_tok = [len(g.tokens) for g in ss]\\n            gaining = sum(1 for a, b in zip(last_tok, now_tok) if b > a)\\n            last_tok = now_tok\\n            first = sum(g.t_first is not None for g in ss)\\n            calm = run > 0 and prev is not None and (pt, pre) == prev and first >= run and gaining >= run\\n            if calm and run >= level and wait == 0:\\n                ok += 1\\n            elif calm and wait > 0:\\n                flat = flat or time.perf_counter()\\n                ok = ok + 1 if time.perf_counter() - flat > 30 else 0\\n            else:\\n                ok, flat = 0, None\\n            prev = (pt, pre)\\n            if ok >= 3:\\n                break\\n        rec["settled"] = ok >= 3\\n        if not rec["settled"]:\\n            rec["status"] = "INCONCLUSIVE_PREFILL_NOT_SETTLED"\\n            return rec\\n        try:\\n            m0 = prom(port)\\n            rec["running_at_start"] = m0.get(RUN)\\n        except Exception:\\n            m0 = {}\\n        rec.update(profile(port, pdir, secs))\\n        try:\\n            m1 = prom(port)\\n        except Exception:\\n            m1 = {}\\n        rec["window_deltas"] = {"prompt_tokens": m1.get(PROMPT, 0) - m0.get(PROMPT, 0), "drafts": m1.get(DRF, 0) - m0.get(DRF, 0),\\n                                "accepted": m1.get(ACC, 0) - m0.get(ACC, 0), "generated": m1.get(GEN, 0) - m0.get(GEN, 0)}\\n        if not m0 or not m1:\\n            rec["status"] = "INCONCLUSIVE_METRICS_UNAVAILABLE"\\n        elif rec["window_deltas"]["prompt_tokens"]:\\n            rec["status"] = "INCONCLUSIVE_PREFILL_IN_PROFILE"\\n        elif spec and not (rec["window_deltas"]["drafts"] and rec["window_deltas"]["accepted"]):\\n            rec["status"] = "INCONCLUSIVE_NO_SPEC_DECODE"            # rev 3e: drafts AND accepted tokens in the window\\n        return rec\\n    finally:\\n        for g in ss:\\n            g.stop()\\n        for g in ss:\\n            g.join(10)\\n        rec["idle_after"] = wait_idle(port)\\n\'\n    (TMP / "e23_lab.py").write_text(LAB_SRC)\n    sys.path.insert(0, str(TMP))\n    import e23_lab as L\n    P60532 = \'diff --git a/tests/v1/attention/test_attention_backends.py b/tests/v1/attention/test_attention_backends.py\\nindex bb8e154b2f37..0f911594931b 100644\\n--- a/tests/v1/attention/test_attention_backends.py\\n+++ b/tests/v1/attention/test_attention_backends.py\\n@@ -500,8 +500,8 @@ def _test_backend_correctness(\\n     if max_num_batched_tokens is not None:\\n         vllm_config.scheduler_config.max_num_batched_tokens = max_num_batched_tokens\\n     if num_speculative_tokens > 0:\\n-        vllm_config.speculative_config = SimpleNamespace(\\n-            num_speculative_tokens=num_speculative_tokens\\n+        vllm_config.speculative_config = SpeculativeConfig(\\n+            method="ngram", num_speculative_tokens=num_speculative_tokens\\n         )\\n     vllm_config.cache_config.cache_dtype = kv_cache_dtype\\n     device = torch.device(f"{DEVICE_TYPE}:0")\\n@@ -1205,6 +1205,7 @@ def test_flashinfer_varlen_cudagraph_capability(\\n         use_v2_model_runner=True,\\n     )\\n     monkeypatch.setattr(fi, "can_use_trtllm_attention", lambda *_, **__: True)\\n+    monkeypatch.setattr(fi.current_platform, "has_device_capability", lambda _: True)\\n     monkeypatch.setattr(\\n         fi.FlashInferMetadataBuilder,\\n         "_get_flashinfer_trtllm_api_decode_kernel",\\n@@ -1715,16 +1716,17 @@ def causal_mask_mod(\\n \\n @pytest.mark.skipif(\\n     AttentionBackendEnum.FLASHINFER not in BACKENDS_TO_TEST\\n-    or not current_platform.is_device_capability_family(120),\\n-    reason="NVFP4 XQA decode requires SM12x.",\\n+    or not any(current_platform.is_device_capability_family(f) for f in (80, 120)),\\n+    reason="FlashInfer decodes an NVFP4 cache with XQA on SM12x and fa2 on SM8x.",\\n )\\n @pytest.mark.parametrize("use_cuda_graph", [False, True])\\n @pytest.mark.parametrize("q_len", [1, 4])\\n-def test_flashinfer_xqa_nvfp4_decode_correctness(\\n+def test_flashinfer_nvfp4_decode_correctness(\\n     default_vllm_config, q_len, use_cuda_graph\\n ):\\n     """Single-token and spec-decode queries over an NVFP4 cache decode with\\n-    XQA, eager and replayed, and match the fa2 prefill that wrote the cache."""\\n+    XQA on SM12x and fa2 on SM8x, eager and replayed, and match the fa2\\n+    prefill that wrote the cache."""\\n     import unittest.mock\\n \\n     from vllm.v1.attention.backends import flashinfer as fi\\n@@ -1801,15 +1803,46 @@ def run(query_lens, use_cuda_graph=False):\\n     # The full prefill writes every token through the NVFP4 writer; its fa2\\n     # output for the last q_len tokens of each request is the reference.\\n     _, reference = run(seq_lens)\\n-    xqa_fn = fi.flashinfer_xqa_batch_decode_with_kv_cache\\n-    with unittest.mock.patch.object(\\n-        fi, "flashinfer_xqa_batch_decode_with_kv_cache", wraps=xqa_fn\\n-    ) as xqa:\\n+    xqa = current_platform.is_device_capability_family(120)\\n+    name = "flashinfer_xqa_batch_decode_with_kv_cache" if xqa else "fast_plan_decode"\\n+    with unittest.mock.patch.object(fi, name, wraps=getattr(fi, name)) as decode:\\n         rows, output = run([q_len] * len(seq_lens), use_cuda_graph)\\n-    assert xqa.called\\n+    assert decode.called\\n+    assert xqa or decode.call_args.kwargs["q_len_per_req"] == q_len\\n     torch.testing.assert_close(output, reference[rows], atol=2e-2, rtol=2e-2)\\n \\n \\n+@pytest.mark.skipif(\\n+    AttentionBackendEnum.FLASHINFER not in BACKENDS_TO_TEST\\n+    or current_platform.has_device_capability(90),\\n+    reason="FlashInfer fa2 spec decode runs before SM90.",\\n+)\\n+@pytest.mark.parametrize("use_cuda_graph", [False, True])\\n+def test_flashinfer_fa2_spec_decode_correctness(default_vllm_config, use_cuda_graph):\\n+    """Uniform spec-decode queries take fa2 decode and match SDPA."""\\n+    import unittest.mock\\n+\\n+    from vllm.v1.attention.backends import flashinfer as flashinfer_backend\\n+\\n+    def causal_mask_mod(b, h, q_idx, kv_idx, *, context_len):\\n+        return (q_idx + context_len) >= kv_idx\\n+\\n+    with unittest.mock.patch.object(\\n+        flashinfer_backend,\\n+        "fast_plan_decode",\\n+        wraps=flashinfer_backend.fast_plan_decode,\\n+    ) as plan:\\n+        _test_backend_correctness(\\n+            BatchSpec(seq_lens=[32, 40, 1024], query_lens=[4, 4, 4]),\\n+            "meta-llama/Meta-Llama-3-8B",\\n+            [AttentionBackendEnum.FLASHINFER],\\n+            causal_mask_mod,\\n+            use_cuda_graph=use_cuda_graph,\\n+            num_speculative_tokens=3,\\n+        )\\n+    assert plan.call_args.kwargs["q_len_per_req"] == 4\\n+\\n+\\n if current_platform.is_rocm():\\n     # FLASH_ATTN is not supported on ROCm\\n     SLIDING_WINDOW_BACKENDS_TO_TEST = [\\ndiff --git a/vllm/v1/attention/backends/flashinfer.py b/vllm/v1/attention/backends/flashinfer.py\\nindex bb9e1b0e4577..1183259d884e 100755\\n--- a/vllm/v1/attention/backends/flashinfer.py\\n+++ b/vllm/v1/attention/backends/flashinfer.py\\n@@ -111,6 +111,11 @@ def _nvfp4_kv_on_fa2() -> bool:\\n     return any(current_platform.is_device_capability_family(f) for f in (80, 90, 120))\\n \\n \\n+def _use_fa2_native_decode() -> bool:\\n+    # Below SM90, native decode runs fa2, which takes uniform multi-token queries.\\n+    return not current_platform.has_device_capability(90)\\n+\\n+\\n _KVPair = tuple[torch.Tensor, torch.Tensor]\\n \\n \\n@@ -885,9 +890,9 @@ def __init__(\\n         )\\n         if self.enable_cuda_graph:\\n             # For full cudagraph capture, one `decode_wrapper` for each batch\\n-            # size is needed for FlashInfer.\\n+            # size and query length is needed for FlashInfer.\\n             self._decode_wrappers_cudagraph: dict[\\n-                int, BatchDecodeWithPagedKVCacheWrapper\\n+                tuple[int, int], BatchDecodeWithPagedKVCacheWrapper\\n             ] = {}\\n             self._decode_cudagraph_max_bs = (1 + num_spec_tokens) * max_num_reqs\\n             if self.compilation_config.max_cudagraph_capture_size is not None:\\n@@ -1021,10 +1026,12 @@ def __init__(\\n             == FlashInferDecodeKernel.TRTLLM_GEN\\n             and not self.use_dcp\\n         )\\n+        self.use_fa2_native_decode = _use_fa2_native_decode()\\n         self._init_reorder_batch_threshold(\\n             1,\\n             supports_spec_as_decode=(\\n                 self.flashinfer_trtllm_api_decode_kernel is not None\\n+                or self.use_fa2_native_decode\\n             ),\\n             # trtllm-gen decode receives no cp_rank/global-seq-len information,\\n             # so its end-aligned causal mask is wrong for q_len > 1 over the\\n@@ -1153,6 +1160,12 @@ def get_cudagraph_support(\\n             return AttentionCGSupport.UNIFORM_SINGLE_TOKEN_DECODE\\n \\n         kv_specs = iter_layer_specs(kv_cache_spec)\\n+        if _use_fa2_native_decode():\\n+            # fa2 decodes uniform causal spec queries.\\n+            if kv_specs and not vllm_config.attention_config.use_non_causal:\\n+                return AttentionCGSupport.UNIFORM_BATCH\\n+            return AttentionCGSupport.UNIFORM_SINGLE_TOKEN_DECODE\\n+\\n         num_qo_heads = vllm_config.model_config.get_num_attention_heads(\\n             vllm_config.parallel_config\\n         )\\n@@ -1197,8 +1210,10 @@ def get_varlen_cudagraph_max_query_len(\\n     ) -> int | None:\\n         # The uniform gate already covers DCP, head geometry, the XQA head-dim\\n         # limit and causality, and yields None for subclasses that force NEVER.\\n+        # The native fa2 decode wrapper takes only uniform query lengths.\\n         if (\\n-            cls.get_cudagraph_support(vllm_config, kv_cache_spec)\\n+            _use_fa2_native_decode()\\n+            or cls.get_cudagraph_support(vllm_config, kv_cache_spec)\\n             != AttentionCGSupport.UNIFORM_BATCH\\n         ):\\n             return None\\n@@ -1417,9 +1432,13 @@ def _get_prefill_wrapper(\\n         assert self._prefill_wrapper is not None\\n         return self._prefill_wrapper\\n \\n-    def _get_decode_wrapper(self, batch_size: int, use_cudagraph: bool = False):\\n+    def _get_decode_wrapper(\\n+        self, batch_size: int, use_cudagraph: bool = False, q_len_per_req: int = 1\\n+    ):\\n         if use_cudagraph:\\n-            decode_wrapper = self._decode_wrappers_cudagraph.get(batch_size, None)\\n+            decode_wrapper = self._decode_wrappers_cudagraph.get(\\n+                (batch_size, q_len_per_req), None\\n+            )\\n         else:\\n             decode_wrapper = self._decode_wrapper\\n \\n@@ -1434,7 +1453,13 @@ def _get_decode_wrapper(self, batch_size: int, use_cudagraph: bool = False):\\n                 paged_kv_last_page_len = None\\n             # NVFP4 KV cache requires the trtllm-gen backend inside\\n             # the wrapper on SM100; SM8x, SM90 and SM12x read it with fa2.\\n-            backend = "trtllm-gen" if self.nvfp4_trtllm else "auto"\\n+            backend = (\\n+                "trtllm-gen"\\n+                if self.nvfp4_trtllm\\n+                else "fa2"\\n+                if self.use_fa2_native_decode\\n+                else "auto"\\n+            )\\n             decode_wrapper = BatchDecodeWithPagedKVCacheWrapper(\\n                 self._get_workspace_buffer(),\\n                 get_flashinfer_layout_string(self.kv_cache_layout),\\n@@ -1451,7 +1476,9 @@ def _get_decode_wrapper(self, batch_size: int, use_cudagraph: bool = False):\\n \\n             # save the decode wrapper\\n             if use_cudagraph:\\n-                self._decode_wrappers_cudagraph[batch_size] = decode_wrapper\\n+                self._decode_wrappers_cudagraph[batch_size, q_len_per_req] = (\\n+                    decode_wrapper\\n+                )\\n             else:\\n                 self._decode_wrapper = decode_wrapper\\n \\n@@ -1890,10 +1917,14 @@ def build(\\n                     and pure_decode\\n                     and num_decode_tokens <= self._decode_cudagraph_max_bs\\n                 )\\n-                num_input_tokens = num_decode_tokens\\n+                num_input_tokens, q_len_per_req = num_decode_tokens, 1\\n+                if self.use_fa2_native_decode:\\n+                    # fa2 plans uniform spec decodes per request.\\n+                    num_input_tokens = num_decodes\\n+                    q_len_per_req = num_decode_tokens // num_decodes\\n \\n                 decode_wrapper = self._get_decode_wrapper(\\n-                    num_input_tokens, use_cudagraph\\n+                    num_input_tokens, use_cudagraph, q_len_per_req\\n                 )\\n                 # Use the persistent buffer with padding length,\\n                 # instead of the same address but chunked version\\n@@ -1942,6 +1973,7 @@ def build(\\n                     o_data_type=o_dtype,\\n                     fixed_split_size=self.decode_fixed_split_size,\\n                     disable_split_kv=self.disable_split_kv,\\n+                    q_len_per_req=q_len_per_req,\\n                 )\\n                 attn_metadata.decode = FIDecode(wrapper=decode_wrapper)\\n         return attn_metadata\\n@@ -2807,6 +2839,7 @@ def fast_plan_decode(\\n     non_blocking: bool = True,\\n     fixed_split_size: int = -1,\\n     disable_split_kv: bool = False,\\n+    q_len_per_req: int = 1,\\n ) -> None:\\n     """A faster version of BatchDecodeWithPagedKVCacheWrapper::plan used for\\n     cudagraph capture/replay, while the no cudagraph version turns back\\n@@ -2851,6 +2884,7 @@ def fast_plan_decode(\\n             seq_lens=seq_lens_cpu,\\n             fixed_split_size=fixed_split_size,\\n             disable_split_kv=disable_split_kv,\\n+            q_len_per_req=q_len_per_req,\\n         )\\n         self.vllm_first_call = False\\n         return\\n@@ -2878,6 +2912,7 @@ def fast_plan_decode(\\n         non_blocking=non_blocking,\\n         fixed_split_size=fixed_split_size,\\n         disable_split_kv=disable_split_kv,\\n+        q_len_per_req=q_len_per_req,\\n     )\\n \\n \\n\'\n    M = INP / "models" / "foysalemonshanto"\n    FP8 = find("config.json", M / "qwen3-8-27b-fp8-repacked-v1").parent\n    _i4 = find("config.json", M / "qwen3-8-27b-int4-redhat", must=False)\n    INT4 = _i4.parent if _i4 else None\n    _mm = find("q27_e23_p1_manifest.json", M / "qwen3-8-27b-w4a16-merlin", must=False)\n    MERLIN = _mm.parent if _mm else None\n    MERLIN_PAY = find("merlin_offline_tar.bin", M / "e23-merlin-env", must=False)\n    EAGLE_PAY = find("eagle_offline_tar.bin", M / "e23-eagle4-env", must=False)\n    _ed = find("config.json", M / "qwen3-8-27b-eagle4", must=False)\n    EAGLE_DRAFT = _ed.parent if _ed else None\n    KITS = find("q27_e23_kernels_manifest.json", INP).parent\n    WHEELHOUSE = find("install_wheelhouse.py", INP / "datasets" / "foysalemonshanto" / "vllm-0-32-0-cu13-wheelhouse")\n    ARC = find("arc-agi_evaluation_challenges.json")\n    R["inputs"] = {k: str(v) for k, v in dict(FP8=FP8, INT4=INT4, MERLIN=MERLIN, MERLIN_PAY=MERLIN_PAY, EAGLE_PAY=EAGLE_PAY,\n                                              EAGLE_DRAFT=EAGLE_DRAFT, KITS=KITS, WHEELHOUSE=WHEELHOUSE, ARC=ARC).items()}\n    MERLIN_OK = False\n    if MERLIN:\n        mt = json.loads((MERLIN / "q27_e23_p1_manifest.json").read_text())\n        MERLIN_OK = all((MERLIN / n).is_file() and (MERLIN / n).stat().st_size == s\n                        for n, (s, _) in mt["files"].items() if n != ".gitattributes")\n    R["inputs"]["merlin_files_ok"] = MERLIN_OK\n    if MODE in ("SMOKE", "TARGET") and PART in ("A", "ALL") and (not MERLIN_OK or MERLIN_PAY is None):\n        save()                   # rev 3c: SMOKE A exists for Merlin; never a silent rc 0 with only P4 / P5 data\n        raise RuntimeError("SMOKE A needs the Merlin checkpoint (qwen3-8-27b-w4a16-merlin/Transformers/default/1) and env "\n                           "(e23-merlin-env/Other/v1/1): " + json.dumps({"MERLIN": str(MERLIN), "MERLIN_PAY": str(MERLIN_PAY),\n                                                                         "merlin_files_ok": MERLIN_OK}))\n    km = json.loads((KITS / "q27_e23_kernels_manifest.json").read_text())       # rev 3: the kit files are what we built\n    R["inputs"]["kits_sha_ok"] = {n: (KITS / n).is_file() and hashlib.sha256((KITS / n).read_bytes()).hexdigest() == v.get("sha256")\n                                  for n, v in km.items() if isinstance(v, dict) and n.endswith((".so", ".py"))}\n    _need = ("p4_sm89_fp8.so", "p5_host_ar.so", "bench_fp8_gemm.py", "bench_host_ar.py")\n    _bad = sorted(n for n in _need if R["inputs"]["kits_sha_ok"].get(n) is not True)\n    if _bad and (MODE == "FULL" or PART in ("A", "ALL")):   # rev 3d (#9) / 3e: only the runs that use P4 / P5 (A)\n        R["errors"].append("KERNEL_KIT_HASH_FAILED: " + repr(_bad))\n        save()\n        raise RuntimeError("KERNEL_KIT_HASH_FAILED: " + repr(_bad))\n    R["identity"] = {k: L.identity(v) for k, v in dict(fp8=FP8, int4=INT4, merlin=MERLIN, eagle_draft=EAGLE_DRAFT).items() if v}\n    ABORT = {"why": None}\n    NVSMI_OK = bool(L.gpu_mem()) and not FAKE                       # rev 3c: a later nvidia-smi failure is not "released"\n    R["setup"]["gpus_at_start"] = L.gpu_mem()\n    if not FAKE and len(R["setup"]["gpus_at_start"] or []) != 4:          # rev 3d (#8): a hard stop, not a note\n        R["errors"].append(f"expected 4 GPUs, nvidia-smi shows {R[\'setup\'][\'gpus_at_start\']}")\n        save()\n        raise RuntimeError("GPU_PREFLIGHT_FAILED: " + repr(R["setup"]["gpus_at_start"]))\n    L.start_watchdog(HARD_STOP, lambda: ABORT.update(why="HARD_STOP reached: watchdog killed live servers"))\n    log("MODE", MODE, "PART", PART, "| budget", BUDGET // 60, "min |", json.dumps(R["inputs"])[:900])\n    save()\n\n\nif not RERUN:\n    # ==================================================================== CELL 2 — engines (stock vLLM 0.32, Merlin, EAGLE fork)\n    def run(name, cmd, timeout=3600, env=None):\n        t = time.time()\n        try:\n            with open(LOGS / f"{name}.log", "w") as f:\n                r = subprocess.run(["bash", "-c", "set -o pipefail; " + cmd], stdout=f, stderr=subprocess.STDOUT, timeout=timeout,\n                                   env={**os.environ, **(env or {})})\n            rc = r.returncode\n        except subprocess.TimeoutExpired:\n            rc = -9\n        txt = (LOGS / f"{name}.log").read_text(errors="replace")\n        R["setup"][name] = {"rc": rc, "sec": round(time.time() - t, 1), "tail": txt[-1500:]}\n        log(name, "rc", rc, f"{time.time() - t:.0f}s")\n        save()\n        return rc, txt\n\n\n    SHIM = TMP / "e23_libcuda"          # FlashInfer JIT links -lcuda; Kaggle has only libcuda.so.1\n    SHIM.mkdir(exist_ok=True)\n    for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/compat", "/usr/lib64"):\n        c = sorted(Path(d).glob("libcuda.so.1")) if Path(d).is_dir() else []\n        if c and not (SHIM / "libcuda.so").exists():\n            (SHIM / "libcuda.so").symlink_to(c[0])\n    if FAKE:\n        STOCK = {"vllm": [sys.executable, FAKE], "env": dict(os.environ), "py": sys.executable, "name": "stock"}\n    else:\n        rc, txt = run("install_stock", f"{sys.executable} {WHEELHOUSE} {TMP}/vllm-venv", timeout=2400)\n        kv = dict(l.split("=", 1) for l in txt.splitlines() if l.startswith(("VENV_PYTHON=", "CUDA_HOME=")))\n        if rc or "CUDA_HOME" not in kv:                                   # rev 3: a clear failure, not a KeyError\n            raise RuntimeError("stock vLLM 0.32 install failed (logs/install_stock.log)")\n        env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH", "LD_PRELOAD")}\n        env.update(CUDA_HOME=kv["CUDA_HOME"], PATH=f"{TMP}/vllm-venv/bin:{kv[\'CUDA_HOME\']}/bin:{env.get(\'PATH\', \'\')}",\n                   LIBRARY_PATH=f"{SHIM}:{env.get(\'LIBRARY_PATH\', \'\')}", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",\n                   VLLM_NO_USAGE_STATS="1", VLLM_CACHE_ROOT=f"{TMP}/vllm-cache", VLLM_ALLOW_LONG_MAX_MODEL_LEN="1",\n                   CUDA_DEVICE_ORDER="PCI_BUS_ID")\n        STOCK = {"vllm": [f"{TMP}/vllm-venv/bin/vllm", "serve"], "env": env, "py": kv["VENV_PYTHON"], "cuda_home": kv["CUDA_HOME"],\n                 "name": "stock"}\n        R["setup"]["stock_version"] = subprocess.run([kv["VENV_PYTHON"], "-c", "import vllm; print(vllm.__version__)"],\n                                                     capture_output=True, text=True, env=env).stdout.strip()\n    HELP = {}\n\n\n    def prefetch_bg(paths, tag):\n        """rev 3: warm the page cache of a cold 17-30 GB /kaggle/input checkpoint in the background (many readers)."""\n        def go():\n            R["setup"]["prefetch_" + tag] = L.prefetch([p for p in paths if p]) if not FAKE else {"fake": True}\n            save()\n        th = threading.Thread(target=go, daemon=True)\n        th.start()\n        return th\n\n\n    MERLIN_ENG = None\n\n\n    def merlin_install():\n        """P2 payload (.bin = a tar Kaggle does not unpack) -> install_merlin_offline.sh (UV_OFFLINE); payload deleted after;\n        a failure is cached so later Merlin steps fail fast (rev 3)."""\n        global MERLIN_ENG\n        if MERLIN_ENG is not None:\n            if MERLIN_ENG.get("failed"):\n                raise RuntimeError("Merlin install failed earlier: " + MERLIN_ENG["failed"])\n            return MERLIN_ENG\n        if not MERLIN_OK or MERLIN_PAY is None:\n            MERLIN_ENG = {"failed": "Merlin checkpoint / env not attached or incomplete"}\n            raise RuntimeError(MERLIN_ENG["failed"])\n        if FAKE:\n            MERLIN_ENG = {"fake": True}\n            return MERLIN_ENG\n        try:\n            pay_root = TMP / "merlin_pay"\n            shutil.rmtree(pay_root, ignore_errors=True)\n            pay_root.mkdir()\n            t = time.time()\n            with tarfile.open(MERLIN_PAY) as tf:\n                tf.extractall(pay_root, filter="data")\n            R["setup"]["merlin_extract_sec"] = round(time.time() - t, 1)\n            pay = pay_root / "merlin_offline"\n            shutil.rmtree(TMP / "merlin_off", ignore_errors=True)\n            rc, txt = run("install_merlin", f"bash {pay}/install_merlin_offline.sh {pay} {TMP}/merlin_off", timeout=5400)\n            kv = dict(re.findall(r"^(MERLIN_[A-Z]+)=(.*)$", txt, re.M))\n            shutil.rmtree(pay_root, ignore_errors=True)\n            if rc or "INSTALL OK" not in txt:\n                raise RuntimeError("Merlin offline install failed (logs/install_merlin.log)")\n            sys.path.insert(0, kv["MERLIN_TREE"] + "/tools/merlin")\n            import launch as ML\n            site = subprocess.run([kv["MERLIN_VENV"] + "/bin/python", "-c", "import sysconfig;print(sysconfig.get_paths()[\'purelib\'])"],\n                                  capture_output=True, text=True).stdout.strip()\n            MERLIN_ENG = {"kv": kv, "ML": ML, "cuda_home": site + "/nvidia/cu13"}\n            return MERLIN_ENG\n        except Exception as e:\n            MERLIN_ENG = {"failed": f"{type(e).__name__}: {e}"[:300]}\n            raise\n\n\n    MERLIN_CC = {"mode": "gcc13"}       # rev 3c: Merlin README:229 / install-env.sh: FlashInfer JIT at first start needs GCC >= 13\n\n\n    def merlin_server(name, profile, gpus, port, kv_bytes, seqs=None, capture=None):\n        """Merlin\'s own launch.build() (pure: the profile\'s flags + env, no host refusal), our max len / port last, and\n        seqs / capture sizes only when given (None = the profile\'s own, the TP1 memory fallback). Compiler: the payload\'s\n        GCC 13 (+ its libstdc++ when present) unless a GCC 13 compile failed earlier (then the system GCC 11)."""\n        eng = merlin_install()\n        extra = ["--served-model-name", "q27", "--max-model-len", str(MAX_LEN), "--port", str(port)]\n        # rev 3h (three GPT SMOKE audits): Merlin\'s profile serves the froggeric v22.5 template with reasoning_effort "medium";\n        # production = the checkpoint\'s own template at xhigh (argparse: the last occurrence wins over the profile\'s flags)\n        extra += ["--chat-template", str(CHAT_TEMPLATE_FILE),\n                  "--default-chat-template-kwargs", json.dumps({"enable_thinking": True, "reasoning_effort": "xhigh"})]\n        if seqs:\n            extra += ["--max-num-seqs", str(seqs)]\n        if capture:\n            extra += ["--compilation-config", json.dumps({"cudagraph_mode": "FULL_AND_PIECEWISE", "cudagraph_capture_sizes": capture,\n                                                          "inductor_compile_config": {"test_configs.force_filter_reduction_configs": True,\n                                                                                      "dynamic_scale_rblock": False}})]\n        if FAKE:\n            return L.Server(name, [sys.executable, FAKE, str(MERLIN), "--tensor-parallel-size", str(len(gpus.split(","))),\n                                   "--max-num-seqs", str(seqs or 2), "--port", str(port)], os.environ, port, LOGS, gpus=gpus)\n        kvv = eng["kv"]\n        base = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH", "CUDA_HOME", "LD_PRELOAD", "CC", "CXX")}\n        base.update(GPUS=gpus, PORT=str(port), KV_BYTES=str(kv_bytes), P2P="off", VENV=kvv["MERLIN_VENV"])\n        plan = eng["ML"].build("27b", profile, str(MERLIN), base, extra=extra, checkpoint="27b", status="untested")\n        env = dict(plan.env)\n        # -lcuda shim (FlashInfer JIT), per-port caches; compiler set below (rev 3c: GCC 13 first, see MERLIN_CC)\n        env.update(LD_PRELOAD=kvv["MERLIN_SHIM"], CUDA_HOME=eng["cuda_home"], VLLM_USE_RUST_FRONTEND="0", HF_HUB_OFFLINE="1",\n                   TRANSFORMERS_OFFLINE="1", LIBRARY_PATH=f"{SHIM}:{os.environ.get(\'LIBRARY_PATH\', \'\')}",\n                   PATH=f"{kvv[\'MERLIN_VENV\']}/bin:{eng[\'cuda_home\']}/bin:{os.environ.get(\'PATH\', \'\')}",\n                   VLLM_CACHE_ROOT=f"{TMP}/merlin-cache", TRITON_CACHE_DIR=f"{TMP}/merlin-triton-{port}",\n                   FLASHINFER_WORKSPACE_BASE=f"{TMP}/merlin-fi", CUDA_DEVICE_ORDER="PCI_BUS_ID")\n        for k in ("CC", "CXX", "CUDAHOSTCXX"):\n            env.pop(k, None)\n        if MERLIN_CC["mode"] == "gcc13" and kvv.get("MERLIN_CC") and kvv.get("MERLIN_CXX"):\n            env.update(CC=kvv["MERLIN_CC"], CXX=kvv["MERLIN_CXX"], CUDAHOSTCXX=kvv["MERLIN_CXX"])\n            lib13 = Path(kvv["MERLIN_CC"]).parents[1] / "lib"\n            if (lib13 / "libstdc++.so.6").exists():\n                env["LD_LIBRARY_PATH"] = f"{lib13}:{env.get(\'LD_LIBRARY_PATH\', \'\')}"\n        dp = getattr(eng["ML"], "distributed_ports", None)                   # Merlin\'s own --instances port scheme\n        if dp:\n            try:\n                ports = dp(port)\n                env.update({k: str(v) for k, v in (ports.items() if isinstance(ports, dict) else\n                                                   zip(("MASTER_PORT", "VLLM_PORT"), ports))})\n            except Exception:\n                env.update(MASTER_PORT=str(29500 + 10 * (port - 8100)), VLLM_PORT=str(29505 + 10 * (port - 8100)))\n        else:\n            env.update(MASTER_PORT=str(29500 + 10 * (port - 8100)), VLLM_PORT=str(29505 + 10 * (port - 8100)))\n        argv = plan.argv()\n        HELP[name] = L.help_check("merlin", argv[:argv.index("serve") + 1], env, argv)\n        return L.Server(name, argv, env, port, LOGS, gpus=gpus)\n\n\n    EAGLE_ENG = None\n\n\n    def eagle_install():\n        global EAGLE_ENG\n        if EAGLE_ENG is not None:\n            if EAGLE_ENG.get("failed"):\n                raise RuntimeError("EAGLE install failed earlier: " + EAGLE_ENG["failed"])\n            return EAGLE_ENG\n        if FAKE or EAGLE_PAY is None or EAGLE_DRAFT is None:\n            return None\n        try:\n            pay_root = TMP / "eagle_pay"\n            shutil.rmtree(pay_root, ignore_errors=True)\n            pay_root.mkdir()\n            with tarfile.open(EAGLE_PAY) as tf:\n                tf.extractall(pay_root, filter="data")\n            pay = pay_root / "eagle_offline"\n            rc, txt = run("install_eagle", f"bash {pay}/install_eagle_offline.sh {pay} {TMP}/eagle-off", timeout=2400)\n            shutil.rmtree(pay_root, ignore_errors=True)\n            if rc or "INSTALL OK" not in txt:\n                raise RuntimeError("EAGLE offline install failed (logs/install_eagle.log)")\n            env = dict(STOCK["env"])                                           # rev 3: the fork\'s own PATH / caches\n            env.update(PATH=f"{TMP}/eagle-off/bin:{env.get(\'PATH\', \'\')}", VLLM_CACHE_ROOT=f"{TMP}/eagle-cache",\n                       VLLM_USE_V2_MODEL_RUNNER="1", TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="4")\n            EAGLE_ENG = {"vllm": [f"{TMP}/eagle-off/bin/vllm", "serve"], "env": env, "name": "eagle"}\n            return EAGLE_ENG\n        except Exception as e:\n            EAGLE_ENG = {"failed": f"{type(e).__name__}: {e}"[:300]}\n            raise\n\n\n    PROFILER = lambda d: json.dumps({"profiler": "torch", "torch_profiler_dir": str(d), "torch_profiler_with_stack": False,\n                                     "ignore_frontend": True})\n\n\n    def stock_flags(model, port, *, tp=4, pp=1, seqs=20, spec=3, kv="fp8_e4m3", backend="FLASHINFER", prefix=True, mem=0.90,\n                    batched=4096, max_len=MAX_LEN, extra=(), profile_dir=None, spec_json=None):\n        f = [str(model), "--tensor-parallel-size", str(tp), "--kv-cache-dtype", kv, "--max-model-len", str(max_len),\n             "--max-num-seqs", str(seqs), "--max-num-batched-tokens", str(batched), "--gpu-memory-utilization", str(mem),\n             "--disable-custom-all-reduce", "--seed", "20261005", "--host", "127.0.0.1", "--port", str(port),\n             "--served-model-name", "q27", "--attention-backend", backend, "--trust-remote-code", "--language-model-only"]\n        if pp > 1:\n            f += ["--pipeline-parallel-size", str(pp)]\n        f += ["--enable-prefix-caching"] if prefix else ["--no-enable-prefix-caching"]\n        if spec_json:\n            f += ["--speculative-config", spec_json]\n        elif spec:\n            f += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": int(spec)})]\n        if profile_dir:\n            f += ["--profiler-config", PROFILER(profile_dir)]\n        return f + list(extra)\n\n\n    def stock_server(name, flags, port, env_extra=None, engine=None, gpus="0,1,2,3"):\n        eng = engine or STOCK\n        env = dict(eng.get("env") or STOCK["env"])\n        env.update(env_extra or {})\n        if "--profiler-config" in flags:\n            env["VLLM_CUSTOM_SCOPES_FOR_PROFILING"] = "1"                    # forward ranges for the per-forward graph verdict\n        if not FAKE:\n            HELP[name] = L.help_check(eng["name"], list(eng["vllm"]), env, flags)\n        return L.Server(name, list(eng["vllm"]) + list(flags), env, port, LOGS, gpus=gpus)\n    save()\n\n\nif not RERUN:\n    # ==================================================================== CELL 3 — distinct ARC prompts per stream (token ids)\n    from tokenizers import Tokenizer\n    TOK = Tokenizer.from_file(str(FP8 / "tokenizer.json"))\n    _tasks = json.load(open(ARC))\n    _ids = sorted(_tasks)\n\n\n    def _grid(g):\n        return "\\n".join("".join(str(v) for v in row) for row in g)\n\n\n    def _task_text(k):\n        t = _tasks[k]\n        parts = [f"## Task {k}"]\n        for i, ex in enumerate(t["train"]):\n            parts.append(f"Example {i + 1} input:\\n{_grid(ex[\'input\'])}\\nExample {i + 1} output:\\n{_grid(ex[\'output\'])}")\n        for ex in t["test"]:\n            parts.append(f"Test input:\\n{_grid(ex[\'input\'])}")\n        return "\\n\\n".join(parts)\n\n\n    # rev 3h (three GPT SMOKE audits): the load prompts are rendered by the PRODUCTION chat template exactly like the\n    # production notebooks (tokenizer.apply_chat_template(..., add_generation_prompt=True, enable_thinking=True,\n    # reasoning_effort="xhigh"), Experiment 1 make_q27x.py:343). With xhigh the template adds a system message with the\n    # reasoning instructions; rev 3g\'s hand-built prompt had none (= no-effort / medium rendering).\n    from transformers import AutoTokenizer\n    _HF = AutoTokenizer.from_pretrained(str(FP8), trust_remote_code=True)\n    _INTRO = "Find the transformation rule of each ARC task below and solve the test inputs.\\n\\n"\n    _PH = "\ue000E23_BODY\ue000"\n\n\n    def _render(effort):\n        return _HF.apply_chat_template([{"role": "user", "content": _INTRO + _PH}], tokenize=False, add_generation_prompt=True,\n                                       enable_thinking=True, reasoning_effort=effort)\n\n\n    _RX, _RM = _render("xhigh"), _render("medium")\n    assert _RX.count(_PH) == 1, "the production template must keep the user body verbatim"\n    _head_txt, _tail_txt = _RX.split(_PH)\n    assert "Reasoning effort is set to xhigh" in _head_txt, "production template did not render the xhigh instructions"\n    _HEAD = TOK.encode(_head_txt, add_special_tokens=False).ids\n    _TAIL = TOK.encode(_tail_txt, add_special_tokens=False).ids\n    for _t in ("<|im_start|>", "<|im_end|>", "<think>"):             # rev 3: the chat markers are single special tokens\n        _id = TOK.token_to_id(_t)\n        assert _id is not None and _id in (_HEAD + _TAIL), (_t, _id)\n    _tmpl = getattr(_HF, "chat_template", None) or ""\n    TPL_MSG = [{"role": "user", "content": _INTRO + _task_text(_ids[0])}]     # rev 3h: the chat-path template probe\n    TPL_IDS = TOK.encode(_HF.apply_chat_template(TPL_MSG, tokenize=False, add_generation_prompt=True, enable_thinking=True,\n                                                 reasoning_effort="xhigh"), add_special_tokens=False).ids\n    CHAT_TEMPLATE_FILE = TMP / "e23_production_chat_template.jinja"     # Merlin serves with this one too (rev 3h)\n    CHAT_TEMPLATE_FILE.write_text(_tmpl if isinstance(_tmpl, str) else json.dumps(_tmpl))\n    R["prompt_render"] = {"effort": "xhigh", "enable_thinking": True, "head_text": _head_txt, "tail_text": _tail_txt,\n                          "head_tokens": len(_HEAD), "tail_tokens": len(_TAIL),\n                          "xhigh_differs_from_medium": _RX != _RM,\n                          "template_sha256": hashlib.sha256(CHAT_TEMPLATE_FILE.read_bytes()).hexdigest(),\n                          "tokenizer_json_sha256": hashlib.sha256((FP8 / "tokenizer.json").read_bytes()).hexdigest()}\n    _TASK_IDS = {k: TOK.encode(_task_text(k) + "\\n\\n", add_special_tokens=False).ids for k in _ids}\n\n\n    def prompt(band, variant):\n        """~band tokens of DISTINCT ARC tasks (stream `variant` starts at its own task offset), chat-formatted, thinking on."""\n        body, i = [], (variant * 7) % len(_ids)\n        need = band - len(_HEAD) - len(_TAIL)\n        while len(body) < need:\n            body += _TASK_IDS[_ids[i % len(_ids)]]\n            i += 1\n        return _HEAD + body[:need] + _TAIL\n\n\n    # The 120k band starts at 116k: the earliest-admitted streams keep decoding while the last 120k prompts prefill, and\n    # 120k + 512 would leave only 10.5k new tokens (the window context is ~116k + decoded tokens ~= 120k; rev 3b).\n    BANDS = {"2k": 2048, "8k": 8192, "40k": 40000, "90k": 90000, "120k": 116000}\n    NSTREAM = 24\n    PROMPTS = {b: [prompt(n, v) for v in range(NSTREAM)] for b, n in BANDS.items()}\n    MAX_NEW = {b: MAX_LEN - n - 512 for b, n in BANDS.items()}       # a stream never ends inside a window\n    ORACLE_P = [PROMPTS["2k"][0], PROMPTS["2k"][1], PROMPTS["8k"][2]]\n    LONG_P = [PROMPTS["40k"][3]]                                      # one long-context greedy row (40k)\n    TF_P = {"40k": PROMPTS["40k"][5], "90k": PROMPTS["90k"][5]}        # teacher-forced long-context rows (NVFP4 vs FP8 KV)\n    R["prompts"] = {b: {"n": len(p), "len": len(p[0]), "sha": hashlib.sha256(json.dumps(p[:4]).encode()).hexdigest()[:12],\n                        "distinct_first_task": len({tuple(x[len(_HEAD):len(_HEAD) + 50]) for x in p})} for b, p in PROMPTS.items()}\n    log("prompts", json.dumps(R["prompts"]))\n    save()\n\n\nif not RERUN:\n    # ==================================================================== CELL 4 — the step runner (budget, GPU gate, fail-closed)\n    WIN = float(os.environ.get("E23_WIN_SEC") or {"SMOKE": 30.0, "TARGET": 60.0, "FULL": 120.0}[MODE])   # 120 s FULL windows\n    EST_SCALE = float(os.environ.get("E23_EST_SCALE") or 1.0) if FAKE else 1.0   # local harness only: walk the whole order\n    SM_TF = MODE == "SMOKE"                                            # SMOKE teacher-forced: the 40k row only\n    ROWS = {}            # oracle greedy rows by key (same weights)\n    TF = {}              # teacher-forced rows by key\n    NTOK = 128 if MODE == "FULL" else 64\n    LIVE = {"rec": None}                 # rev 3c: windows land in the step record before every save (a hard kill keeps them)\n    COMPILE_ERR = re.compile(r"ninja: build stopped|Error building extension|error: command \'.*(nvcc|g\\+\\+|gcc|c\\+\\+)\' failed|"\n                             r"GLIBCXX_[0-9.]+\' not found|unsupported GNU version|fatal error: ")\n    OOM_ERR = re.compile(r"OutOfMemoryError|CUDA out of memory|out of memory")\n    XID = re.compile(r"NVRM: Xid|\\bXid \\d+|CUDA error: an illegal memory access|illegal memory access was encountered|"\n                     r"unspecified launch failure|CUDA error: misaligned address")       # rev 3e: device faults, not any CUDA error\n\n\n    def step(sec, name, est_min, fn):\n        """One measured step: skipped when the budget cannot hold it, recorded on any exception, never PASS by default."""\n        rec = R["sections"].setdefault(sec, {}).setdefault(name, {"status": "NOT_RUN"})\n        R["order"].append(f"{sec}/{name}")\n        if ABORT["why"]:\n            rec.update(status="SKIPPED_ABORT", why=ABORT["why"])\n            log("---", sec, name, "SKIPPED_ABORT")\n            save()\n            return rec\n        if left() < est_min * EST_SCALE * 60 + 300:\n            rec.update(status="SKIPPED_BUDGET", left_min=round(left() / 60, 1), est_min=est_min)\n            log("---", sec, name, "SKIPPED_BUDGET", f"(est {est_min} min, left {left() / 60:.0f} min)")\n            save()\n            return rec\n        LIVE["rec"] = rec\n        t = time.time()\n        log(">>>", sec, name, f"(est {est_min} min, left {left() / 60:.0f} min)")\n        try:\n            out = fn() or {}\n            rec.update(out)\n            if rec.get("status") in (None, "NOT_RUN"):\n                rec["status"] = "DONE"\n        except Exception as e:\n            rec.update(status="ERROR", error=f"{type(e).__name__}: {e}"[:800], tb=traceback.format_exc()[-2500:])\n        rec["sec"] = round(time.time() - t, 1)\n        LIVE["rec"] = None\n        if ABORT["why"] and rec["status"] in ("DONE", "SERVER_DIED", "LAUNCH_FAILED", "BODY_ERROR"):\n            rec["status"] = "PARTIAL_HARD_STOP"\n        log("<<<", sec, name, rec.get("status"), f"{rec[\'sec\']:.0f}s")\n        save()\n        return rec\n\n\n    def with_servers(servers, timeout, body):\n        """Start, wait ready, record facts + memory, run body(servers), always kill; then the GPU-release gate on these\n        servers\' GPUs only; abort the lab if they stay held."""\n        out = {"servers": {}}\n        timeout = max(60, min(timeout, left() - 120))\n        try:\n            for s in servers:\n                s.start()\n            for s in servers:\n                ok, why = s.wait_ready(timeout)\n                out["servers"][s.name] = {"ready": ok, "why": why, **s.facts(), "argv_tail": s.argv[-44:], "gpus": s.gpus,\n                                          "help_check": HELP.get(s.name)}\n                if not ok:\n                    out["status"] = "LAUNCH_FAILED"\n                    out["servers"][s.name]["log_tail"] = s.log()[-4000:]\n                    _xl = [l for l in (s.log() or "").splitlines() if XID.search(l)][:8]   # rev 3e/3f: start-up / capture faults\n                    out["servers"][s.name].update(xid=bool(_xl), xid_lines=_xl)\n                    return out\n            mem0 = L.gpu_mem()\n            for s in servers:\n                out["servers"][s.name].update(gpu_mem_after_ready=mem0, host_rss_mb=s.rss_mb())\n            try:                             # rev 3: an exception in the body keeps everything measured so far\n                out.update(body(servers) or {})\n            except Exception as e:\n                out.update(status="BODY_ERROR", body_error=f"{type(e).__name__}: {e}"[:800], body_tb=traceback.format_exc()[-2000:])\n            for s in servers:\n                out["servers"][s.name].update(s.facts(), gpu_mem_end=L.gpu_mem(), host_rss_mb_end=s.rss_mb(),\n                                              alive_after_body=s.alive(), rc=None if s.alive() else s.proc.returncode,\n                                              xid_lines=[l for l in (s.log() or "").splitlines() if XID.search(l)][:8])\n                out["servers"][s.name]["xid"] = bool(out["servers"][s.name]["xid_lines"])\n                if not s.alive():\n                    out["status"] = "SERVER_DIED"\n                    out["servers"][s.name]["log_tail"] = s.log()[-4000:]\n            return out\n        finally:\n            for s in servers:\n                s.kill()\n            gpus = ",".join(sorted({g for s in servers for g in s.gpus.split(",")}))\n            rel = L.wait_gpu_release(180, fake=bool(FAKE), gpus=gpus)\n            if rel is False:                 # rev 3c: one more kill + 300 s before the lab aborts\n                for s in servers:\n                    s.kill()\n                rel = L.wait_gpu_release(300, fake=bool(FAKE), gpus=gpus)\n            if rel is None and NVSMI_OK:     # nvidia-smi worked at start: a failure now is not "released"\n                rel = False\n            out["gpu_released"] = rel\n            if rel is False:\n                ABORT["why"] = f"GPUs {gpus} still held after {[s.name for s in servers]}"\n                out["status"] = "FAIL_GPU_RELEASE"\n\n\n    def oracle(port, key=None, refs=()):\n        """Greedy rows (2k, 2k, 8k) + one 40k row; saved under `key` (a target-only oracle) or compared with ROWS[ref]."""\n        rows = L.greedy(port, ORACLE_P + LONG_P, n_tok=NTOK, timeout=max(120, min(900, left() / 6)))\n        clean = all(not r.get("error") and r.get("tokens") and not L.degenerate(r.get("text")) for r in rows)\n        if key and clean:            # rev 3d: an errored / empty / garbage reference is never stored (consumers: NO_ORACLE)\n            ROWS[key] = rows\n        elif key:\n            R["errors"].append(f"oracle {key}: reference not stored (error / empty / garbage row)")\n        res = {"garbage": [L.degenerate(r["text"]) for r in rows], "errors": [r["error"] for r in rows],\n               "text_head": [r["text"][:120] for r in rows]}\n        if not FAKE:                 # rev 3h: chat-path render vs the production xhigh render (recorded per arm)\n            res["template_probe"] = L.template_probe(port, TPL_MSG, TPL_IDS)\n        for ref in refs:\n            res["vs_" + ref] = L.compare(ROWS.get(ref), rows) if ROWS.get(ref) else {"status": "INCONCLUSIVE_NO_ORACLE"}\n        return res\n\n\n    def tforced(port, key=None, ref=None):\n        """Teacher-forced 40k / 90k prompt logprobs (last 256 positions); saved under `key` or compared with TF[ref].\n        rev 3c: on stock vLLM only on its own short launch (prompt logprobs = chunk x 248k-vocab logits: OOM at 0.90)."""\n        rows = {b: L.teacher_forced(port, ids) for b, ids in TF_P.items() if not (SM_TF and b != "40k")}\n        if key and L.tf_complete(rows):                                 # rev 3e: complete references only\n            TF[key] = rows\n        elif key:\n            R["errors"].append(f"teacher-forced {key}: reference not stored (incomplete band)")\n        if ref:\n            return {b: L.compare_tf((TF.get(ref) or {}).get(b), rows[b]) for b in rows}\n        return {b: {"n": (rows[b] or {}).get("n"), "error": (rows[b] or {}).get("error")} for b in rows}\n\n\n    def levels(port, band, lv, server=None, **kw):\n        """rev 3c: the guard includes admission (prefill) time; a band stops once KV-bound (the fit is recorded); a dead\n        server stops the band; every window is written into the live step record before save()."""\n        out, fit = {}, None\n        for n in lv:\n            k = f"{band}@{n}"\n            need = WIN + 90 + min(3600, n * BANDS[band] / 1500)\n            if ABORT["why"] or left() < need + 120:\n                out[k] = {"status": "SKIPPED_BUDGET", "need_sec": round(need)}\n            elif fit is not None:\n                out[k] = {"status": "SKIPPED_KV_BOUND", "streams_fit": fit}\n            elif server is not None and not server.alive():\n                out[k] = {"status": "SKIPPED_SERVER_DEAD"}\n            else:\n                try:                         # rev 3: one failing level never loses the others\n                    out[k] = L.steady(port, PROMPTS[band], n, secs=WIN, max_new=MAX_NEW[band], **kw)\n                except Exception as e:\n                    out[k] = {"status": "ERROR", "error": f"{type(e).__name__}: {e}"[:300]}\n                rm = out[k].get("running_max")\n                if out[k].get("status") == "CAPACITY_LIMITED" or (rm is not None and 0 < rm < n and\n                                                                  str(out[k].get("status")).startswith("INCONCLUSIVE")):\n                    fit = rm\n            if LIVE["rec"] is not None:\n                LIVE["rec"].setdefault("live_windows", {})[k] = out[k]\n            save()\n        return out\n    save()\n\n\nif not RERUN:\n    # ==================================================================== CELL 5 — the plan (dependency order; report = user numbering)\n    import concurrent.futures as CF\n    import signal\n    SM = MODE == "SMOKE"\n    TG = MODE == "TARGET"                            # rev 3h: targeted validation (FULL levels where it measures, SMOKE elsewhere)\n    LITE = SM or TG                                  # steps TARGET only needs as references / controls run their SMOKE form\n    DO_A = MODE == "FULL" or PART in ("A", "ALL")    # Merlin, P5, P4\n    DO_B = MODE == "FULL" or PART in ("B", "ALL")    # stock vLLM + EAGLE + PP4\n    PP4_RESERVE = 25 if MODE == "FULL" else 0                        # rev 3h: PP4 eager runs after the A/A fill\n    CAP_TP1 = [1, 2, 5, 10, 15, 20]                                  # seqs 4 x (1 + MTP 4) = 20 (V2 runner rounds to x5)\n    CAP_TP4 = [1, 2, 4, 5, 8, 10, 15, 20, 25, 30, 35, 40, 60, 80, 100, 120]   # seqs 24 x 5 = 120\n    KV_TP1, KV_TP4 = 8055029760, 16301686784                         # Merlin preflight\'s L4 (22.5 GiB) pins\n    P4_FUT = None\n    P4_PROC = {"p": None}\n    TP1_CFG = {"seqs": 4, "capture": CAP_TP1}\n    TP1_KW = {"prefill_tok_s": 250.0}                                # rev 3c: one L4 prefills ~250-400 tok/s (not TP4\'s 700)\n\n\n    def kit(name):\n        return str(KITS / name)\n\n\n    # ---- rev 3d (four GPT audits, REV3C_GPT4_PRIORITY_FIX_LIST #3): every arm\'s status reflects its correctness evidence.\n    # A request error / garbage / missing or non-clean comparison is never DONE; _done() (A/A) needs DONE.\n    CLEAN = ("EXACT", "NEAR_TIE")\n\n\n    def _windows(o):\n        """Every steady window of a step record (levels dicts + steady_multi replicas)."""\n        out = []\n        for v in (o.get("steady") or {}).values():\n            if isinstance(v, dict):\n                out.append(v)\n        for k in ("40k", "90k"):\n            for r in ((o.get(k) or {}).get("replicas") or []) if isinstance(o.get(k), dict) else []:\n                if isinstance(r, dict):\n                    out.append(r)\n        return out\n\n\n    def oracle_gate(o, need=(), extra=()):\n        """None when clean; otherwise the NO_GO / INCONCLUSIVE status. `need` = refs that must compare clean; `extra` =\n        further comparison dicts (replica greedy, 90k greedy) that must compare clean too. rev 3e (audits B-O1, C-F2):\n        garbage under load in any window and failed arithmetic probes (real servers; the fake has no chat) are NO_GO too."""\n        orc = o.get("oracle") or {}\n        if any(orc.get("errors") or []):\n            return "NO_GO_ORACLE_REQUEST_ERROR"\n        if any(orc.get("garbage") or []):\n            return "NO_GO_ORACLE_GARBAGE"\n        if any(w.get("status") == "FAIL_GARBAGE_UNDER_LOAD" for w in _windows(o)):\n            return "NO_GO_GARBAGE_UNDER_LOAD"\n        if not FAKE and isinstance(o.get("probes"), dict) and o["probes"].get("ok") is False:\n            return "NO_GO_SHORT_PROBE"\n        sts = [(orc.get("vs_" + k) or {}).get("status") for k in need] + [(x or {}).get("status") for x in extra]\n        if any(s == "FAIL_HIGH_MARGIN" for s in sts):\n            return "NO_GO_ORACLE"\n        if any(s not in CLEAN for s in sts):\n            return "INCONCLUSIVE_NO_ORACLE" if any(s in (None, "INCONCLUSIVE_NO_ORACLE") for s in sts) else "INCONCLUSIVE_ORACLE"\n        return None\n\n\n    def gated(o, need=(), extra=()):\n        g = oracle_gate(o, need, extra)\n        o["correctness"] = g or "CLEAN"\n        if g and o.get("status") in (None, "DONE"):\n            o["status"] = g\n        return o\n\n\n    # rev 3f (GPT rev 3e audit): Xid by recovery class (NVIDIA XID catalog). These need a GPU reset (driver / bus / GSP /\n    # uncontained ECC / NVLink / channel faults): the campaign stops. Application-class faults (an illegal memory access,\n    # Xid 13 / 31 / 43 ...) are recovered by the process restart every arm already does, and are continued only after the\n    # GPU health probe passes (all 4 GPUs, no reset pending, a clean CUDA matmul on each).\n    XID_RESET = {48, 61, 62, 63, 64, 68, 69, 74, 79, 92, 94, 95, 109, 110, 119, 120, 121, 136, 140, 143, 144, 145, 146, 147,\n                 148, 149, 150, 151, 154, 155, 156, 157}\n    HEALTH_CODE = ("import torch\\nn = torch.cuda.device_count()\\nok = n == 4\\nfor i in range(n):\\n"\n                   "    a = torch.ones(1024, 1024, device=f\'cuda:{i}\')\\n"\n                   "    ok = ok and float((a @ a).sum().item()) == float(1024 ** 3)\\n"\n                   "torch.cuda.synchronize()\\nprint(\'HEALTH_OK\' if ok else \'HEALTH_BAD\', n)\\n")\n\n\n    def gpu_health():\n        """rev 3f: nvidia-smi sees 4 GPUs with no reset pending, then a fresh process runs a checked matmul on each."""\n        if FAKE:\n            return {"ok": True, "fake": True}\n        out = {}\n        try:\n            q = subprocess.run(["nvidia-smi", "--query-gpu=index,reset_status.reset_required,memory.used", "--format=csv,noheader"],\n                               capture_output=True, text=True, timeout=60)\n            rows = [l for l in q.stdout.strip().splitlines() if l.strip()]\n            out["nvidia_smi_rows"] = rows\n            out["reset_required"] = any("yes" in l.lower() for l in rows) if q.returncode == 0 else None\n            if q.returncode != 0 or any("not a valid field" in l for l in rows):   # Kaggle\'s driver has no reset_status\n                out["reset_required"] = None\n                out["reset_status_field"] = "unsupported (health = 4 GPUs + the CUDA probe)"\n                rows = [l for l in subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"], capture_output=True,\n                                                  text=True, timeout=60).stdout.strip().splitlines() if l.strip()]\n            out["gpus"] = len(rows)\n            h = subprocess.run([STOCK["py"], "-c", HEALTH_CODE], capture_output=True, text=True, timeout=300, env=STOCK["env"])\n            out["cuda_probe"] = (h.stdout + h.stderr)[-300:]\n            out["ok"] = out["gpus"] == 4 and out["reset_required"] is not True and h.returncode == 0 and "HEALTH_OK" in h.stdout\n        except Exception as e:\n            out.update(ok=False, error=f"{type(e).__name__}: {e}"[:200])\n        return out\n\n\n    def fault_gate(texts, arm):\n        """rev 3f: classify GPU fault lines in `texts` (server logs, bench output). Returns None (no fault), \'CONTINUE\'\n        (application fault + healthy GPUs: this arm is NO_GO, the run goes on) or \'ABORT\' (a reset-class Xid, or the GPUs\n        are not healthy after the fault: the campaign stops)."""\n        lines = [l for t in texts for l in (t or "").splitlines() if XID.search(l)]\n        if not lines:\n            return None\n        # "NVRM: Xid (PCI:0000:00:04): 79, pid=..." (the PCI address comes first) or a bare "Xid 79"\n        nums = {int(x) for l in lines for x in re.findall(r"Xid \\([^)]*\\):\\s*(\\d+)|\\bXid\\s+(\\d+)\\b", l) for x in x if x}\n        rec = {"arm": arm, "lines": lines[:5], "xid": sorted(nums)}\n        if nums & XID_RESET:\n            rec["class"] = "RESET_REQUIRED"\n            ABORT["why"] = f"GPU fault needing a reset (Xid {sorted(nums & XID_RESET)}) in {arm}"\n        else:\n            rec["health"] = gpu_health()\n            rec["class"] = "APP_RESTART" if rec["health"].get("ok") else "UNHEALTHY_AFTER_FAULT"\n            if not rec["health"].get("ok"):\n                ABORT["why"] = f"GPUs unhealthy after a GPU fault in {arm}"\n        R["setup"].setdefault("gpu_faults", []).append(rec)\n        return "ABORT" if ABORT["why"] and rec["class"] != "APP_RESTART" else "CONTINUE"\n\n\n    def taint_check(o):\n        """rev 3e/3f: a GPU fault line in a server log (with_servers records `xid_lines`, at launch too) marks THIS arm\n        NO_GO_GPU_FAULT; a reset-class Xid or unhealthy GPUs afterwards abort the campaign (fault_gate); an application-class\n        fault with healthy GPUs lets the run go on, every later record labelled `after_gpu_fault`."""\n        texts = {n: "\\n".join((v or {}).get("xid_lines") or []) for n, v in (o.get("servers") or {}).items() if (v or {}).get("xid")}\n        prior = list(R["setup"].get("gpu_fault_arms") or [])\n        if prior:\n            o["after_gpu_fault"] = prior\n        if texts:\n            R["setup"]["gpu_fault_arms"] = prior + list(texts)\n            o["gpu_fault"] = list(texts)\n            o["gpu_fault_class"] = fault_gate(list(texts.values()), ",".join(texts))\n            o["correctness"] = "NO_GO_GPU_FAULT"\n            if o.get("status") in (None, "DONE"):\n                o["status"] = "NO_GO_GPU_FAULT"\n        return o\n\n\n    def _killpg(p):\n        try:\n            os.killpg(p.pid, signal.SIGKILL)\n        except Exception:\n            pass\n\n\n    # ---- #2 host-staged all-reduce: all four GPUs free, before any server\n    def s2():\n        out = str(OUT / "p5_bench.json")\n        Path(out).unlink(missing_ok=True)\n        cmd = [STOCK["py"], kit("bench_host_ar.py"), "--so", kit("p5_host_ar.so"), "--out", out,\n               "--timeout-s", "900" if LITE else "3000"] + (["--quick"] if LITE else [])\n        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=STOCK["env"],\n                             start_new_session=True)\n        try:\n            so, _ = p.communicate(timeout=1000 if LITE else 3300)\n            rc, tail = p.returncode, (so or "")[-2000:]\n        except subprocess.TimeoutExpired:\n            _killpg(p)                                    # rev 3c: the bench\'s children die with it\n            so, _ = p.communicate()\n            rc, tail = -9, (so or "")[-1000:]\n        res = json.loads(Path(out).read_text()) if Path(out).is_file() else {}\n        fg = fault_gate([so or ""], "p5_bench")          # rev 3f: the standalone benches go through the same fault gate\n        if fg:\n            return {"rc": rc, "summary": res.get("summary"), "tail": tail, "status": "NO_GO_GPU_FAULT", "gpu_fault_class": fg}\n        rel = L.wait_gpu_release(180, fake=bool(FAKE), gpus="0,1,2,3")\n        if rel is None and NVSMI_OK:\n            rel = False\n        o = {"rc": rc, "summary": res.get("summary"), "tail": tail, "gpu_released": rel}\n        if rel is False:                  # rev 3d (#7): Merlin must never start on GPUs P5 still holds\n            ABORT["why"] = f"P5 GPU release unverified: {rel}"\n            return {**o, "status": "FAIL_GPU_RELEASE"}\n        passed = rc == 0 and ((res.get("summary") or {}).get("all_checks_pass") is True or bool(FAKE))\n        return {**o, "status": "DONE" if passed else ("CHECK_FAILED" if rc in (0, 1) else "BENCH_ERROR")}\n\n\n    def s4_thread():\n        out = str(OUT / "p4_bench.json")\n        Path(out).unlink(missing_ok=True)\n        per_shape = 420 if SM else 600\n        cmd = [STOCK["py"], kit("bench_fp8_gemm.py"), "--so", kit("p4_sm89_fp8.so"), "--out", out, "--gpu", "3",\n               "--weights", "real", "--model-dir", str(FP8), "--timeout-per-shape", str(per_shape)] + \\\n              (["--m", "1,16,80", "--shapes", "gate_up_proj,down_proj"] if SM else [])\n        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=STOCK["env"],\n                             start_new_session=True)\n        P4_PROC["p"] = p\n        try:\n            so, _ = p.communicate(timeout=2 * per_shape + 300 if SM else 5 * per_shape + 300)\n            rc, tail = p.returncode, (so or "")[-2000:]\n        except subprocess.TimeoutExpired:\n            _killpg(p)                                    # rev 3c: per-shape children hold GPU 3 otherwise\n            so, _ = p.communicate()\n            rc, tail = -9, (so or "")[-1000:]\n        res = json.loads(Path(out).read_text()) if Path(out).is_file() else {}\n        fg = fault_gate([so or ""], "p4_bench_gpu3")     # rev 3f: same fault gate as the servers\n        if fg:\n            return {"rc": rc, "tail": tail, "status": "NO_GO_GPU_FAULT", "gpu_fault_class": fg}\n        kinds = {n: ((v or {}).get("weights_source") or {}).get("kind") for n, v in (res.get("shapes") or {}).items()}\n        synth = [n for n, k in kinds.items() if k == "synthetic"]\n        no_result = [n for n, k in kinds.items() if k is None]\n        o = {"rc": rc, "summary_per_m": res.get("summary_per_m"), "gates": res.get("gates") or res.get("gate"), "tail": tail,\n             "weights_kind": {n: ((v or {}).get("weights_source") or {}).get("kind") for n, v in (res.get("shapes") or {}).items()}}\n        if synth:                          # rev 3d (#10): only the real Qwen3.8-27B FP8 weights + scales count\n            return {**o, "status": "CHECK_FAILED", "why": "SYNTHETIC_WEIGHTS_NOT_ALLOWED", "shapes": synth}\n        if no_result:                      # rev 3e: a crashed / timed-out shape is not "synthetic"\n            return {**o, "status": "CHECK_FAILED" if rc in (0, 1) else "BENCH_ERROR", "why": "SHAPE_NO_RESULT", "shapes": no_result}\n        passed = rc == 0 and res.get("all_ok") is True\n        return {**o, "status": "DONE" if passed else ("CHECK_FAILED" if rc in (0, 1) else "BENCH_ERROR")}\n\n\n    def join_p4():\n        """P4 holds GPU 3; join it (bounded) before any step that needs GPU 3; a GPU 3 that stays held aborts the lab."""\n        global P4_FUT\n        if P4_FUT is None:\n            return\n        try:\n            res = P4_FUT.result(timeout=max(60, min(3600, left() - 600)))\n        except Exception as e:\n            res = {"status": "ERROR", "error": repr(e)[:300]}\n            if P4_PROC["p"] is not None:\n                _killpg(P4_PROC["p"])\n        rel = L.wait_gpu_release(300, fake=bool(FAKE), gpus="3")\n        if rel is None and NVSMI_OK:\n            rel = False\n        res["gpu3_released"] = rel\n        R["sections"].setdefault("4_sm89_fp8_gemm", {})["p4_bench_gpu3"] = res\n        P4_FUT = None\n        if rel is False:\n            ABORT["why"] = "GPU 3 still held after the P4 bench"\n        save()\n\n\n    # ---- Merlin launch with the two recoveries the audits asked for: a GCC 13 compile failure -> the system GCC once;\n    # an OOM (launch or mid-body) on TP1 -> Merlin\'s own TP1 profile values. Never after the hard stop.\n    def _logs_of(o):\n        return " ".join((v or {}).get("log_tail", "") for v in (o.get("servers") or {}).values())\n\n\n    def merlin_run(make, timeout, body, tp1=False):\n        o = with_servers(make(), timeout, body)\n        tries = []\n        for _ in range(2):\n            if ABORT["why"] or o.get("status") not in ("LAUNCH_FAILED", "SERVER_DIED"):\n                break\n            logs = _logs_of(o)\n            if COMPILE_ERR.search(logs) and MERLIN_CC["mode"] == "gcc13":\n                MERLIN_CC["mode"] = "system"\n                why = "gcc13_compile_error -> system GCC"\n            elif tp1 and (OOM_ERR.search(logs) or o.get("status") == "LAUNCH_FAILED") and TP1_CFG["seqs"] is not None:\n                TP1_CFG.update(seqs=None, capture=None)\n                why = "TP1 OOM / launch failure -> profile defaults (seqs 2, capture [1,2,5,10])"\n            else:\n                break\n            tries.append({"why": why, "status": o.get("status"), "servers": o.get("servers")})\n            o = with_servers(make(), timeout, body)\n        if tries:\n            o["earlier_attempts"] = tries\n        o["merlin_compiler"] = MERLIN_CC["mode"]\n        return taint_check(o)\n\n\n    # ---- #1 Merlin TP1 (GPU 0) with #4 P4 on GPU 3 meanwhile; replicas; Merlin TP4\n    def _tp1_body(ss):\n        s = ss[0]\n        port = s.port\n        o = {"probes": L.chat_probes(port), "oracle": oracle(port, key="merlin")}\n        g90 = L.greedy(port, [PROMPTS["90k"][4]], n_tok=NTOK, timeout=max(300, min(1800, left() / 6)))\n        if all(not r.get("error") and r.get("tokens") and not L.degenerate(r.get("text")) for r in g90):\n            ROWS["merlin_90k"] = g90                                     # rev 3e: clean references only\n        o["greedy_90k"] = {"garbage": [L.degenerate(r["text"]) for r in g90], "errors": [r["error"] for r in g90]}\n        o["steady"] = levels(port, "40k", [2] if LITE else [2, 3, 4], server=s, **TP1_KW)\n        o["steady"].update(levels(port, "90k", [] if LITE else [1, 2, 3, 4], server=s, **TP1_KW))   # SMOKE: 90k = the greedy row\n        o["steady"].update(levels(port, "120k", [1] if LITE else [1, 2], server=s, **TP1_KW))\n        if s.alive():                    # Merlin\'s fork budgets prompt-logprob memory; still last (nothing measured after it)\n            o["teacher_forced"] = tforced(port, key="merlin")\n        g90 = {"status": "EXACT"} if not (any(o["greedy_90k"]["garbage"]) or any(o["greedy_90k"]["errors"])) else \\\n            {"status": "FAIL_HIGH_MARGIN" if any(o["greedy_90k"]["garbage"]) else "INCONCLUSIVE"}\n        return gated(o, extra=(g90,))\n\n\n    def s1_tp1():\n        g = merlin_sha_gate()\n        if g:\n            return g\n        o = merlin_run(lambda: [merlin_server("merlin_tp1", "27b-tp1-sm89-24g", "0", 8100, KV_TP1, **TP1_CFG)], 2400,\n                       _tp1_body, tp1=True)\n        o["config"] = ("profile defaults (seqs 2, capture [1,2,5,10])" if TP1_CFG["seqs"] is None\n                       else "seqs 4, capture " + str(CAP_TP1))\n        return o\n\n\n    def _tp1_level(band, default):\n        rec = R["sections"]["1_tp1x4_pp4"].get("a_merlin_tp1_gpu0") or {}\n        st = rec.get("steady") or {}\n        ok = []\n        for k, v in st.items():\n            if k.startswith(band + "@") and v.get("status") in ("STEADY", "CAPACITY_LIMITED"):\n                ok.append(int(k.split("@")[1]) if v["status"] == "STEADY" else int(v.get("running_max") or 0))\n        ok = [x for x in ok if x > 0]\n        return max(ok) if ok else default\n\n\n    def s1_rep(n, bands=None):\n        def f():\n            g = merlin_sha_gate()\n            if g:\n                return g\n            lv40, lv90 = _tp1_level("40k", 2), _tp1_level("90k", 1)\n\n            def make():\n                return [merlin_server(f"merlin_tp1_rep{i}", "27b-tp1-sm89-24g", str(i), 8100 + i, KV_TP1, **TP1_CFG)\n                        for i in range(n)]\n\n            def body(ss):\n                o = {"per_replica_level": {"40k": lv40, "90k": lv90},\n                     "replica_greedy": {s.name: L.compare(ROWS.get("merlin")[:1], L.greedy(s.port, ORACLE_P[:1], n_tok=NTOK))\n                                        if ROWS.get("merlin") else {"status": "INCONCLUSIVE_NO_ORACLE"} for s in ss}}\n                o["40k"] = L.steady_multi([(s.port, PROMPTS["40k"][i * 6:], lv40) for i, s in enumerate(ss)], secs=WIN,\n                                          max_new=MAX_NEW["40k"], **TP1_KW)\n                if "90k" in (bands or ("40k", "90k")):\n                    o["90k"] = L.steady_multi([(s.port, PROMPTS["90k"][i * 6:], lv90) for i, s in enumerate(ss)], secs=WIN,\n                                              max_new=MAX_NEW["90k"], **TP1_KW)\n                return gated(o, extra=tuple(o["replica_greedy"].values()))\n            return merlin_run(make, 2400, body)\n        return f\n\n\n    def s1_tp4():\n        g = merlin_sha_gate()\n        if g:\n            return g\n\n        def body(ss):\n            s = ss[0]\n            port = s.port\n            o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("merlin",))}\n            g90 = L.greedy(port, [PROMPTS["90k"][4]], n_tok=NTOK, timeout=max(300, min(1800, left() / 6)))\n            o["greedy_90k_vs_tp1"] = L.compare(ROWS.get("merlin_90k"), g90) if ROWS.get("merlin_90k") else \\\n                {"status": "INCONCLUSIVE_NO_ORACLE"}\n            # rev 3h (third GPT SMOKE audit): the long-context sweep the plan needs; a KV-bound band stops at its fit\n            o["steady"] = levels(port, "40k", [8] if SM else [8, 16, 20, 24], server=s)\n            o["steady"].update(levels(port, "90k", [] if SM else [8, 12, 16, 20], server=s))\n            o["steady"].update(levels(port, "120k", [8] if SM else [8, 12, 16], server=s))\n            if s.alive():\n                o["teacher_forced_vs_tp1"] = tforced(port, ref="merlin")\n                o["teacher_forced_vs_tp1_note"] = "info only: same engine, same weights (TP4 vs TP1 numerics)"\n            return gated(o, need=("merlin",), extra=(o["greedy_90k_vs_tp1"],))\n        o = merlin_run(lambda: [merlin_server("merlin_tp4", "27b-tp4-sm89-24g", "0,1,2,3", 8100, KV_TP4, seqs=24,\n                                              capture=CAP_TP4)], 2400, body)\n        if o.get("status") == "LAUNCH_FAILED" and not ABORT["why"]:      # the profile\'s own capture list (validated to 40)\n            o2 = merlin_run(lambda: [merlin_server("merlin_tp4_fallback", "27b-tp4-sm89-24g", "0,1,2,3", 8100, KV_TP4,\n                                                   seqs=8)], 2400, body)\n            o2["first_attempt"] = {k: o[k] for k in ("status", "servers") if k in o}\n            o2["config"] = "fallback seqs 8 with the profile\'s capture list (graphs cover every batch)"\n            return o2\n        return o\n\n\n    # ---- #6 MTP OFF / K1 / K3 (FP8, stock vLLM 0.32, seqs 20); K3 = G0 of #3 (FlashInfer, PIECEWISE pinned), profiled.\n    # rev 3c: every K3 launch (main + A/A reps) has the same command (profiler config included); only the main one profiles.\n    def s6(depth, profile=False, tag=None):\n        def f():\n            pdir = TMP / f"e23_profile_mtp{depth}_{tag or \'main\'}" if depth == 3 else None\n            extra = ["--compilation-config", json.dumps({"cudagraph_mode": "PIECEWISE"})] if depth == 3 else []\n            srv = stock_server(tag or f"fp8_mtp{depth}", stock_flags(FP8, 8200, spec=depth, extra=extra, profile_dir=pdir), 8200)\n\n            def body(ss):\n                s = ss[0]\n                port = s.port\n                o = {"probes": L.chat_probes(port),\n                     "oracle": oracle(port, key=("fp8_mtp0" if depth == 0 else ("fp8_g0" if depth == 3 else None)) if not tag else None,\n                                      refs=() if depth == 0 and not tag else ("fp8_mtp0",))}\n                if depth == 0:\n                    o["steady"] = {} if LITE else levels(port, "40k", [8, 12, 16, 20], server=s)\n                    if not LITE:\n                        o["steady"].update(levels(port, "90k", [12, 20], server=s))\n                elif depth == 1:\n                    o["steady"] = levels(port, "40k", [8, 12, 16, 20], server=s)\n                else:\n                    o["steady"] = levels(port, "40k", [16] if LITE else [8, 12, 16, 20], server=s)\n                    if not LITE:\n                        o["steady"].update(levels(port, "90k", [8, 12, 20], server=s))\n                        o["steady"].update(levels(port, "120k", [8], server=s))\n                if profile and s.alive():\n                    o["profile"] = L.profiled_load(port, PROMPTS["40k"], 16, pdir, max_new=MAX_NEW["40k"], spec=True,\n                                                     prefill_tok_s=450.0)   # rev 3e: settle limit ~1540 s (cold cache)\n                    _keep_trace(pdir, tag or f"mtp{depth}")\n                if pdir:\n                    shutil.rmtree(pdir, ignore_errors=True)\n                return gated(o, need=() if depth == 0 and not tag else ("fp8_mtp0",))\n            o = taint_check(with_servers([srv], 1800, body))\n            if profile:\n                o["graph_proof"] = graph_proof(o, expect="PIECEWISE")\n            return o\n        return f\n\n\n    def graph_proof(o, expect=None):\n        """rev 3d (#6): a graph verdict counts only from a settled, decode-only, spec-active profile with per-forward\n        annotations and no graph-mode downgrade line; `expect` = the mode this arm must show (G0 = PIECEWISE control)."""\n        pr = o.get("profile") or {}\n        tr = pr.get("trace") or {}\n        downgr = [l for v in (o.get("servers") or {}).values() for l in ((v or {}).get("graph_downgrade_lines") or [])]\n        if pr.get("status"):                                  # INCONCLUSIVE_PREFILL_NOT_SETTLED / _IN_PROFILE / _NO_SPEC\n            return {"status": pr["status"]}\n        if not tr or tr.get("error"):\n            return {"status": "INCONCLUSIVE_GRAPH", "why": tr.get("error") or "no trace"}\n        if tr.get("graph_verdict") in (None, "UNKNOWN", "UNPROVEN"):\n            return {"status": "INCONCLUSIVE_GRAPH", "why": f"verdict {tr.get(\'graph_verdict\')} ({tr.get(\'verdict_basis\')})"}\n        # rev 3e: a downgrade counts only when vLLM fell back to PIECEWISE / NONE (FULL_AND_PIECEWISE -> FULL_DECODE_ONLY\n        # still runs full verify graphs)\n        to_low = [l for l in downgr if re.search(r"(?:=|to)\\s*[\'\\"]?(?:CUDAGraphMode\\.)?(PIECEWISE|NONE)\\b", l)]\n        if to_low:\n            return {"status": "DOWNGRADED", "why": "graph mode downgraded to PIECEWISE / NONE", "lines": to_low[:3],\n                    "verdict": tr.get("graph_verdict")}\n        v = tr["graph_verdict"]\n        if expect and v != expect:\n            return {"status": "MISMATCH_EXPECTED", "verdict": v, "expected": expect}\n        return {"status": "PROVEN_" + v, "verdict": v, "per_forward": [tr.get("per_forward_median_graph_launches"),\n                                                                       tr.get("per_forward_median_kernel_launches")]}\n\n\n    def _keep_trace(pdir, tag):\n        """rev 3c: the rank-0 trace is kept in OUT (the verdict code has never seen a real 0.32 trace)."""\n        try:\n            fs = sorted(Path(pdir).rglob("*.pt.trace.json*"), key=lambda p: p.stat().st_size)\n            f0 = next((p for p in fs if re.search(r"rank0\\.|rank[-_]?0[._]|tp0_", p.name)), fs[0] if fs else None)\n            if f0 is not None and f0.stat().st_size < 400e6:\n                (OUT / "traces").mkdir(exist_ok=True)\n                shutil.copy2(f0, OUT / "traces" / f"{tag}_{f0.name}")\n        except Exception as e:\n            R["errors"].append(f"keep_trace {tag}: {e!r}"[:200])\n\n\n    def _site():\n        return subprocess.run([STOCK["py"], "-c", "import sysconfig;print(sysconfig.get_paths()[\'purelib\'])"],\n                              capture_output=True, text=True).stdout.strip()\n\n\n    PATCH_ORIG = {}\n\n\n    def _patch(reverse=False):\n        """forward = back up flashinfer.py, dry-run, apply (restore the backup on a failed apply); reverse = restore the\n        backup by copy and verify it against the sha recorded BEFORE patching (rev 3c: not against the backup itself)."""\n        target = Path(_site()) / "vllm/v1/attention/backends/flashinfer.py"\n        bak = TMP / "flashinfer.py.e23_orig"\n        if reverse:\n            if not bak.is_file():\n                return {"applied": False, "error": "no backup"}\n            shutil.copy2(bak, target)\n            now = hashlib.sha256(target.read_bytes()).hexdigest()\n            return {"applied": now == PATCH_ORIG.get("sha"), "restored_sha": now[:16]}\n        shutil.copy2(target, bak)\n        PATCH_ORIG["sha"] = hashlib.sha256(target.read_bytes()).hexdigest()\n        diff = P60532.split("diff --git a/vllm/v1/attention/backends/flashinfer.py")[1]\n        Path(TMP / "pr60532_vllm_only.diff").write_text("diff --git a/vllm/v1/attention/backends/flashinfer.py" + diff.split("\\ndiff --git")[0] + "\\n")\n        base = ["patch", "-p1", "-d", _site(), "--batch", "--forward", "--no-backup-if-mismatch", "-i", str(TMP / "pr60532_vllm_only.diff")]\n        dry = subprocess.run(base + ["--dry-run"], capture_output=True, text=True, stdin=subprocess.DEVNULL, timeout=120)\n        if dry.returncode:\n            return {"rc": dry.returncode, "dry_run": (dry.stdout + dry.stderr)[-600:], "applied": False}\n        p = subprocess.run(base, capture_output=True, text=True, stdin=subprocess.DEVNULL, timeout=120)\n        if p.returncode:\n            shutil.copy2(bak, target)\n        return {"rc": p.returncode, "out": (p.stdout + p.stderr)[-600:], "applied": p.returncode == 0,\n                "orig_sha": PATCH_ORIG["sha"][:16]}\n\n\n    def s3(tag, backend, graph, patch=False):\n        def f():\n            pre = None\n            try:                          # rev 3c: the patch is inside the try, so any exception still restores the file\n                pre = _patch() if patch and not FAKE else None\n                if pre and not pre["applied"]:\n                    return {"status": "PATCH_FAILED", "patch": pre}\n                pdir = TMP / f"e23_profile_{tag}"\n                extra = ["--compilation-config", json.dumps({"cudagraph_mode": graph})] if graph else []\n                srv = stock_server(f"fp8_{tag}", stock_flags(FP8, 8200, backend=backend, extra=extra, profile_dir=pdir), 8200)\n\n                def body(ss):\n                    s = ss[0]\n                    port = s.port\n                    o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("fp8_mtp0", "fp8_g0")), "patch": pre}\n                    o["steady"] = levels(port, "40k", [16], server=s)\n                    if s.alive():\n                        o["profile"] = L.profiled_load(port, PROMPTS["40k"], 16, pdir, max_new=MAX_NEW["40k"], spec=True,\n                                                     prefill_tok_s=450.0)   # rev 3e: settle limit ~1540 s (cold cache)\n                        _keep_trace(pdir, tag)\n                    shutil.rmtree(pdir, ignore_errors=True)\n                    return gated(o, need=("fp8_mtp0",))\n                o = taint_check(with_servers([srv], 1800, body))\n                o["graph_proof"] = graph_proof(o, expect="FULL")\n                g0 = ((R["sections"].get("6_mtp_depth") or {}).get("mtp3_eq_G0_piecewise") or {}).get("graph_proof") or {}\n                gs = g0.get("status")\n                if gs in ("PROVEN_FULL", "MISMATCH_EXPECTED"):   # the PIECEWISE-pinned control read FULL: the verdict logic lies\n                    o["graph_proof"] = {"status": "INCONCLUSIVE_GRAPH", "why": f"G0 control broken: {gs}", "own": o["graph_proof"]}\n                elif gs != "PROVEN_PIECEWISE" and str(o["graph_proof"].get("status", "")).startswith("PROVEN_"):\n                    # rev 3f (GPT rev 3e audit): without the PIECEWISE control a FULL verdict is not proven\n                    o["graph_proof"] = {"status": "INCONCLUSIVE_GRAPH", "why": f"no G0 PIECEWISE control ({gs})",\n                                        "own": o["graph_proof"]}\n                return o\n            finally:\n                if patch and not FAKE and (TMP / "flashinfer.py.e23_orig").is_file() and PATCH_ORIG.get("sha"):\n                    rv = _patch(reverse=True)\n                    R["sections"].setdefault("3_full_graph", {})[f"{tag}_patch_revert"] = rv\n                    if not rv.get("applied"):\n                        ABORT["why"] = "PR #60532 revert failed: the stock venv is no longer pristine"\n        return f\n\n\n    # ---- #5 NVFP4 KV. Teacher-forced on its own short launch per KV dtype (same settings both sides, prompt-logprob\n    # memory fits); target-only = correctness + one census level; NVFP4 + MTP3 = the residency census + streams increase.\n    def s5_tf(kv, key=None, ref=None, name=None):\n        """Teacher-forced rows on a short launch (same settings for every KV dtype). With `ref`, each band must be a\n        complete comparison (compare_tf: every expected position valid on both sides) for DONE."""\n        def f():\n            srv = stock_server(name or f"tf_{kv}", stock_flags(FP8, 8200, spec=0, kv=kv, seqs=2, batched=512, mem=0.80), 8200)\n\n            def body(ss):\n                o = {"teacher_forced": tforced(ss[0].port, key=key, ref=ref)}\n                if ref:\n                    st = [v.get("status") for v in o["teacher_forced"].values()]\n                    o["status"] = "DONE" if st and all(x == "MEASURED" for x in st) else "INCONCLUSIVE_TF"\n                elif key and not L.tf_complete(TF.get(key) or {}):\n                    o["status"] = "INCONCLUSIVE_TF"                    # rev 3e: every position of every band valid\n                return o\n            return taint_check(with_servers([srv], 1800, body))\n        return f\n\n\n    def _num(x, default):\n        return x if isinstance(x, (int, float)) else default\n\n\n    NV_BANDS = {"40k", "90k"}          # rev 3h: the NVFP4 gate needs both long-context bands on both TF arms\n\n\n    def nvfp4_gate():\n        """rev 3e (four agent reports on rev 3d): an FP8-vs-FP8 A/A is exactly 0 on a deterministic engine, so the\n        tolerance is calibrated against the production KV dtype instead. BF16 KV (kv auto) is the teacher-forced\n        reference; FP8 KV and NVFP4 KV are both measured against it; NVFP4 passes when it is not much worse than FP8 KV:\n        mean |dlogprob| <= max(3 x FP8, 0.02), p99 <= max(3 x FP8, 0.2), rank-1 agreement >= FP8 - 0.02 (screening policy,\n        not an ARC accuracy proof). Plus: target-only DONE with a clean oracle vs fp8_mtp0, every TF step DONE (complete)."""\n        sec = R["sections"].get("5_nvfp4_kv") or {}\n        tgt, ref, f8, nv = (sec.get(k) or {} for k in ("nvfp4_target_only", "tf_bf16kv_reference", "tf_fp8kv_vs_bf16",\n                                                        "tf_nvfp4_vs_bf16"))\n        out = {"target_only": tgt.get("status"), "target_oracle": ((tgt.get("oracle") or {}).get("vs_fp8_mtp0") or {}).get("status"),\n               "tf_bf16_reference": ref.get("status"), "tf_fp8": f8.get("status"), "tf_nvfp4": nv.get("status"), "bands": {}}\n        if out["target_only"] != "DONE" or out["target_oracle"] not in CLEAN:\n            return False, {**out, "why": "TARGET_ONLY_NOT_CLEAN"}\n        if out["tf_bf16_reference"] != "DONE" or out["tf_fp8"] != "DONE" or out["tf_nvfp4"] != "DONE":\n            return False, {**out, "why": "TEACHER_FORCED_INCOMPLETE"}\n        for arm, row in (("tf_fp8kv_vs_bf16", f8), ("tf_nvfp4_vs_bf16", nv)):   # rev 3h (GPT SMOKE audits): both bands\n            bands = row.get("teacher_forced") or {}\n            miss = sorted(NV_BANDS - set(bands))\n            if miss:\n                return False, {**out, "why": "MISSING_REQUIRED_TF_BANDS", "arm": arm, "missing": miss}\n            if any((bands[b] or {}).get("status") != "MEASURED" or (bands[b] or {}).get("n") != 256 for b in NV_BANDS):\n                return False, {**out, "why": "INVALID_TF_BAND", "arm": arm}\n        ok = True\n        for b, r in (nv.get("teacher_forced") or {}).items():\n            a = (f8.get("teacher_forced") or {}).get(b) or {}\n            if r.get("status") != "MEASURED" or a.get("status") != "MEASURED":\n                ok = False\n                out["bands"][b] = {"pass": False, "why": "band not MEASURED", "nvfp4": r.get("status"), "fp8": a.get("status")}\n                continue\n            lim = {"mean_max": max(3 * _num(a.get("mean_abs_dlogprob"), 9e9), 0.02),\n                   "p99_max": max(3 * _num(a.get("p99_abs_dlogprob"), 9e9), 0.2),\n                   "rank1_min": _num(a.get("rank1_agreement"), 2.0) - 0.02}\n            good = (_num(r.get("mean_abs_dlogprob"), 9e9) <= lim["mean_max"] and _num(r.get("p99_abs_dlogprob"), 9e9) <= lim["p99_max"]\n                    and _num(r.get("rank1_agreement"), -1.0) >= lim["rank1_min"])\n            keys = ("n", "rank1_agreement", "mean_abs_dlogprob", "p99_abs_dlogprob", "max_abs_dlogprob")\n            out["bands"][b] = {"nvfp4": {k: r.get(k) for k in keys}, "fp8": {k: a.get(k) for k in keys}, "limits": lim, "pass": good}\n            ok = ok and good\n        if not out["bands"]:\n            return False, {**out, "why": "NO_TF_BANDS"}\n        return ok, {**out, "why": None if ok else "TF_OUTSIDE_TOLERANCE"}\n\n\n    def s5(spec, tag=None):\n        def f():\n            gate = None\n            if spec:\n                ok, gate = nvfp4_gate()\n                if not ok:\n                    return {"status": "SKIPPED_CORRECTNESS_GATE", "gate": gate}\n            srv = stock_server(tag or f"nvfp4_mtp{spec}", stock_flags(FP8, 8200, spec=spec, kv="nvfp4", seqs=24), 8200)\n\n            def body(ss):\n                s = ss[0]\n                port = s.port\n                o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("fp8_mtp0",) if spec == 0 else ("fp8_g0", "fp8_mtp0"))}\n                if spec == 0:\n                    g = oracle_gate(o, need=("fp8_mtp0",))           # rev 3d: not only FAIL_HIGH_MARGIN stops it\n                    if g:\n                        o.update(status=g, correctness=g)\n                        return o\n                    o["steady"] = levels(port, "120k", [8], server=s) if LITE else levels(port, "90k", [12], server=s)\n                else:\n                    o["steady"] = levels(port, "90k", [12, 16, 20, 24], server=s)\n                    o["steady"].update(levels(port, "120k", [12, 16, 20], server=s))\n                if gate:\n                    o["nvfp4_gate"] = gate                               # rev 3e: the admitted evidence stays on record\n                return gated(o, need=("fp8_mtp0",))\n            return taint_check(with_servers([srv], 1800, body))\n        return f\n\n\n    # ---- #4 INT4 (stock vLLM, RedHat W4A16). rev 3d (#4): the same-checkpoint target-only oracle (MTP off) first; the\n    # INT4 MTP3 control (against Merlin\'s low-M INT4 engine) and PP4 compare with it, never with themselves or with FP8\n    def s4_int4_ref():\n        if INT4 is None:\n            return {"status": "SKIPPED_REQUIRED_ARTIFACT", "why": "RedHat INT4 checkpoint not attached"}\n        srv = stock_server("int4_mtp0_ref", stock_flags(INT4, 8200, spec=0, seqs=8), 8200)\n        return taint_check(with_servers([srv], 1800, lambda ss: gated({"oracle": oracle(ss[0].port, key="int4_mtp0")})))\n\n\n    def s4_int4():\n        if INT4 is None:\n            return {"status": "SKIPPED_REQUIRED_ARTIFACT", "why": "RedHat INT4 checkpoint not attached"}\n        srv = stock_server("int4_mtp3_s20", stock_flags(INT4, 8200, spec=3, seqs=20), 8200)\n\n        def body(ss):\n            s = ss[0]\n            port = s.port\n            o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("int4_mtp0",)),\n                 "note": "engine + checkpoint comparison vs Merlin (RedHat INT4 on stock vLLM), not a patch-only A/B"}\n            o["steady"] = levels(port, "40k", [8, 20], server=s)\n            o["steady"].update(levels(port, "90k", [12], server=s))\n            return gated(o, need=("int4_mtp0",))\n        return taint_check(with_servers([srv], 1800, body))\n\n\n    # ---- #1e PP4 branch (stock vLLM, INT4 RedHat). rev 3d (#5): eager first (#63505: compiled PP faults / gibberish on the\n    # hybrid GDN model), compiled as a later separate arm only after eager passed; staged fail-fast, each stage against\n    # int4_mtp0: short probes -> 2k -> 8k -> 40k greedy, then throughput\n    def s1_pp4(eager):\n        def f():\n            if INT4 is None:\n                return {"status": "SKIPPED_REQUIRED_ARTIFACT", "why": "RedHat INT4 checkpoint not attached"}\n            if not ROWS.get("int4_mtp0"):\n                return {"status": "SKIPPED_NO_ORACLE", "why": "no clean int4_mtp0 reference"}\n            extra = ["--enforce-eager"] if eager else []\n            srv = stock_server("int4_pp4_eager" if eager else "int4_pp4_compiled",\n                               stock_flags(INT4, 8200, tp=1, pp=4, spec=0, seqs=16, extra=extra), 8200)\n\n            def body(ss):\n                s = ss[0]\n                port = s.port\n                o = {"probes": L.chat_probes(port), "stages": {},\n                     "note": "PP4 uses stock vLLM + RedHat INT4 (Merlin\'s fork has no PP profile); #63505 open on 0.32; "\n                             "the V1 runner forces async scheduling off under PP, so these rows understate PP"}\n                if not o["probes"]["ok"]:\n                    o["status"] = "NO_GO_SHORT_PROBE"\n                    return o\n                ref = ROWS["int4_mtp0"]\n                for name, idx in (("2k", [0]), ("8k", [2]), ("40k", [3])):      # ORACLE_P + LONG_P order of oracle()\n                    rows = L.greedy(port, [(ORACLE_P + LONG_P)[i] for i in idx], n_tok=NTOK, timeout=max(120, min(900, left() / 6)))\n                    cmp_ = L.compare([ref[i] for i in idx], rows)\n                    bad = any(r.get("error") or not r.get("tokens") or L.degenerate(r.get("text")) for r in rows)\n                    o["stages"][name] = {"compare": cmp_, "bad_output": bad, "text_head": [r["text"][:120] for r in rows]}\n                    if bad or cmp_.get("status") not in CLEAN:\n                        o["status"] = f"NO_GO_{name}"\n                        o["correctness"] = o["status"]\n                        return o\n                o["steady"] = levels(port, "40k", [8] if SM else [8, 16], server=s)\n                return gated(o)                  # rev 3e: garbage under load in the throughput windows is NO_GO too\n            return taint_check(with_servers([srv], 1800, body))\n        return f\n\n\n    def pp4():\n        rec = step("1_tp1x4_pp4", "e_pp4_int4_eager", 15 if SM else 25, s1_pp4(True))\n        if rec.get("status") == "DONE" and not SM:                  # rev 3g: compiled PP4 only in FULL\n            step("1_tp1x4_pp4", "f_pp4_int4_compiled", 15 if SM else 25, s1_pp4(False))\n        else:\n            R["sections"].setdefault("1_tp1x4_pp4", {})["f_pp4_int4_compiled"] = {"status": "SKIPPED_EAGER_NOT_PROVEN",\n                                                                                  "eager_status": rec.get("status")}\n\n\n    # ---- EAGLE-4 feasibility: a BF16-KV FP8-weight target-only oracle first (same KV dtype as tree verify), then EAGLE\n    def s7_kvauto_oracle():\n        srv = stock_server("fp8_kvauto_mtp0", stock_flags(FP8, 8300, spec=0, kv="auto", seqs=16), 8300)\n        return taint_check(with_servers([srv], 1800, lambda ss: gated({"oracle": oracle(ss[0].port, key="fp8_kvauto_mtp0")})))\n\n\n    def s7_eagle():\n        eng = eagle_install()\n        if eng is None:\n            return {"status": "SKIPPED_REQUIRED_ARTIFACT", "why": "EAGLE env / drafter not attached (or fake mode)"}\n        spec = json.dumps({"method": "eagle_harness", "model": str(EAGLE_DRAFT),\n                           "method_config": {"mode": "static", "node_top_k": 4, "tree_depth": 14, "max_verify_width": 16}})\n\n        def body(ss):\n            s = ss[0]\n            port = s.port\n            o = {"probes": L.chat_probes(port), "oracle": oracle(port, refs=("fp8_kvauto_mtp0", "fp8_mtp0"))}\n            o["steady"] = levels(port, "8k", [8, 16], server=s)\n            o["steady"].update(levels(port, "40k", [8], server=s))\n            return gated(o, need=("fp8_kvauto_mtp0",))       # same KV dtype (BF16) as EAGLE\'s tree verify\n        # rev 3d (#12): the official recipe first (--linear-backend flashinfer_cutedsl; CuTe-DSL may lack sm_89 kernels),\n        # then the default linear backend; rev 3c: prefix caching ON first, OFF last and labelled confounded\n        tries = [("eagle4_fp8_cutedsl", ["--linear-backend", "flashinfer_cutedsl"], True),\n                 ("eagle4_fp8", [], True),\n                 ("eagle4_fp8_noprefix", [], False)]\n        attempts = []\n        deadline = time.time() + 40 * 60                    # rev 3e: all EAGLE variants share one 40-min window\n        for name, extra, prefix in tries:\n            if ABORT["why"]:\n                return {"status": "SKIPPED_ABORT", "attempts": attempts}\n            if time.time() > deadline or left() < 20 * 60:\n                return {"status": "SKIPPED_EAGLE_DEADLINE", "attempts": attempts}\n            o = taint_check(with_servers([stock_server(name, stock_flags(FP8, 8300, seqs=16, kv="auto", backend="CUSTOM",\n                                                                         prefix=prefix, spec_json=spec, extra=extra),\n                                                       8300, engine=eng)],\n                                         max(300, min(1800, deadline - time.time())), body))\n            o.update(variant=name, linear_backend=extra[1] if extra else "default", prefix_caching=prefix)\n            if not prefix:\n                o["confounded"] = "prefix caching OFF (E19: spec decode + prefix off flips tokens)"\n            if o.get("status") in (None, "DONE"):\n                o["earlier_attempts"] = attempts\n                return o\n            # rev 3e: a launch failure, a crash after ready, a body error or a NO_GO all fall through to the next variant\n            attempts.append({k: o[k] for k in ("variant", "status", "correctness", "servers") if k in o})\n        return {"status": "NO_GO_ALL_EAGLE_VARIANTS", "attempts": attempts}\n\n\n    MERLIN_SHA = {"th": None, "ok": None, "bad": []}\n\n\n    def _merlin_sha():\n        """rev 3d (#14): full sha256 of every Merlin checkpoint file against the P1 manifest (size-only is not identity);\n        runs after the prefetch so the bytes come from the page cache."""\n        mt = json.loads((MERLIN / "q27_e23_p1_manifest.json").read_text())\n        bad = []\n        for n, (size, sha) in mt["files"].items():\n            if n == ".gitattributes" or not sha:\n                continue\n            h = hashlib.sha256()\n            with open(MERLIN / n, "rb") as fh:\n                for blk in iter(lambda: fh.read(16 << 20), b""):\n                    h.update(blk)\n            if h.hexdigest() != sha:\n                bad.append(n)\n        MERLIN_SHA.update(ok=not bad, bad=bad)\n        R["inputs"]["merlin_sha_ok"] = not bad\n        try:\n            save()\n        except Exception:\n            pass                          # rev 3e: a save race with another thread never turns a good hash into FAIL\n\n\n    def merlin_sha_gate():\n        """rev 3e (audits A/B/C/D): every Merlin step. The install runs first (it overlaps the prefetch + hash); a mismatch\n        fails every Merlin step; a hash still running after the wait lets the step run, labelled UNVERIFIED."""\n        if FAKE or MERLIN_SHA["th"] is None:\n            return None\n        try:\n            merlin_install()\n        except Exception:\n            pass                          # the step itself reports the install failure\n        MERLIN_SHA["th"].join(timeout=max(60, min(900, left() - 900)))\n        if MERLIN_SHA["ok"] is False:\n            return {"status": "FAIL_CHECKPOINT_SHA", "bad": MERLIN_SHA["bad"]}\n        if MERLIN_SHA["ok"] is None:\n            R["setup"]["merlin_sha"] = "UNVERIFIED_TIMEOUT"\n        return None\n\n\n    def start_prefetch():\n        pref = []\n        if DO_A and MERLIN:\n            th = prefetch_bg([MERLIN], "merlin")\n            pref.append(th)\n            if not FAKE:\n                def _after():\n                    th.join()\n                    try:\n                        _merlin_sha()\n                    except Exception as e:\n                        MERLIN_SHA.update(ok=False, bad=[f"{type(e).__name__}: {e}"[:200]])\n                MERLIN_SHA["th"] = threading.Thread(target=_after, daemon=True)\n                MERLIN_SHA["th"].start()\n        if DO_B:\n            pref.append(prefetch_bg([FP8], "fp8"))\n        return pref\n\n\n    # ---- run order (audit5 rev 3b: realistic estimates, trimmed levels the plan does not need, PP4 before the A/A fill).\n    # SMOKE A: P5, Merlin TP1 (+P4), Merlin TP4, Merlin TP1 x4 (40k). SMOKE B: mtp0 (oracle) -> mtp3/G0 -> G3 -> TF FP8 ref\n    # -> NVFP4 target-only -> TF NVFP4 -> [kvauto oracle + EAGLE] -> PP4. FULL: P5, Merlin a-d, mtp0, INT4 control, mtp1,\n    # mtp3/G0, G1, G2, G3, TF FP8 ref, NVFP4 target-only, TF NVFP4, NVFP4 + MTP3, PP4, then A/A repeats fill the time.\n    PREF = []\n    if TG:\n        # rev 3h TARGET (user choice 2026-10-30, option ka): GPU-validate the three GPT SMOKE-audit fixes before FULL.\n        # A: P5 (fixed bound check), Merlin TP1 (the TP1 reference rows) -> Merlin TP4 long-context sweep on xhigh prompts.\n        # B: FP8 MTP0 oracle -> G0 / G3 graph proof (new scope parser) -> corrected NVFP4 gate (40k + 90k, top-1 ids).\n        TE = int                     # TARGET estimates (not part of the FULL main-sequence estimate below)\n        if DO_A:\n            step("2_host_allreduce", "p5_bench", TE(12), s2)\n        PREF = start_prefetch() if not ABORT["why"] else []\n        if DO_A:\n            step("1_tp1x4_pp4", "a_merlin_tp1_gpu0", TE(30), s1_tp1)\n            step("1_tp1x4_pp4", "d_merlin_tp4", TE(70), s1_tp4)\n        if DO_B:\n            step("6_mtp_depth", "mtp0_target_only", TE(12), s6(0))\n            step("6_mtp_depth", "mtp3_eq_G0_piecewise", TE(18), s6(3, profile=True))\n            step("3_full_graph", "G3_flashinfer_pr60532", TE(18), s3("G3", "FLASHINFER", "FULL_AND_PIECEWISE", patch=True))\n            step("5_nvfp4_kv", "tf_bf16kv_reference", TE(14), s5_tf("auto", key="bf16kv_tf"))\n            step("5_nvfp4_kv", "tf_fp8kv_vs_bf16", TE(14), s5_tf("fp8_e4m3", ref="bf16kv_tf"))\n            step("5_nvfp4_kv", "nvfp4_target_only", TE(22), s5(0))\n            step("5_nvfp4_kv", "tf_nvfp4_vs_bf16", TE(14), s5_tf("nvfp4", ref="bf16kv_tf"))\n            _ok, _gate = nvfp4_gate()\n            R["sections"].setdefault("5_nvfp4_kv", {})["nvfp4_gate_verdict"] = {"status": "PASS" if _ok else "STOP", **_gate}\n            save()\n    if DO_A and not TG:\n        step("2_host_allreduce", "p5_bench", 12 if SM else 30, s2)      # before the prefetch (host memory + PCIe quiet)\n    if not TG:\n        PREF = start_prefetch() if not ABORT["why"] else []\n    if DO_A and not TG:\n        if not ABORT["why"] and left() > (40 if SM else 50) * 60 + 300:     # rev 3e: never after the P5 release abort\n            P4_FUT = CF.ThreadPoolExecutor(1).submit(s4_thread)\n        step("1_tp1x4_pp4", "a_merlin_tp1_gpu0", 30 if SM else 50, s1_tp1)\n        join_p4()\n        if not SM:\n            step("1_tp1x4_pp4", "b_merlin_tp1_x2", 25, s1_rep(2))\n            step("1_tp1x4_pp4", "c_merlin_tp1_x4", 25, s1_rep(4))\n        step("1_tp1x4_pp4", "d_merlin_tp4", 30 if SM else 62, s1_tp4)\n        if SM:\n            step("1_tp1x4_pp4", "c_merlin_tp1_x4", 20, s1_rep(4, bands=("40k",)))\n    if DO_B and not TG:\n        if INT4:\n            PREF.append(prefetch_bg([INT4], "int4"))\n        step("6_mtp_depth", "mtp0_target_only", 12 if SM else 45, s6(0))\n        step("4_sm89_fp8_gemm", "int4_mtp0_target_only_ref", 12, s4_int4_ref)   # rev 3d (#4): SMOKE_B + FULL, early\n        if not SM:\n            step("4_sm89_fp8_gemm", "int4_mtp3_s20_control", 35, s4_int4)\n            step("6_mtp_depth", "mtp1", 30, s6(1))\n        step("6_mtp_depth", "mtp3_eq_G0_piecewise", 18 if SM else 62, s6(3, profile=True))\n        if not SM:\n            step("3_full_graph", "G1_triton_full_and_piecewise", 25, s3("G1", "TRITON_ATTN", "FULL_AND_PIECEWISE"))\n            step("3_full_graph", "G2_triton_full_decode_only", 25, s3("G2", "TRITON_ATTN", "FULL_DECODE_ONLY"))\n        step("3_full_graph", "G3_flashinfer_pr60532", 18 if SM else 25, s3("G3", "FLASHINFER", "FULL_AND_PIECEWISE", patch=True))\n        # rev 3e: BF16 KV = the teacher-forced reference; FP8 KV (production) and NVFP4 KV are measured against it\n        step("5_nvfp4_kv", "tf_bf16kv_reference", 11 if SM else 15, s5_tf("auto", key="bf16kv_tf"))\n        if not SM:                     # rev 3g: the FP8 calibration only feeds the FULL NVFP4 gate\n            step("5_nvfp4_kv", "tf_fp8kv_vs_bf16", 15, s5_tf("fp8_e4m3", ref="bf16kv_tf"))\n        step("5_nvfp4_kv", "nvfp4_target_only", 22 if SM else 30, s5(0))\n        step("5_nvfp4_kv", "tf_nvfp4_vs_bf16", 11 if SM else 15, s5_tf("nvfp4", ref="bf16kv_tf"))\n        if not SM:\n            step("5_nvfp4_kv", "nvfp4_mtp3", 75, s5(3))\n        if os.environ.get("E23_EAGLE_FULL") == "1" and left() >= (11 + 40 + 30) * 60:   # rev 3g: not in the 8 h SMOKE      # rev 3e: EAGLE window 40 min, PP4 eager 30 min after it\n            step("7_eagle4", "fp8_kvauto_target_only_oracle", 11, s7_kvauto_oracle)\n            step("7_eagle4", "eagle4_fp8_tp4", 40, s7_eagle)\n    if MODE == "FULL":      # A/A repeats of the decisive comparisons fill whatever time is left (greedy; nothing reserved after)\n        def _done(path):              # rev 3d (#3): DONE and a clean correctness record (rev 3e: a GPU fault elsewhere only labels)\n            sec, name = path.split("/")\n            rec = (R["sections"].get(sec) or {}).get(name) or {}\n            return rec.get("status") == "DONE" and rec.get("correctness") == "CLEAN"\n\n        def AA(i):       # only configurations whose main run completed (a failed config is not relaunched)\n            c = [(f"mtp3_control_rep{i}", 60, s6(3, tag=f"fp8_mtp3_rep{i}"), "6_mtp_depth/mtp3_eq_G0_piecewise"),\n                 (f"merlin_tp1_x4_rep{i}", 30, s1_rep(4), "1_tp1x4_pp4/c_merlin_tp1_x4"),\n                 (f"nvfp4_mtp3_rep{i}", 75, s5(3, tag=f"nvfp4_mtp3_rep{i}"), "5_nvfp4_kv/nvfp4_mtp3")]\n            return [(n, e, fn) for n, e, fn, main in c if _done(main)]\n        for i in range(1, 7):\n            todo = [(n, e, fn) for n, e, fn in AA(i) if left() >= e * EST_SCALE * 60 + 300 + PP4_RESERVE * 60]\n            if not todo or ABORT["why"]:\n                break\n            for n, e, fn in todo:\n                step("8_aa_repeats", n, e, fn)\n    if DO_B and not TG:      # rev 3h (third GPT SMOKE audit): PP4 last, after the A/A fill (its 8k fault cannot touch them)\n        pp4()\n\n\nif not RERUN:\n    # ==================================================================== CELL 6 — report\n    def _flat(sec):\n        rows = []\n        for name, rec in (R["sections"].get(sec) or {}).items():\n            if not isinstance(rec, dict):\n                continue\n            for k, v in (rec.get("steady") or {}).items():\n                rows.append([sec, name, k, v.get("status"), v.get("tok_s"), v.get("running_mean"), v.get("mean_acceptance_length"),\n                             v.get("step_ms"), v.get("kv_usage_max"), v.get("preemptions")])\n            for k in ("40k", "90k"):\n                if isinstance(rec.get(k), dict) and "replicas" in rec[k]:\n                    rows.append([sec, name, k + " aggregate", rec[k]["status"], rec[k].get("tok_s"), rec[k].get("running_mean"),\n                                 None, None, None, None])\n        return rows\n\n\n    lines = ["| section | step | window | status | tok/s | running | accept len | step ms | kv max | preempt |",\n             "|---|---|---|---|---|---|---|---|---|---|"]\n    for sec in sorted(R["sections"]):\n        for r_ in _flat(sec):\n            lines.append("| " + " | ".join("-" if x is None else str(x) for x in r_) + " |")\n    st = ["| section | step | status | correctness | windows steady/all | oracle | probes | KV tokens | ready s | graph capture | "\n          "trace verdict | graph proof |", "|---|---|---|---|---|---|---|---|---|---|---|---|"]\n    for sec in sorted(R["sections"]):\n        for name, rec in R["sections"][sec].items():\n            if not isinstance(rec, dict):\n                continue\n            o = rec.get("oracle") or {}\n            vs = ", ".join(f"{k[3:]}={v.get(\'status\')}" for k, v in o.items() if k.startswith("vs_") and isinstance(v, dict))\n            if isinstance(o.get("template_probe"), dict):          # rev 3h: chat-path render vs production xhigh\n                vs += f"; template={o[\'template_probe\'].get(\'status\')}"\n            sv = next(iter((rec.get("servers") or {}).values()), {}) if rec.get("servers") else {}\n            tr_ = (rec.get("profile") or {}).get("trace") or {}\n            gv = tr_.get("graph_verdict") or tr_.get("error") or "-"\n            if tr_.get("graph_verdict"):\n                gv += (f" (g/fw {tr_.get(\'per_forward_median_graph_launches\')}, k/fw {tr_.get(\'per_forward_median_kernel_launches\')}, "\n                       f"compute-eager/fw {tr_.get(\'per_forward_median_compute_eager\')}, scopes {tr_.get(\'forwards\')})")\n            wins = list((rec.get("steady") or {}).values())\n            ws = f"{sum(1 for w in wins if w.get(\'status\') in (\'STEADY\', \'CAPACITY_LIMITED\'))}/{len(wins)}" if wins else "-"\n            st.append(f"| {sec} | {name} | {rec.get(\'status\')} | {rec.get(\'correctness\', \'-\')} | {ws} | "\n                      f"{vs or (\'saved\' if o else \'-\')} | {(rec.get(\'probes\') or {}).get(\'ok\', \'-\')} | {sv.get(\'kv_tokens\', \'-\')} | "\n                      f"{sv.get(\'ready_sec\', \'-\')} | {\',\'.join(sv.get(\'graph_capture_modes\') or []) or \'-\'} | {gv} | "\n                      f"{(rec.get(\'graph_proof\') or {}).get(\'status\', \'-\')} |")\n    R["help_check"] = HELP\n    NOTES = ["Notes (rev 3d):",\n             "- MEASURED teacher-forced rows are measurements; NVFP4 passes only when its drift from BF16 KV is within 3x the "\n             "production FP8 KV drift (mean, p99) and rank-1 within 0.02 (nvfp4_gate; a screening policy, not an ARC accuracy proof).",\n             "- The P5 bench is a 983,040 B 4-rank graph microbenchmark against NCCL\'s 343.88 us; a microbenchmark gain is not "\n             "an engine gain until the collective runs inside vLLM.",\n             "- The INT4 control vs Merlin is an engine + checkpoint comparison (RedHat INT4 on stock vLLM), not a patch-only A/B.",\n             "- tok/s is compared with the PDF baselines (INT4 MTP3 332, FP8 SSM16 312 at 90k) only with the same workload, actual "\n             "running streams (running_mean) and accepted-output accounting (generation_tokens_total).",\n             "- A graph verdict counts only as PROVEN_* (settled, decode-only, spec-active, annotated forwards, G0 control "\n             "PIECEWISE, else UNCONTROLLED); arms with a GPU fault line (NO_GO_GPU_FAULT): " + str(R["setup"].get("gpu_fault_arms")) +\n             "; Merlin checkpoint sha: " + str(R["inputs"].get("merlin_sha_ok", R["setup"].get("merlin_sha")))]\n    R["report_md"] = "\\n".join(st) + "\\n\\n" + "\\n".join(lines) + "\\n\\n" + "\\n".join(NOTES)\n    R["minutes"] = round((time.time() - T_START) / 60, 1)\n    save()\n    print(R["report_md"])\n    print("E23", MODE, PART, "done in", R["minutes"], "min | abort:", ABORT["why"])\n\n\n'
NB = {"FULL": NBF, "TARGET_A": NBF.replace('os.environ.get("Q27X_E23_MODE", "FULL")', 'os.environ.get("Q27X_E23_MODE", "TARGET")', 1)
      .replace('os.environ.get("Q27X_E23_PART", "ALL")', 'os.environ.get("Q27X_E23_PART", "A")', 1)}
NB["TARGET_B"] = NB["TARGET_A"].replace('os.environ.get("Q27X_E23_PART", "A")', 'os.environ.get("Q27X_E23_PART", "B")', 1)
(T / "fake_vllm_server.py").write_text('"""E23 copy of the Experiment 14 harness fake server (only text_for differs).\nFake vLLM 0.32.0 OpenAI server for the VMS CPU harness (no model, no GPU). Launched by the REAL VMS cell with the REAL\n`vllm serve` flags (argv after the script = [model, --flag, value, ...]).\n- startup log lines the notebook parses: "GPU KV cache size: N tokens", "Maximum concurrency for M tokens per request: Xx"\n- GET /health, GET /metrics (Prometheus: vllm:num_requests_running, vllm:generation_tokens_total,\n  vllm:spec_decode_num_accepted_tokens_total, vllm:spec_decode_num_drafts_total, vllm:kv_cache_usage_perc,\n  vllm:num_preemptions_total)\n- POST /v1/completions stream=true: SSE chunks; temperature 0 = deterministic text from the prompt; ignore_eos keeps\n  streaming until max_tokens or the client disconnects (= vLLM\'s abort on disconnect)\n- running is capped at max_num_seqs x data_parallel_size (the rest wait)\nFAKE_SPEED = tokens per second per request."""\nimport json, os, sys, threading, time, hashlib\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\nargv = sys.argv[1:]\n\n\ndef arg(name, default=None):\n    return argv[argv.index(name) + 1] if name in argv else default\n\n\nMODEL = argv[0] if argv and not argv[0].startswith("--") else "?"\nPORT = int(arg("--port", 8011))\nSEQS = int(arg("--max-num-seqs", 16))\nDP = int(arg("--data-parallel-size", 1))\nTP = int(arg("--tensor-parallel-size", 4))\nCAP = SEQS * DP\nSPEC = json.loads(arg("--speculative-config", "null") or "null")\nSPEED = float(os.environ.get("FAKE_SPEED", 400))\nKV = 900000 if "int4" in MODEL.lower() or "w4a16" in MODEL.lower() else 600000\nprint(f"fake vllm serve {\' \'.join(argv)}", flush=True)\nassert TP * DP <= 4, "more GPUs than the box has"\ntime.sleep(0.5)\n_lb = arg("--linear-backend", "auto")\n# vLLM 0.32.0 on SM 8.9: auto (and marlin) -> MarlinFP8ScaledMMLinearKernel; triton -> TritonFp8BlockScaledMMKernel\nprint(f"INFO [__init__.py] Selected {\'TritonFp8BlockScaledMMKernel\' if _lb == \'triton\' else \'MarlinFP8ScaledMMLinearKernel\'} for Fp8LinearMethod", flush=True)\nif os.environ.get("VLLM_ALLREDUCE_USE_FLASHINFER_PCIE_IPC") == "1":   # vLLM 0.32.0 flashinfer_pcie_ipc_all_reduce.py log line\n    print("INFO Initialized FlashInfer PCIe IPC all-reduce for TP4, hidden_dim=5120, dtype=torch.bfloat16, max_tokens=64.", flush=True)\nprint(f"INFO GPU KV cache size: {KV:,} tokens", flush=True)\nprint(f"INFO Maximum concurrency for {arg(\'--max-model-len\', \'106496\')} tokens per request: {KV / int(arg(\'--max-model-len\', 106496)):.2f}x", flush=True)\nLOCK = threading.Lock()\nRUN, WAIT = set(), []\nCNT = {"gen": 0.0, "acc": 0.0, "drf": 0.0}\nCTX = {}\nT_READY = time.time() + 0.5\n\n\n_SYL = ["ka", "lo", "mi", "ne", "ru", "sa", "ti", "vo", "ze", "pa", "qu", "ri", "bel", "dor", "fen", "gul"]\n_VOCAB = [a + b + c for a in _SYL for b in _SYL for c in _SYL]       # 4,096 alphabetic words\n_TOKID = {w + " ": 1000 + i for i, w in enumerate(_VOCAB)}          # one token id per word\n\n\ndef text_for(prompt, n, greedy):\n    """E23 copy: varied words (one token = one word), so the lab\'s garbage check sees healthy text; deterministic for\n    greedy, the prompt tail + time otherwise (the E14 original emitted a repeating 64-char hex string)."""\n    import random\n    h = hashlib.sha256((json.dumps(prompt[-64:]) + ("" if greedy else str(time.time()))).encode()).hexdigest()\n    rng = random.Random(h)\n    return [_VOCAB[rng.randrange(len(_VOCAB))] + " " for _ in range(n)]\n\n\nclass H(BaseHTTPRequestHandler):\n    protocol_version = "HTTP/1.1"\n\n    def log_message(self, *a):\n        pass\n\n    def _send(self, body, code=200, ctype="application/json"):\n        b = body.encode() if isinstance(body, str) else body\n        self.send_response(code)\n        self.send_header("Content-Type", ctype)\n        self.send_header("Content-Length", str(len(b)))\n        self.send_header("Connection", "close")\n        self.end_headers()\n        self.close_connection = True\n        self.wfile.write(b)\n\n    def do_GET(self):\n        if self.path == "/health":\n            return self._send("", 200 if time.time() >= T_READY else 503)\n        if self.path == "/metrics":\n            with LOCK:\n                n, ctx = len(RUN), sum(CTX.values())\n                lines = [f\'vllm:num_requests_running{{engine="0"}} {n}\', f\'vllm:num_requests_waiting{{engine="0"}} {len(WAIT)}\',\n                         f\'vllm:generation_tokens_total{{engine="0"}} {CNT["gen"]}\',\n                         f\'vllm:spec_decode_num_accepted_tokens_total{{engine="0"}} {CNT["acc"]}\',\n                         f\'vllm:spec_decode_num_drafts_total{{engine="0"}} {CNT["drf"]}\',\n                         f\'vllm:kv_cache_usage_perc{{engine="0"}} {min(0.99, ctx / KV):.4f}\',\n                         \'vllm:num_preemptions_total{engine="0"} 0\',\n                         f\'vllm:prompt_tokens_total{{engine="0"}} {CNT.get("prompt", 0.0)}\',\n                         f\'vllm:prefix_cache_queries_total{{engine="0"}} {CNT.get("pq", 0.0)}\',\n                         f\'vllm:prefix_cache_hits_total{{engine="0"}} {CNT.get("ph", 0.0)}\',\n                         f\'vllm:prompt_tokens_cached_total{{engine="0"}} {CNT.get("ph", 0.0)}\']\n            return self._send("\\n".join(lines) + "\\n", ctype="text/plain")\n        return self._send("{}", 404)\n\n    def do_POST(self):\n        n = int(self.headers.get("Content-Length", 0))\n        req = json.loads(self.rfile.read(n) or b"{}")\n        if self.path != "/v1/completions":\n            return self._send("{}", 404)\n        rid = f"r{time.time_ns()}"\n        prompt, max_t = req.get("prompt") or [], int(req.get("max_tokens", 16))\n        greedy = float(req.get("temperature", 1.0)) == 0.0\n        nlp = int(req.get("logprobs") or 0)\n        with LOCK:\n            WAIT.append(rid)\n        while True:\n            with LOCK:\n                if WAIT and WAIT[0] == rid and len(RUN) < CAP:\n                    WAIT.pop(0)\n                    RUN.add(rid)\n                    CTX[rid] = len(prompt)\n                    break\n            time.sleep(0.01)\n        self.send_response(200)\n        self.send_header("Content-Type", "text/event-stream")\n        self.send_header("Transfer-Encoding", "chunked")\n        self.send_header("Connection", "close")\n        self.end_headers()\n        self.close_connection = True\n        toks = text_for(prompt, max_t, greedy)\n        k = int((SPEC or {}).get("num_speculative_tokens", 0))\n        step = (1 + 0.6 * k) if k else 1.0\n        try:\n            sent = 0\n            while sent < max_t:\n                chunk = toks[sent:sent + 8]\n                sent += len(chunk)\n                d = {"choices": [{"text": "".join(chunk), "index": 0,\n                                  "finish_reason": "length" if sent >= max_t else None}]}\n                if req.get("return_token_ids"):      # vLLM 0.32.0: token_ids = the delta ids of this chunk (ints)\n                    d["choices"][0]["token_ids"] = [_TOKID.get(c, 0) for c in chunk]\n                if nlp:     # vLLM completions logprobs shape: tokens / token_logprobs / top_logprobs (dict per token)\n                    d["choices"][0]["logprobs"] = {"tokens": list(chunk), "token_logprobs": [-0.05] * len(chunk),\n                                                   "top_logprobs": [{c: -0.05, "␀": -2.5} for c in chunk]}\n                b = b"data: " + json.dumps(d).encode() + b"\\n\\n"\n                self.wfile.write(f"{len(b):x}\\r\\n".encode() + b + b"\\r\\n")\n                self.wfile.flush()\n                with LOCK:\n                    CNT["gen"] += len(chunk)\n                    if sent == len(chunk):           # first chunk: the prompt counted once (prefill done)\n                        CNT["prompt"] = CNT.get("prompt", 0.0) + len(prompt)\n                        CNT["pq"] = CNT.get("pq", 0.0) + len(prompt)\n                        CNT["ph"] = CNT.get("ph", 0.0) + (len(prompt) * 0.9 if "--no-enable-prefix-caching" not in argv else 0)\n                    if k:\n                        CNT["drf"] += len(chunk) / step\n                        CNT["acc"] += len(chunk) - len(chunk) / step\n                    CTX[rid] = CTX.get(rid, 0) + len(chunk)\n                time.sleep(len(chunk) / SPEED if not greedy else 0.0)\n            b = b"data: [DONE]\\n\\n"\n            self.wfile.write(f"{len(b):x}\\r\\n".encode() + b + b"\\r\\n" + b"0\\r\\n\\r\\n")\n            self.wfile.flush()\n        except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError):\n            pass\n        finally:\n            with LOCK:\n                RUN.discard(rid)\n                CTX.pop(rid, None)\n\n\nThreadingHTTPServer.request_queue_size = 256      # rev 2e-e: default 5 drops simultaneous connects (Windows)\nsrv = ThreadingHTTPServer(("127.0.0.1", PORT), H)\nsrv.daemon_threads = True\nprint(f"INFO Started server process; Uvicorn running on http://127.0.0.1:{PORT}", flush=True)\nsrv.serve_forever()\n')
(T / "e23_lab.py").write_text('"""E23 lab, rev 3: port-parameterized server lifecycle and measurement for the six-lever notebook (stdlib only).\n\nBase: the Exp 14 VMS harness (VServer / VGen: token ids from return_token_ids, [DONE] + finish_reason required, socket\nshutdown on stop, top-1/top-2 margins, garbage checks on every window). Adopted from the consolidated v3 notebook:\nsame-weight greedy oracle classified by the oracle\'s logprob margin at the first differing token, a steady decode window\nthat starts only after prefill settled, a GPU-release gate, fail-closed statuses, checkpoint identity, a runtime\n--help flag check, KV + GPU memory samples.\nrev 3 (validation agents 2026-10-29, measurement + robustness):\n  - port check by connect (a TIME_WAIT socket of the previous server is not "busy"), with a 90 s wait;\n  - kill(): SIGKILL the whole session after SIGTERM, even when the leader already exited;\n  - steady(): idle gate at entry, admission cap from the prefill to do (700 tok/s assumed), settle needs no preemption\n    and every running stream gaining tokens; in the window: stalled streams, a changing running/waiting set (KV-bound),\n    preemptions and prefills make it INCONCLUSIVE (numbers kept); garbage under load fails it; steps from the\n    vllm:iteration_tokens_total histogram count; replicas measured in one shared window (barrier);\n  - profiled_load(): settle as steady(), bounded profile (delay / max iterations), with_stack off, frontend ignored;\n  - trace_summary(): categories on the kernel base name (attention before GDN: "fla" is inside "flashinfer"), per-forward\n    graph / kernel launch counts inside the gpu_model_runner forward ranges (VLLM_CUSTOM_SCOPES_FOR_PROFILING=1) and the\n    CUPTI correlation fallback -> FULL / PIECEWISE / EAGER / UNKNOWN; startup cudagraph lines parsed explicitly;\n  - teacher_forced(): prompt logprobs of the last N positions of a long prompt (NVFP4 vs FP8 KV at 40k / 90k);\n  - logprobs keyed by token id (return_tokens_as_token_ids); stop() race closed; compare() flags garbage only when the\n    oracle itself is clean.\n"""\nimport gzip\nimport hashlib\nimport json\nimport os\nimport re\nimport signal\nimport socket\nimport statistics\nimport subprocess\nimport threading\nimport time\nimport urllib.request as U\nfrom pathlib import Path\n\nFAIL_MARGIN = 0.5          # first differing token: oracle top-1 minus top-2 logprob at or above this = real divergence\nLIVE = set()               # every running Server, for the hard-stop watchdog\n\n\n# ------------------------------------------------------------------ server lifecycle\ndef port_listening(port):\n    """True when something accepts connections on 127.0.0.1:port (TIME_WAIT leftovers do not)."""\n    s = socket.socket()\n    s.settimeout(1.0)\n    try:\n        s.connect(("127.0.0.1", int(port)))\n        return True\n    except OSError:\n        return False\n    finally:\n        s.close()\n\n\nclass Server:\n    def __init__(self, name, argv, env, port, log_dir, gpus="0,1,2,3"):\n        self.name, self.argv, self.env, self.port = name, [str(a) for a in argv], dict(env), int(port)\n        self.gpus = str(gpus)\n        self.log_path = Path(log_dir) / f"server_{name}.log"\n        self.proc, self.t0, self.ready_sec = None, None, None\n\n    @property\n    def url(self):\n        return f"http://127.0.0.1:{self.port}"\n\n    def start(self):\n        end = time.time() + 90\n        while port_listening(self.port):\n            if time.time() > end:\n                raise RuntimeError(f"PORT_BUSY {self.port} (a listener is still there)")\n            time.sleep(3)\n        self.log_path.parent.mkdir(parents=True, exist_ok=True)\n        self._fh = open(self.log_path, "w", encoding="utf-8")\n        self.t0 = time.perf_counter()\n        self.proc = subprocess.Popen(self.argv, env=self.env, stdout=self._fh, stderr=subprocess.STDOUT,\n                                     start_new_session=True, cwd="/tmp" if os.path.isdir("/tmp") else None)\n        LIVE.add(self)\n        return self\n\n    def alive(self):\n        return self.proc is not None and self.proc.poll() is None\n\n    def log(self):\n        try:\n            return self.log_path.read_text(encoding="utf-8", errors="replace")\n        except Exception:\n            return ""\n\n    def rss_mb(self):\n        """Host RAM of the server\'s whole session (engine + workers), MB; None off Linux."""\n        try:\n            r = subprocess.run(["ps", "-o", "rss=", "--sid", str(self.proc.pid)], capture_output=True, text=True, timeout=20)\n            return round(sum(int(x) for x in r.stdout.split() if x.isdigit()) / 1024, 1)\n        except Exception:\n            return None\n\n    def wait_ready(self, timeout):\n        while time.perf_counter() - self.t0 < timeout:\n            if not self.alive():\n                return False, f"exited rc={self.proc.returncode}"\n            try:\n                U.urlopen(self.url + "/health", timeout=10).read()\n                self.ready_sec = round(time.perf_counter() - self.t0, 1)\n                return True, f"ready in {self.ready_sec}s"\n            except Exception:\n                time.sleep(3)\n        return False, "startup timeout"\n\n    def kill(self):\n        if self.proc is None:\n            return\n        pg = self.proc.pid\n        try:\n            os.killpg(pg, signal.SIGTERM) if hasattr(os, "killpg") else self.proc.terminate()\n        except Exception:\n            pass\n        try:\n            self.proc.wait(timeout=40)\n        except Exception:\n            pass\n        # rev 3: the leader may exit while EngineCore / workers of the session still hold GPUs: always SIGKILL the group\n        try:\n            os.killpg(pg, signal.SIGKILL) if hasattr(os, "killpg") else self.proc.kill()\n        except Exception:\n            pass\n        try:\n            self.proc.wait(timeout=30)\n        except Exception:\n            pass\n        try:\n            self._fh.close()\n        except Exception:\n            pass\n        LIVE.discard(self)\n\n    def facts(self):\n        """What vLLM logged at startup: KV pool, max concurrency, selected kernels, graph mode, backend, KV dtype."""\n        t = self.log()\n        kv = [int(x.replace(",", "")) for x in re.findall(r"GPU KV cache size: ([\\d,]+) tokens", t)]\n        mc = re.findall(r"Maximum concurrency for [\\d,]+ tokens per request: ([\\d.]+)x", t)\n        kern = sorted(set(re.findall(r"Selected (\\w+Kernel) for", t)))\n        errs = [l[:300] for l in t.splitlines() if re.search(r"Traceback|CUDA error|illegal memory|out of memory|"\n                                                               r"RuntimeError|AssertionError", l)][:12]\n        pick = lambda pat, n=8: [l.strip()[:240] for l in t.splitlines() if re.search(pat, l, re.I)][:n]\n        # rev 3 (measurement H5): the decisive startup evidence for the graph mode actually used with spec decode\n        downgrade = pick(r"not supported with spec-decode.*setting cudagraph_mode|setting cudagraph_mode\\s*=|"\n                         r"[Oo]verriding cudagraph_mode to", 4)       # rev 3e: + the 0.32 wording\n        capture = sorted(set(re.findall(r"Capturing CUDA graphs \\(([^)]+)\\)", t)))\n        configured = sorted(set(re.findall(r"CUDAGraphMode\\.(\\w+)", t)))\n        return {"kv_tokens": min(kv) if kv else None, "kv_tokens_all": kv, "max_concurrency": float(mc[-1]) if mc else None,\n                "kernels": kern, "error_lines": errs, "ready_sec": self.ready_sec,\n                "graph_capture_modes": capture, "graph_downgrade_lines": downgrade, "cudagraph_modes_logged": configured,\n                "backend_lines": pick(r"attention backend|Using \\w+ backend|backend=|AttentionBackend"),\n                "kv_dtype_lines": pick(r"kv_cache_dtype|kv cache dtype|nvfp4"),\n                "host_embed_lines": pick(r"host[- _]?(resident|embed)|HOST_EMBED|embed.*(cpu|host)", 6)}\n\n\ndef gpu_pids(gpus=None):\n    """Compute PIDs on all GPUs, or only on the GPU indices in `gpus` (a bench on another GPU must not fail the release\n    gate of a server that used GPU 0 only). None when nvidia-smi is unavailable."""\n    try:\n        a = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,gpu_uuid", "--format=csv,noheader,nounits"],\n                           capture_output=True, text=True, timeout=30)\n        g = subprocess.run(["nvidia-smi", "--query-gpu=index,uuid", "--format=csv,noheader,nounits"],\n                           capture_output=True, text=True, timeout=30)\n    except Exception:\n        return None\n    if a.returncode or g.returncode:\n        return None\n    idx = {u.strip(): int(i) for i, u in (l.split(",", 1) for l in g.stdout.strip().splitlines() if "," in l)}\n    want = None if gpus is None else {int(x) for x in str(gpus).split(",") if str(x).strip() != ""}\n    out = set()\n    for l in a.stdout.strip().splitlines():\n        if "," not in l:\n            continue\n        pid, uuid = (x.strip() for x in l.split(",", 1))\n        if pid.isdigit() and (want is None or idx.get(uuid) in want):\n            out.add(int(pid))\n    return sorted(out)\n\n\ndef gpu_mem():\n    try:\n        r = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"],\n                           capture_output=True, text=True, timeout=30)\n        return [[int(v) for v in l.split(",")] for l in r.stdout.strip().splitlines() if l.strip()]\n    except Exception:\n        return None\n\n\ndef wait_gpu_release(timeout=180, fake=False, gpus=None, free_mib=1024):\n    """True when the GPUs in `gpus` (all when None) hold no compute PID AND use < free_mib MiB each (rev 3: compute-app\n    PIDs are often invisible inside Kaggle containers; memory.used is the proven E18 gate); None without nvidia-smi."""\n    if fake:\n        return True\n    want = None if gpus is None else {int(x) for x in str(gpus).split(",") if str(x).strip() != ""}\n    end = time.time() + timeout\n    while time.time() < end:\n        p, mem = gpu_pids(gpus), gpu_mem()\n        if p is None or mem is None:\n            return None\n        used = [m[1] for m in mem if want is None or m[0] in want]\n        if not p and all(u < free_mib for u in used):\n            return True\n        time.sleep(3)\n    return False\n\n\ndef identity(model_dir):\n    """Checkpoint identity: sha256 of config / tokenizer / index files (an oracle is only compared with the same weights)."""\n    root = Path(model_dir)\n    files = [p for n in ("config.json", "tokenizer.json", "tokenizer_config.json", "generation_config.json")\n             for p in [root / n] if p.is_file()] + sorted(root.glob("*.safetensors.index.json"))\n    h = hashlib.sha256()\n    for p in files:\n        h.update(p.name.encode())\n        h.update(hashlib.sha256(p.read_bytes()).digest())\n    return {"dir": str(root), "files": [p.name for p in files], "sha256": h.hexdigest()[:16]}\n\n\n_HELP = {}\n\n\ndef help_check(key, argv_prefix, env, flags, timeout=300):\n    """The engine\'s own `serve --help=all` (run once per engine `key`, cached) must list every flag of THIS launch\n    (rev 3: every launch is checked, not only the first); returns the missing ones."""\n    if key not in _HELP:\n        try:\n            r = subprocess.run(list(argv_prefix) + ["--help=all"], capture_output=True, text=True, timeout=timeout, env=env)\n            _HELP[key] = r.stdout + r.stderr\n        except Exception as e:\n            _HELP[key] = "ERROR " + repr(e)[:200]\n    txt = _HELP[key]\n    if len(txt) < 2000:\n        return {"ran": False, "error": "help output too short or failed", "tail": txt[-400:]}\n    want = sorted({f for f in flags if isinstance(f, str) and f.startswith("--")})\n    return {"ran": True, "missing": [f for f in want if f not in txt]}\n\n\ndef start_watchdog(hard_stop_epoch, on_fire):\n    """A real hard stop: at `hard_stop_epoch` every live Server is killed and on_fire() is called."""\n    def run():\n        while time.time() < hard_stop_epoch:\n            time.sleep(5)\n        try:\n            on_fire()\n        finally:\n            for s in list(LIVE):\n                try:\n                    s.kill()\n                except Exception:\n                    pass\n    t = threading.Thread(target=run, daemon=True, name="e23-watchdog")\n    t.start()\n    return t\n\n\ndef prefetch(paths, threads=16, chunk=16 << 20):\n    """Read files once with many readers (page cache warm-up of a cold /kaggle/input mount); returns bytes and seconds."""\n    files = [p for root in paths for p in sorted(Path(root).rglob("*")) if p.is_file()]\n    t0, done, lock = time.time(), [0], threading.Lock()\n\n    def worker(fs):\n        for p in fs:\n            try:\n                with open(p, "rb") as f:\n                    while True:\n                        b = f.read(chunk)\n                        if not b:\n                            break\n                        with lock:\n                            done[0] += len(b)\n            except Exception:\n                pass\n    big = sorted(files, key=lambda p: -p.stat().st_size)\n    parts = [big[i::threads] for i in range(threads)]\n    ths = [threading.Thread(target=worker, args=(x,), daemon=True) for x in parts if x]\n    for t in ths:\n        t.start()\n    for t in ths:\n        t.join()\n    dt = time.time() - t0\n    return {"files": len(files), "bytes": done[0], "sec": round(dt, 1), "MBps": round(done[0] / 1e6 / max(dt, 1e-3), 1)}\n\n\n# ------------------------------------------------------------------ metrics\ndef prom(port):\n    txt = U.urlopen(f"http://127.0.0.1:{port}/metrics", timeout=15).read().decode("utf-8", "replace")\n    out = {}\n    for line in txt.splitlines():\n        if not line or line.startswith("#"):\n            continue\n        m = re.match(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\\{[^}]*\\})?\\s+(\\S+)", line)\n        if not m:\n            continue\n        try:\n            v = float(m.group(3))\n        except ValueError:\n            continue\n        if v != v:\n            continue\n        out[m.group(1)] = out.get(m.group(1), 0.0) + v\n        out[m.group(1) + "@max"] = max(out.get(m.group(1) + "@max", v), v)\n    return out\n\n\nGEN, PROMPT, RUN, WAIT = ("vllm:generation_tokens_total", "vllm:prompt_tokens_total", "vllm:num_requests_running",\n                          "vllm:num_requests_waiting")\nPREEMPT, KVU, ITER = "vllm:num_preemptions_total", "vllm:kv_cache_usage_perc", "vllm:iteration_tokens_total_count"\nACC, DRF, DTOK = ("vllm:spec_decode_num_accepted_tokens_total", "vllm:spec_decode_num_drafts_total",\n                  "vllm:spec_decode_num_draft_tokens_total")\n\n\n# ------------------------------------------------------------------ one streaming request (Exp 14 VGen, port-aware)\nclass Stream(threading.Thread):\n    def __init__(self, port, ids, max_new, temperature=1.0, seed=None, logprobs=0, ignore_eos=True, model="q27"):\n        super().__init__(daemon=True)\n        self.port = int(port)\n        self.body = {"model": model, "prompt": list(ids), "max_tokens": int(max_new), "temperature": temperature,\n                     "top_p": 0.95 if temperature else 1.0, "stream": True, "ignore_eos": ignore_eos, "return_token_ids": True}\n        if temperature:\n            self.body["top_k"] = 20\n        if seed is not None:\n            self.body["seed"] = int(seed)\n        if logprobs:\n            self.body["logprobs"] = int(logprobs)\n            self.body["return_tokens_as_token_ids"] = True      # rev 3: logprob keys by id (no string collisions)\n        self.text, self.tokens, self.margins = [], [], []\n        self.err, self.done_seen, self.finish_reason = None, False, None\n        self.t_start = self.t_first = self.t_end = None\n        self._resp, self._halt = None, False\n\n    def run(self):\n        self.t_start = time.perf_counter()\n        try:\n            req = U.Request(f"http://127.0.0.1:{self.port}/v1/completions", data=json.dumps(self.body).encode(),\n                            method="POST", headers={"Content-Type": "application/json"})\n            self._resp = U.urlopen(req, timeout=7200)\n            for raw in self._resp:\n                if self._halt:\n                    break\n                line = raw.strip()\n                if not line.startswith(b"data:"):\n                    continue\n                payload = line[5:].strip()\n                if payload == b"[DONE]":\n                    self.done_seen = True\n                    break\n                d = json.loads(payload)\n                if "error" in d or d.get("object") == "error":\n                    self.err = ("stream error: " + json.dumps(d.get("error") or d.get("message") or d))[:300]\n                    break\n                for c in d.get("choices") or []:\n                    self.text.append(c.get("text") or "")\n                    ids_ = c.get("token_ids")\n                    lp = c.get("logprobs") or {}\n                    if ids_ is None:\n                        if c.get("text") or lp.get("tokens"):\n                            self.err = "MISSING_STREAM_TOKEN_IDS"\n                    elif any(not isinstance(v, int) or isinstance(v, bool) for v in ids_):\n                        self.err = "INVALID_STREAM_TOKEN_IDS"\n                    else:\n                        if ids_ and self.t_first is None:\n                            self.t_first = time.perf_counter()\n                        self.tokens.extend(ids_)\n                    for top in lp.get("top_logprobs") or []:\n                        vals = sorted((v for v in (top or {}).values() if isinstance(v, (int, float))), reverse=True)\n                        self.margins.append(round(vals[0] - vals[1], 4) if len(vals) >= 2 else None)\n                    if c.get("finish_reason") is not None:\n                        self.finish_reason = c["finish_reason"]\n                if self.err:\n                    break\n        except Exception as e:\n            if not self._halt:\n                self.err = repr(e)[:300]\n        finally:                         # rev 3: a stop() before the response existed still closes the socket here\n            try:\n                if self._resp is not None:\n                    self._resp.close()\n            except Exception:\n                pass\n        self.t_end = time.perf_counter()\n        if not self._halt and not self.err and not (self.done_seen and self.finish_reason):\n            self.err = f"INCOMPLETE_SSE done={self.done_seen} finish={self.finish_reason}"\n\n    def stop(self):\n        self._halt = True\n        try:\n            self._resp.fp.raw._sock.shutdown(socket.SHUT_RDWR)\n        except Exception:\n            pass\n        try:\n            if self._resp is not None:\n                self._resp.close()\n        except Exception:\n            pass\n\n    def row(self):\n        return {"text": "".join(self.text), "tokens": list(self.tokens), "margins": list(self.margins), "error": self.err,\n                "done": self.done_seen, "finish_reason": self.finish_reason}\n\n\n# ------------------------------------------------------------------ correctness\ndef greedy(port, prompts, n_tok=128, logprobs=2, timeout=900, model="q27"):\n    """Greedy rows for the same-weight oracle (one request at a time: batch-size effects stay out of the comparison)."""\n    rows = []\n    for ids in prompts:\n        g = Stream(port, ids, n_tok, temperature=0.0, logprobs=logprobs, ignore_eos=True, model=model)\n        g.start()\n        g.join(timeout)\n        if g.is_alive():\n            g.stop()\n            g.join(10)\n            r = g.row()\n            r["error"] = r["error"] or "GREEDY_TIMEOUT"\n        else:\n            r = g.row()\n        rows.append(r)\n    return rows\n\n\ndef degenerate(text):\n    """Objective garbage checks: empty, a run of one char, \'!!!!\' (the E3 / E13 garbage), very low variety."""\n    t = text or ""\n    if not t.strip():\n        return "EMPTY"\n    if re.search(r"(\\S)\\1{200,}", t, re.S):      # rev 3: grid rows / separators are not garbage\n        return "REPEATED_CHAR"\n    if t.count("!") > 0.5 * max(1, len(t)):\n        return "BANG_GARBAGE"\n    words = re.findall(r"[A-Za-z]{3,}", t)            # rev 3c: words only; ARC grid dumps (digits) are not garbage\n    if len(words) > 60 and len(set(words)) < 0.08 * len(words):\n        return "LOW_VARIETY"\n    return None\n\n\ndef compare(ref, cand):\n    """Same-weight oracle: at the first differing token the ORACLE\'s top-1 - top-2 logprob margin decides:\n    >= FAIL_MARGIN -> FAIL_HIGH_MARGIN, below -> NEAR_TIE; identical -> EXACT; missing data -> INCONCLUSIVE."""\n    if not ref or not cand or len(ref) != len(cand):\n        return {"status": "INCONCLUSIVE_PAIRS_MISSING"}\n    per, worst = [], "EXACT"\n    rank = {"EXACT": 0, "NEAR_TIE": 1, "INCONCLUSIVE": 2, "FAIL_HIGH_MARGIN": 3}\n    for a, b in zip(ref, cand):\n        if a.get("error") or b.get("error") or not a.get("tokens") or not b.get("tokens"):\n            st, d, m = "INCONCLUSIVE", None, None\n        else:\n            x, y = a["tokens"], b["tokens"]\n            d = next((i for i in range(min(len(x), len(y))) if x[i] != y[i]), None)\n            if d is None:\n                st, m = ("EXACT" if len(x) == len(y) else "INCONCLUSIVE"), None\n            else:\n                mm = a.get("margins") or []\n                m = mm[d] if d < len(mm) else None\n                st = "INCONCLUSIVE" if m is None else ("FAIL_HIGH_MARGIN" if m >= FAIL_MARGIN else "NEAR_TIE")\n        g = degenerate(b.get("text")) if (b.get("tokens") and not b.get("error")) else None\n        if g and degenerate(a.get("text")) is None:           # rev 3c: a request error is not corruption\n            st = "FAIL_HIGH_MARGIN"\n        per.append({"first_diff": d, "oracle_margin": m, "status": st, "garbage": g})\n        worst = st if rank[st] > rank[worst] else worst\n    return {"status": worst, "rows": per}\n\n\ndef chat_probes(port, model="q27", cases=(("Reply with exactly one number: 7 * 6 =", "42"),\n                                          ("Reply with exactly one number: 2 + 2 =", "4"),\n                                          ("Reply with exactly one number: 9 - 3 =", "6")), timeout=300):\n    """Arithmetic through /v1/chat/completions with thinking off. vLLM 0.32 returns the answer in message.content and\n    the thinking in message.reasoning (NOT reasoning_content: the consolidated notebook\'s blocker)."""\n    out = []\n    for q, want in cases:\n        body = {"model": model, "messages": [{"role": "user", "content": q}], "temperature": 0, "max_tokens": 32,\n                "chat_template_kwargs": {"enable_thinking": False}}\n        try:\n            r = json.loads(U.urlopen(U.Request(f"http://127.0.0.1:{port}/v1/chat/completions", data=json.dumps(body).encode(),\n                                               headers={"Content-Type": "application/json"}), timeout=timeout).read())\n            msg = r["choices"][0]["message"]\n            got = (msg.get("content") or "").strip()\n            out.append({"q": q, "want": want, "content": got[:80], "reasoning_chars": len(msg.get("reasoning") or ""),\n                        "ok": re.search(r"(?<!\\d)" + want + r"(?!\\d)", got) is not None})\n        except Exception as e:\n            out.append({"q": q, "want": want, "error": repr(e)[:200], "ok": False})\n    return {"ok": all(x["ok"] for x in out), "cases": out}\n\n\ndef template_probe(port, messages, want_ids, model="q27", timeout=120):\n    """rev 3h (three GPT SMOKE audits): what the SERVER\'s chat path renders for `messages` (its own template and default\n    chat-template kwargs, no per-request kwargs) against the production xhigh render `want_ids`. MATCH / MISMATCH /\n    ERROR; the load requests themselves use pre-rendered ids, so this shows chat-path equivalence only."""\n    body = {"model": model, "messages": messages, "add_generation_prompt": True}\n    try:\n        r = json.loads(U.urlopen(U.Request(f"http://127.0.0.1:{port}/tokenize", data=json.dumps(body).encode(),\n                                           headers={"Content-Type": "application/json"}), timeout=timeout).read())\n        got = list(r.get("tokens") or [])\n    except Exception as e:\n        return {"status": "ERROR", "error": repr(e)[:200]}\n    want = list(want_ids)\n    first = next((i for i, (a, b) in enumerate(zip(got, want)) if a != b), None if len(got) == len(want) else min(len(got), len(want)))\n    return {"status": "MATCH" if got == want else "MISMATCH", "server_tokens": len(got), "want_tokens": len(want),\n            "first_diff": first}\n\n\ndef teacher_forced(port, ids, last=256, model="q27", timeout=1800):\n    """rev 3 (measurement M3): prompt logprobs of the last `last` positions of a long prompt (one prefill, 1 token out),\n    for a lossy-KV comparison (NVFP4 vs FP8 KV) at the context lengths that matter. Returns per-position\n    [logprob of the actual token, its rank, the engine\'s own rank-1 token id] or an error. rev 3h (three GPT SMOKE\n    audits): the rank-1 token id is kept, so top-1 agreement compares the two engines\' predictions."""\n    body = {"model": model, "prompt": list(ids), "max_tokens": 1, "temperature": 0, "prompt_logprobs": 1,\n            "return_tokens_as_token_ids": True}\n    try:\n        r = json.loads(U.urlopen(U.Request(f"http://127.0.0.1:{port}/v1/completions", data=json.dumps(body).encode(),\n                                           headers={"Content-Type": "application/json"}), timeout=timeout).read())\n        pl = r["choices"][0].get("prompt_logprobs") or []\n    except Exception as e:\n        return {"error": repr(e)[:300]}\n    out = []\n    for pos in range(max(1, len(ids) - last), len(ids)):\n        d = pl[pos] if pos < len(pl) else None\n        if not isinstance(d, dict):\n            out.append(None)\n            continue\n        e = d.get(str(ids[pos])) or d.get(f"token_id:{ids[pos]}")     # rev 3c: no fallback to another token\n        top1 = [_tok_id(k) for k, v in d.items() if isinstance(v, dict) and v.get("rank") == 1]\n        out.append([e.get("logprob"), e.get("rank"), top1[0] if len(top1) == 1 else None] if isinstance(e, dict) else None)\n    return {"positions": out, "n": len(out)}\n\n\ndef _tok_id(k):\n    """A prompt-logprob key (\'123\' or \'token_id:123\') as an int id; anything else stays a tagged string."""\n    m = re.fullmatch(r"(?:token_id:)?(-?\\d+)", str(k))\n    return int(m.group(1)) if m else "str:" + str(k)\n\n\ndef compare_tf(ref, cand):\n    """Teacher-forced comparison: mean / max |delta logprob| of the actual tokens and rank-1 agreement. rev 3d (four\n    GPT audits): MEASURED only when EVERY expected position is valid on both sides (1 of 256 used to be MEASURED);\n    MEASURED is a measurement, not a PASS (the caller applies the A/A-derived tolerance)."""\n    if not ref or not cand or ref.get("error") or cand.get("error"):\n        return {"status": "INCONCLUSIVE"}\n    a_, b_ = ref.get("positions") or [], cand.get("positions") or []\n    if not a_ or len(a_) != len(b_):\n        return {"status": "INCONCLUSIVE_INCOMPLETE_TF", "ref_n": len(a_), "cand_n": len(b_)}\n    if any(not a or not b or a[0] is None or b[0] is None for a, b in zip(a_, b_)):\n        return {"status": "INCONCLUSIVE_INCOMPLETE_TF", "valid": sum(1 for a, b in zip(a_, b_) if a and b and a[0] is not None\n                                                                     and b[0] is not None), "expected": len(a_)}\n    pairs = list(zip(a_, b_))\n    if any(len(a) < 3 or len(b) < 3 or a[2] is None or b[2] is None for a, b in pairs):\n        return {"status": "INCONCLUSIVE_TOP1_MISSING", "expected": len(pairs),\n                "with_top1": sum(1 for a, b in pairs if len(a) >= 3 and len(b) >= 3 and a[2] is not None and b[2] is not None)}\n    d = sorted(abs(a[0] - b[0]) for a, b in pairs)\n    # rev 3h (three GPT SMOKE audits): top-1 agreement = both engines predict the same rank-1 token id. rev 3d-3g counted\n    # "the forced token is rank 1 on both sides or on neither", which reads 1.0 for ranks 2 vs 3.\n    r1 = sum(1 for a, b in pairs if a[2] == b[2]) / len(pairs)\n    both = sum(1 for a, b in pairs if (a[1] == 1) == (b[1] == 1)) / len(pairs)\n    p99 = d[min(len(d) - 1, int(round(0.99 * (len(d) - 1))))]          # rev 3e: robust to one outlier position\n    return {"status": "MEASURED", "n": len(pairs), "mean_abs_dlogprob": round(statistics.mean(d), 4),\n            "p99_abs_dlogprob": round(p99, 3), "max_abs_dlogprob": round(d[-1], 3), "rank1_agreement": round(r1, 4),\n            "forced_token_rank1_both_or_neither": round(both, 4)}\n\n\ndef tf_complete(rows):\n    """rev 3e: a teacher-forced row set counts as a reference only when every position of every band is valid\n    (rev 3h: including the rank-1 token id)."""\n    return bool(rows) and all(isinstance(r, dict) and not r.get("error") and r.get("positions") and\n                              all(p and p[0] is not None and len(p) >= 3 and p[2] is not None for p in r["positions"])\n                              for r in rows.values())\n\n\n# ------------------------------------------------------------------ steady decode\ndef steady(port, prompts, level, secs=60.0, admit=None, max_new=8192, temperature=1.0, seed=1234, model="q27",\n           sample=2.0, prefill_tok_s=700.0, gpu_sample=True, barrier=None):\n    """`level` distinct prompts, then a decode window that starts only once prefill has settled. Status:\n    STEADY / CAPACITY_LIMITED (fewer running than asked: KV) / INCONCLUSIVE_* / FAIL_*; numbers are kept on an\n    INCONCLUSIVE window so a KV census survives; never a made-up number. `barrier` (steady_multi) starts every replica\'s\n    window at the same moment."""\n    plen = len(prompts[0]) if prompts else 0\n    admit = admit or max(420.0, 120.0 + level * plen / prefill_tok_s)\n    r = {"level": level, "status": "INCONCLUSIVE_NOT_STARTED", "port": port, "prompt_tokens": plen, "max_new": max_new,\n         "admit_limit_sec": round(admit)}\n    if not wait_idle(port, 60):\n        r["status"] = "INCONCLUSIVE_NOT_IDLE_AT_START"\n        if barrier:\n            barrier.abort()\n        return r\n    try:\n        p0 = prom(port)\n    except Exception as e:\n        r.update(status="INCONCLUSIVE_METRICS_UNAVAILABLE", error=repr(e)[:200])\n        if barrier:\n            barrier.abort()\n        return r\n    if GEN not in p0 or PROMPT not in p0:\n        r["status"] = "INCONCLUSIVE_COUNTERS_MISSING"\n        if barrier:\n            barrier.abort()\n        return r\n    ss = [Stream(port, prompts[i % len(prompts)], max_new, temperature, seed + i, model=model) for i in range(level)]\n    for g in ss:\n        g.start()\n    try:\n        t_adm, best, settled, prev, samples, flat_since = time.perf_counter(), 0, 0, None, [], None\n        last_tok = [0] * len(ss)\n        while time.perf_counter() - t_adm < admit:\n            time.sleep(sample)\n            if all(not g.is_alive() for g in ss):          # rev 3: checked before the metrics read (a dead server)\n                r.update(status="INCONCLUSIVE_STREAMS_ENDED_IN_ADMISSION", errors=[g.err for g in ss if g.err][:3])\n                if barrier:\n                    barrier.abort()                       # rev 3c: the other replicas must not wait for this one\n                return r\n            try:\n                m = prom(port)\n            except Exception:\n                continue\n            run, wait, pt, pre = int(m.get(RUN, 0)), int(m.get(WAIT, 0)), m.get(PROMPT), m.get(PREEMPT, 0)\n            now_tok = [len(g.tokens) for g in ss]\n            first = sum(g.t_first is not None for g in ss)\n            gaining = sum(1 for a, b in zip(last_tok, now_tok) if b > a)\n            last_tok = now_tok\n            best = max(best, run)\n            samples.append([round(time.perf_counter() - t_adm, 1), run, wait, first, gaining])\n            # rev 3: no preemption in the sample, every running request emitted and is gaining tokens\n            calm = run > 0 and prev is not None and pt == prev[0] and pre == prev[1] and first >= run and gaining >= run\n            if calm and run >= level and wait == 0:\n                settled += 1\n            elif calm and run < level and wait > 0:       # KV-bound: the rest wait; accept a plateau held 30 s\n                flat_since = flat_since or time.perf_counter()\n                settled = settled + 1 if time.perf_counter() - flat_since > 30 else 0\n            else:\n                settled, flat_since = 0, None\n            prev = (pt, pre)\n            if settled >= 3:\n                break\n        r["admission"] = {"sec": round(time.perf_counter() - t_adm, 1), "best_running": best, "tail": samples[-6:],\n                          "prefill_tok_s": round((prom(port).get(PROMPT, 0) - p0.get(PROMPT, 0)) /\n                                                 max(1e-3, time.perf_counter() - t_adm), 1)}\n        if settled < 3:\n            r["status"] = "INCONCLUSIVE_NO_SETTLE"\n            if barrier:\n                barrier.abort()\n            return r\n        if barrier:\n            try:\n                barrier.wait(timeout=max(60.0, admit))\n            except threading.BrokenBarrierError:\n                r["status"] = "INCONCLUSIVE_REPLICA_BARRIER"\n                return r\n        m0, t0 = prom(port), time.perf_counter()\n        runs, waits, kvs, mids, gmem, stalls = [], [], [], [], [], [0] * len(ss)\n        last_tok, track, misses = [len(g.tokens) for g in ss], None, 0\n        while time.perf_counter() - t0 < secs:\n            time.sleep(min(sample, max(0.1, secs - (time.perf_counter() - t0))))\n            try:\n                m = prom(port)\n            except Exception:                              # rev 3c: one failed metrics read is not a crash\n                misses += 1\n                if misses >= 5:\n                    break\n                continue\n            mids.append(m)\n            runs.append(m.get(RUN, 0))\n            waits.append(m.get(WAIT, 0))\n            if KVU in m:\n                kvs.append(m[KVU])\n            now_tok = [len(g.tokens) for g in ss]\n            if track is None:                              # rev 3c: only streams decoding at window start can stall\n                track = {i for i in range(len(ss)) if now_tok[i] > last_tok[i]}\n            for i, g in enumerate(ss):\n                if i in track and g.is_alive():\n                    stalls[i] = stalls[i] + 1 if now_tok[i] <= last_tok[i] else 0\n            last_tok = now_tok\n            if any(x >= 3 for x in stalls):\n                break\n            if gpu_sample and len(mids) % 5 == 1:\n                g_ = gpu_mem()\n                if g_:\n                    gmem.append(g_)\n        m1, t1 = prom(port), time.perf_counter()\n        gen = m1.get(GEN, 0) - m0.get(GEN, 0)\n        dt = t1 - t0\n        acc, drf, dtk, it = (m1.get(k, 0) - m0.get(k, 0) for k in (ACC, DRF, DTOK, ITER))\n        rm = statistics.mean(runs) if runs else 0\n        steps = it if it else ((drf / rm) if (drf and rm) else ((gen / rm) if rm else 0))\n        r.update(tok_s=round(gen / dt, 1) if dt else None, gen_tokens=int(gen), window_sec=round(dt, 1),\n                 running_mean=round(rm, 2), running_min=min(runs or [0]), running_max=max(runs or [0]),\n                 waiting_min=min(waits or [0]), waiting_max=max(waits or [0]),\n                 tok_s_per_stream=round(gen / dt / rm, 2) if rm and dt else None,\n                 accept_per_draft=round(acc / drf, 3) if drf else None,\n                 mean_acceptance_length=round(1 + acc / drf, 3) if drf else None,\n                 accept_rate=round(acc / dtk, 3) if dtk else None,\n                 steps=int(steps), steps_source="iteration_histogram" if it else "estimate",\n                 step_ms=round(dt * 1000 / steps, 1) if steps else None,\n                 tokens_per_step=round(gen / steps, 2) if steps else None,\n                 kv_usage_mean=round(statistics.mean(kvs), 3) if kvs else None, kv_usage_max=max(kvs) if kvs else None,\n                 gpu_mem_peak_mib=[max(row[i][1] for row in gmem if len(row) > i) for i in range(max(map(len, gmem)))] if gmem else None,\n                 preemptions=m1.get(PREEMPT, 0) - m0.get(PREEMPT, 0), t0=t0, t1=t1)\n        if misses >= 5:\n            r["status"] = "INCONCLUSIVE_METRICS_LOST"\n        elif any(x >= 3 for x in stalls):\n            r["status"] = "INCONCLUSIVE_STALLED_STREAM"\n        elif sum(g.is_alive() for g in ss) < min(level, best):\n            r["status"] = "INCONCLUSIVE_STREAMS_ENDED_IN_WINDOW"\n        elif any(x.get(PROMPT) != m0.get(PROMPT) for x in mids + [m1]):\n            r["status"] = "INCONCLUSIVE_PREFILL_IN_WINDOW"\n        elif r["preemptions"] > 0:\n            r["status"] = "INCONCLUSIVE_PREEMPTION"\n        elif gen <= 0:\n            r["status"] = "FAIL_NO_DECODE"\n        elif min(runs or [0]) < level and (len(set(runs)) > 1 or len(set(waits)) > 1):\n            r["status"] = "INCONCLUSIVE_RUNNING_SET_CHANGED"      # KV-bound window must hold one fixed set\n        elif min(runs or [0]) < 0.8 * max(runs or [1]):\n            r["status"] = "INCONCLUSIVE_RUNNING_UNSTABLE"\n        else:\n            r["status"] = ("STEADY" if min(runs or [0]) == max(runs or [0]) == level else\n                           "CAPACITY_LIMITED" if max(runs or [0]) < level else "INCONCLUSIVE_RUNNING_ABOVE_LEVEL")\n        return r\n    finally:\n        errs = sorted({g.err for g in ss if g.err})[:3]           # rev 3: snapshot before stop() sets _halt\n        for g in ss:\n            g.stop()\n        for g in ss:\n            g.join(10)\n        r["client_leak"] = any(g.is_alive() for g in ss)\n        r["stream_errors"] = errs\n        gb = [degenerate("".join(g.text)[-4000:]) if g.tokens else None for g in ss]\n        r["garbage"] = {"streams": len(ss), "with_output": sum(1 for g in ss if g.tokens),\n                        "garbage": [x for x in gb if x], "sample_tail": "".join(ss[0].text)[-200:] if ss and ss[0].tokens else ""}\n        if r["garbage"]["garbage"] and r.get("status") in ("STEADY", "CAPACITY_LIMITED"):\n            r["status"] = "FAIL_GARBAGE_UNDER_LOAD"\n        r["idle_after"] = wait_idle(port)\n\n\ndef steady_multi(jobs, secs=60.0, **kw):\n    """Several servers at once (TP1 replicas): jobs = [(port, prompts, level)]; one shared window (barrier). The aggregate\n    counts only when every window is STEADY / CAPACITY_LIMITED and the windows overlap >= 80 % of the shortest."""\n    out = [None] * len(jobs)\n    bar = threading.Barrier(len(jobs))\n\n    def one(i, j):\n        try:\n            out[i] = steady(j[0], j[1], j[2], secs=secs, barrier=bar, **kw)\n        except Exception as e:          # rev 3: a dead replica is recorded, not printed to stderr only\n            out[i] = {"status": "ERROR_REPLICA", "error": f"{type(e).__name__}: {e}"[:300], "port": j[0]}\n            try:\n                bar.abort()\n            except Exception:\n                pass\n    th = [threading.Thread(target=one, args=(i, j), daemon=True) for i, j in enumerate(jobs)]\n    for t in th:\n        t.start()\n    for t in th:\n        t.join()\n    good = all(o and o.get("status") in ("STEADY", "CAPACITY_LIMITED") for o in out)\n    ov = (min(o["t1"] for o in out) - max(o["t0"] for o in out)) if good else 0\n    aligned = good and ov > 0.8 * min(o["window_sec"] for o in out)\n    return {"replicas": out, "aligned": aligned, "overlap_sec": round(ov, 1),\n            "tok_s": round(sum(o["tok_s"] for o in out), 1) if aligned else None,\n            "running_mean": round(sum(o["running_mean"] for o in out), 2) if aligned else None,\n            "status": "AGGREGATE_STEADY" if aligned else "INCONCLUSIVE_REPLICA_WINDOWS"}\n\n\ndef wait_idle(port, wait=90.0):\n    end, hits = time.perf_counter() + wait, 0\n    while time.perf_counter() < end:\n        try:\n            m = prom(port)\n            hits = hits + 1 if m.get(RUN, 0) == 0 and m.get(WAIT, 0) == 0 else 0\n        except Exception:\n            hits = 0\n        if hits >= 3:\n            return True\n        time.sleep(2)\n    return False\n\n\n# ------------------------------------------------------------------ profiler (FULL-graph proof + per-category time)\n# rev 3: matched on the kernel BASE name, attention before GDN ("fla" is a substring of "flashinfer"), real 0.32 GDN names\nCATS = [("nccl", ("nccl", "allreduce", "all_reduce", "allgather", "all_gather")),\n        ("attention", ("flashinfer", "flash_fwd", "unified_attention", "paged", "attn", "attention", "batchdecode", "batchprefill",\n                   "reshape_and_cache")),\n        ("gdn_linear_attn", ("gdn", "delta_rule", "chunk_", "recurrent", "causal_conv", "l2norm", "solve_tril", "merge_16x16",\n                             "recompute_w_u", "kkt", "cumsum", "post_conv", "layer_norm_gated", "gated_delta")),\n        ("gemm_marlin", ("marlin",)),\n        ("gemm_other", ("gemm", "cutlass", "xmma", "cublas", "sm89_", "ampere_", "sm80_")),\n        ("sampling", ("sampl", "topk", "top_k", "argmax", "softmax", "rejection"))]\n\n\ndef _cat(name):\n    # rev 3h: "void (anonymous namespace)::gdn_decode_post_conv_mtp_kernel<...>(...)" used to reduce to "void " -> other\n    s = re.sub(r"\\(anonymous namespace\\)::", "", str(name))\n    s = re.sub(r"^\\s*void\\s+", "", s)\n    base = s.split("<")[0].split("(")[0].lower()\n    return next((k for k, pats in CATS if any(p in base for p in pats)), "other")\n\n\nCOMPUTE_CATS = ("nccl", "attention", "gdn_linear_attn", "gemm_marlin", "gemm_other")\n# rev 3h: vLLM 0.32 marks each model execution as execute_context_N(N)_generation_N(N) (no gpu_model_runner range)\nSCOPE_RE = re.compile(r"^execute_context_\\d+\\(\\d+\\)_generation_\\d+\\(\\d+\\)$")\n\n\ndef _is_scope(e):\n    n = str(e.get("name", ""))\n    return SCOPE_RE.match(n) is not None or ("forward" in n.lower() and "gpu_model_runner" in n.lower())\n\n\ndef _scope_kind(g, compute_eager, unresolved):\n    """FULL = graph-launched with no eager model compute (attention / GDN / GEMM / NCCL) inside the execution scope;\n    eager input-preparation kernels are counted apart and allowed. Unresolved eager launches never read FULL."""\n    if g == 0:\n        return "EAGER"\n    if compute_eager:\n        return "PIECEWISE"\n    if unresolved:\n        return "PIECEWISE" if g > 1 else "MIXED"\n    return "FULL"\n\n\ndef trace_summary(pdir):\n    files = sorted(Path(pdir).rglob("*.pt.trace.json*"), key=lambda p: p.stat().st_mtime)\n    if not files:\n        return {"error": "NO_TRACE_FILE", "listing": [str(p) for p in Path(pdir).rglob("*")][:20]}\n    f = next((p for p in files if re.search(r"rank0\\.|rank[-_]?0[._]|tp0_", p.name)), files[0])\n    with (gzip.open if f.name.endswith(".gz") else open)(f, "rt", encoding="utf-8") as fh:\n        tr = json.load(fh)\n    ev = tr.get("traceEvents", [])\n    kern = [e for e in ev if e.get("ph") == "X" and str(e.get("cat", "")).lower() in ("kernel", "gpu_kernel")]\n    rt = [e for e in ev if e.get("ph") == "X" and str(e.get("cat", "")).lower() in ("cuda_runtime", "cuda_driver")]\n    is_graph = lambda e: "GraphLaunch" in str(e.get("name", ""))\n    is_kl = lambda e: re.search(r"LaunchKernel|cuLaunchKernel", str(e.get("name", ""))) is not None\n    # per-scope counts inside the execution annotations (rev 3h: execute_context_*_generation_*; the older\n    # gpu_model_runner forward ranges still count). rev 3c: user_annotation only, not the gpu_user_annotation copies.\n    fw = sorted([e for e in ev if e.get("ph") == "X" and str(e.get("cat", "")) == "user_annotation" and _is_scope(e)],\n                key=lambda e: e.get("ts", 0))\n    kname = {(e.get("args") or {}).get("correlation"): str(e.get("name", "")) for e in kern\n             if (e.get("args") or {}).get("correlation") is not None}\n    per_fw, aux = [], {}\n    for a in fw[:400]:\n        lo, hi, tid = a.get("ts", 0), a.get("ts", 0) + a.get("dur", 0), a.get("tid")\n        inside = [e for e in rt if e.get("tid") == tid and lo <= e.get("ts", -1) <= hi]\n        kls = [e for e in inside if is_kl(e)]\n        ce = un = 0\n        for e in kls:\n            kn = kname.get((e.get("args") or {}).get("correlation"))\n            if kn is None:\n                un += 1\n            elif _cat(kn) in COMPUTE_CATS:\n                ce += 1\n            else:\n                b = re.sub(r"^\\s*void\\s+", "", re.sub(r"\\(anonymous namespace\\)::", "", kn)).split("<")[0].split("(")[0][:60]\n                aux[b] = aux.get(b, 0) + 1\n        per_fw.append([sum(1 for e in inside if is_graph(e)), len(kls), ce, un])\n    corr = {(e.get("args") or {}).get("correlation"): str(e.get("name", "")) for e in rt\n            if (e.get("args") or {}).get("correlation") is not None}\n    per, tot, names = {k: 0.0 for k, _ in CATS + [("other", ())]}, 0.0, {}\n    src = {k: {"graph": 0, "eager": 0, "unknown": 0} for k in per}\n    for e in kern:\n        d = float(e.get("dur", 0.0))\n        tot += d\n        names[e.get("name", "")] = names.get(e.get("name", ""), 0.0) + d\n        k = _cat(e.get("name", ""))\n        per[k] += d\n        launcher = corr.get((e.get("args") or {}).get("correlation"))\n        src[k]["graph" if launcher and "GraphLaunch" in launcher else ("eager" if launcher else "unknown")] += 1\n    tot_k = sum(v["graph"] + v["eager"] for v in src.values())\n    eager_share = round(sum(v["eager"] for v in src.values()) / tot_k, 3) if tot_k else None\n    fw_g = statistics.median([x[0] for x in per_fw]) if per_fw else None\n    fw_k = statistics.median([x[1] for x in per_fw]) if per_fw else None\n    kinds_l = [_scope_kind(g, ce, un) for g, k, ce, un in per_fw]\n    if per_fw:                   # rev 3f (GPT rev 3e audit: 2 FULL + 1 EAGER forwards read FULL by the median)\n        kinds = set(kinds_l)\n        verdict = kinds.pop() if len(kinds) == 1 else "MIXED"       # every scope must agree\n        basis = "per_scope_annotation (all scopes; eager input-prep kernels allowed, eager model compute is not)"\n    elif tot_k:                  # rev 3d (GPT-2 reproduced FULL with 0 forwards): CUPTI alone never proves FULL\n        att = src["attention"]\n        hint = ("FULL" if att["eager"] == 0 and (eager_share or 0) < 0.05 else\n                "PIECEWISE" if att["eager"] > 0 and any(v["graph"] for v in src.values()) else\n                "EAGER" if not any(v["graph"] for v in src.values()) else "MIXED")\n        verdict, basis = "UNPROVEN", f"no forward annotations (cupti hint: {hint})"\n    else:\n        verdict, basis = "UNKNOWN", "none"\n    return {"file": f.name, "files": len(files), "kernels": len(kern), "kernel_us_sum": round(tot),\n            "graph_launches": sum(1 for e in rt if is_graph(e)), "kernel_launches": sum(1 for e in rt if is_kl(e)),\n            "forwards": len(per_fw), "forward_kinds": {k_: kinds_l.count(k_) for k_ in ("FULL", "PIECEWISE", "EAGER", "MIXED")},\n            "per_forward_median_graph_launches": fw_g, "per_forward_median_kernel_launches": fw_k,\n            "per_forward_median_compute_eager": statistics.median([x[2] for x in per_fw]) if per_fw else None,\n            "per_forward_unresolved_eager": sum(x[3] for x in per_fw),\n            "scope_names": sorted({str(a.get("name", "")) for a in fw})[:5],\n            "aux_eager_kernels_in_scopes": dict(sorted(aux.items(), key=lambda x: -x[1])[:20]),\n            "launch_source": src, "eager_kernel_share": eager_share, "graph_verdict": verdict, "verdict_basis": basis,\n            "by_category_pct": {k: round(100 * v / tot, 1) for k, v in per.items()} if tot else {},\n            "top_kernels_us": dict(sorted(((k, round(v)) for k, v in names.items()), key=lambda x: -x[1])[:25]),\n            "note": "kernel time sums overlapping streams; NCCL time includes rank-skew waiting"}\n\n\ndef profile(port, pdir, secs=6.0, delay_iters=20, max_iters=40):\n    """POST /start_profile (bounded by iterations; falls back to a bare POST), decode, POST /stop_profile, summary."""\n    rec = {}\n    try:\n        U.urlopen(U.Request(f"http://127.0.0.1:{port}/start_profile", method="POST",\n                            data=json.dumps({"delay_iterations": delay_iters, "max_iterations": max_iters}).encode(),\n                            headers={"Content-Type": "application/json"}), timeout=900).read()\n        rec["start_profile"] = "ok_bounded"\n    except Exception as e:\n        rec["start_profile_bounded_error"] = repr(e)[:200]\n        try:\n            U.urlopen(U.Request(f"http://127.0.0.1:{port}/start_profile", data=b"", method="POST"), timeout=900).read()\n            rec["start_profile"] = "ok"\n        except Exception as e2:\n            rec["start_profile"] = repr(e2)[:200]\n    time.sleep(secs)\n    try:\n        U.urlopen(U.Request(f"http://127.0.0.1:{port}/stop_profile", data=b"", method="POST"), timeout=900).read()\n        rec["stop_profile"] = "ok"\n    except Exception as e:\n        rec["stop_profile"] = repr(e)[:200]\n    time.sleep(20)\n    try:\n        rec["trace"] = trace_summary(pdir)\n    except Exception as e:\n        rec["trace"] = {"error": f"TRACE_PARSE {type(e).__name__}: {e}"[:300]}\n    return rec\n\n\ndef profiled_load(port, prompts, level, pdir, secs=6.0, max_new=8192, model="q27", prefill_tok_s=700.0, spec=False):\n    """`level` decoding streams (own window: the profiler slows decode), profile() only after prefill settled exactly\n    as in steady() (prompt counter + preemptions flat, every running request emitted and gaining tokens). rev 3d: no\n    profile when unsettled; the profile window must be decode-only (prompt counter flat) and, with `spec`, show\n    speculative drafts (else the trace cannot speak for multi-token verification)."""\n    plen = len(prompts[0]) if prompts else 0\n    settle = max(300.0, 120.0 + level * plen / prefill_tok_s)\n    ss = [Stream(port, prompts[i % len(prompts)], max_new, 1.0, 4321 + i, model=model) for i in range(level)]\n    for g in ss:\n        g.start()\n    rec, t0, flat, ok, prev, last_tok = {"level": level}, time.perf_counter(), None, 0, None, [0] * level\n    try:\n        while time.perf_counter() - t0 < settle:\n            time.sleep(2)\n            try:\n                m = prom(port)\n            except Exception:\n                continue\n            run, wait, pt, pre = int(m.get(RUN, 0)), int(m.get(WAIT, 0)), m.get(PROMPT), m.get(PREEMPT, 0)\n            now_tok = [len(g.tokens) for g in ss]\n            gaining = sum(1 for a, b in zip(last_tok, now_tok) if b > a)\n            last_tok = now_tok\n            first = sum(g.t_first is not None for g in ss)\n            calm = run > 0 and prev is not None and (pt, pre) == prev and first >= run and gaining >= run\n            if calm and run >= level and wait == 0:\n                ok += 1\n            elif calm and wait > 0:\n                flat = flat or time.perf_counter()\n                ok = ok + 1 if time.perf_counter() - flat > 30 else 0\n            else:\n                ok, flat = 0, None\n            prev = (pt, pre)\n            if ok >= 3:\n                break\n        rec["settled"] = ok >= 3\n        if not rec["settled"]:\n            rec["status"] = "INCONCLUSIVE_PREFILL_NOT_SETTLED"\n            return rec\n        try:\n            m0 = prom(port)\n            rec["running_at_start"] = m0.get(RUN)\n        except Exception:\n            m0 = {}\n        rec.update(profile(port, pdir, secs))\n        try:\n            m1 = prom(port)\n        except Exception:\n            m1 = {}\n        rec["window_deltas"] = {"prompt_tokens": m1.get(PROMPT, 0) - m0.get(PROMPT, 0), "drafts": m1.get(DRF, 0) - m0.get(DRF, 0),\n                                "accepted": m1.get(ACC, 0) - m0.get(ACC, 0), "generated": m1.get(GEN, 0) - m0.get(GEN, 0)}\n        if not m0 or not m1:\n            rec["status"] = "INCONCLUSIVE_METRICS_UNAVAILABLE"\n        elif rec["window_deltas"]["prompt_tokens"]:\n            rec["status"] = "INCONCLUSIVE_PREFILL_IN_PROFILE"\n        elif spec and not (rec["window_deltas"]["drafts"] and rec["window_deltas"]["accepted"]):\n            rec["status"] = "INCONCLUSIVE_NO_SPEC_DECODE"            # rev 3e: drafts AND accepted tokens in the window\n        return rec\n    finally:\n        for g in ss:\n            g.stop()\n        for g in ss:\n            g.join(10)\n        rec["idle_after"] = wait_idle(port)\n')
(T / "gate_tests.py").write_text('"""E23 rev 3d gate tests (four GPT audits, REV3C_GPT4_PRIORITY_FIX_LIST #11): injected faults against the REAL gate code.\nThe notebook functions are taken from the generated FULL notebook code by name (ast), executed with stubs, and driven\nwith the cases each audit listed. Usage: python gate_tests.py <nb_FULL.py> <e23_lab.py>. Prints `GATES {json}` and\nexits 0 only when every case gives the expected result."""\nimport ast\nimport importlib.util\nimport json\nimport sys\nimport types\nfrom pathlib import Path\n\nNB, LABP = sys.argv[1], sys.argv[2]\nspec = importlib.util.spec_from_file_location("e23_lab", LABP)\nL = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(L)\nsrc = Path(NB).read_text(encoding="utf-8")\ntree = ast.parse(src)\nWANT = {"oracle_gate", "gated", "nvfp4_gate", "graph_proof", "s2", "s4_thread", "_killpg", "_windows", "_num", "taint_check",\n        "s1_pp4", "_done"}\ncode = {}\nfor node in ast.walk(tree):\n    if isinstance(node, ast.FunctionDef) and node.name in WANT and node.name not in code:\n        code[node.name] = ast.get_source_segment(src, node)\nmissing = sorted(WANT - set(code))\nRES, BAD = {"missing_functions": missing}, []\n\n\ndef check(name, got, want):\n    ok = got == want if not callable(want) else bool(want(got))\n    RES[name] = {"got": got, "ok": ok}\n    if not ok:\n        BAD.append(name)\n\n\nimport re, subprocess, os, signal, hashlib, time   # noqa: E401  (names the extracted functions use)\nG = {"CLEAN": ("EXACT", "NEAR_TIE"), "R": {"sections": {}, "setup": {}, "errors": [], "inputs": {}}, "L": L, "json": json,\n     "re": re, "subprocess": subprocess, "os": os, "signal": signal, "Path": Path, "FAKE": None, "SM": False, "LITE": False, "TG": False,\n     "NVSMI_OK": True, "ABORT": {"why": None}, "time": time, "hashlib": hashlib}\nfor node in ast.walk(tree):            # rev 3f: module-level constants the fault gate uses\n    if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name) \\\n            and node.targets[0].id in ("XID", "XID_RESET", "HEALTH_CODE", "NV_BANDS") and node.targets[0].id not in G:\n        exec(ast.get_source_segment(src, node), G)\nfor node in ast.walk(tree):\n    if isinstance(node, ast.FunctionDef) and node.name in ("gpu_health", "fault_gate") and node.name not in code:\n        code[node.name] = ast.get_source_segment(src, node)\nfor n in ("_windows", "_num", "oracle_gate", "gated", "nvfp4_gate", "graph_proof", "_killpg", "gpu_health", "fault_gate",\n          "s2", "s4_thread", "taint_check"):\n    if n in code:\n        exec(code[n], G)\n\n# ---- oracle_gate / gated (#3)\nog = G["oracle_gate"]\ncheck("gate_clean", og({"oracle": {"errors": [None], "garbage": [None], "vs_x": {"status": "NEAR_TIE"}}}, need=("x",)), None)\ncheck("gate_fail_margin", og({"oracle": {"errors": [None], "garbage": [None], "vs_x": {"status": "FAIL_HIGH_MARGIN"}}}, need=("x",)),\n      "NO_GO_ORACLE")\ncheck("gate_request_error", og({"oracle": {"errors": ["timeout"], "garbage": [None], "vs_x": {"status": "EXACT"}}}, need=("x",)),\n      "NO_GO_ORACLE_REQUEST_ERROR")\ncheck("gate_garbage", og({"oracle": {"errors": [None], "garbage": ["LOW_VARIETY"]}}), "NO_GO_ORACLE_GARBAGE")\ncheck("gate_missing_ref", og({"oracle": {"errors": [None], "garbage": [None]}}, need=("x",)), "INCONCLUSIVE_NO_ORACLE")\ncheck("gate_inconclusive", og({"oracle": {"errors": [None], "garbage": [None], "vs_x": {"status": "INCONCLUSIVE"}}}, need=("x",)),\n      "INCONCLUSIVE_ORACLE")\no = G["gated"]({"oracle": {"errors": [None], "garbage": [None], "vs_x": {"status": "FAIL_HIGH_MARGIN"}}}, need=("x",))\ncheck("gated_nested_fail_not_done", (o.get("status"), o.get("correctness")), ("NO_GO_ORACLE", "NO_GO_ORACLE"))\n\n# ---- compare_tf completeness (#1a); rev 3h: [logprob, rank, rank-1 token id] and true top-1 agreement\nfull = {"positions": [[-0.1, 1, 5]] * 256, "n": 256}\none = {"positions": [[-0.1, 1, 5]] + [None] * 255, "n": 256}\ncheck("tf_1_of_256_inconclusive", L.compare_tf(full, one)["status"], "INCONCLUSIVE_INCOMPLETE_TF")\ncheck("tf_missing_ref", L.compare_tf(None, full)["status"], "INCONCLUSIVE")\ncheck("tf_complete_measured", L.compare_tf(full, full)["status"], "MEASURED")\nbad_tf = {"positions": [[-25.0, 9, 7]] * 256, "n": 256}\nr = L.compare_tf(full, bad_tf)\ncheck("tf_bad_is_measurement_only", (r["status"], r["rank1_agreement"]), ("MEASURED", 0.0))\n# the GPT SMOKE-audit repro: forced token rank 2 vs 3 (neither rank 1) with different top-1 ids used to read 1.0\nr23 = L.compare_tf({"positions": [[-1.0, 2, 11]] * 256}, {"positions": [[-1.2, 3, 12]] * 256})\ncheck("tf_rank2_vs_rank3_diff_top1_is_0", (r23["status"], r23["rank1_agreement"]), ("MEASURED", 0.0))\nr23s = L.compare_tf({"positions": [[-1.0, 2, 11]] * 256}, {"positions": [[-1.2, 3, 11]] * 256})\ncheck("tf_same_top1_is_1", r23s["rank1_agreement"], 1.0)\ncheck("tf_top1_missing_inconclusive", L.compare_tf(full, {"positions": [[-0.1, 1, None]] * 256})["status"],\n      "INCONCLUSIVE_TOP1_MISSING")\ncheck("tf_old_two_field_rows_inconclusive", L.compare_tf(full, {"positions": [[-0.1, 1]] * 256})["status"],\n      "INCONCLUSIVE_TOP1_MISSING")\ncheck("tf_complete_needs_top1", (L.tf_complete({"40k": full}), L.tf_complete({"40k": {"positions": [[-0.1, 1, None]] * 256}})),\n      (True, False))\ncheck("tok_id_parse", (L._tok_id("token_id:42"), L._tok_id("17"), L._tok_id("abc")), (42, 17, "str:abc"))\n\n\n# rev 3h: teacher_forced() on a vLLM 0.32-shaped /v1/completions response (the fake server has no prompt_logprobs)\nclass _Resp:\n    def __init__(self, obj):\n        self.b = json.dumps(obj).encode()\n\n    def read(self):\n        return self.b\n\n\n_pl = [None, {"5": {"logprob": -0.1, "rank": 1, "decoded_token": "a"}},\n       {"6": {"logprob": -2.0, "rank": 2, "decoded_token": "b"}, "9": {"logprob": -0.2, "rank": 1, "decoded_token": "c"}},\n       {"7": {"logprob": -3.0, "rank": 4}}]\n_orig_urlopen = L.U.urlopen\nL.U.urlopen = lambda *a, **k: _Resp({"choices": [{"prompt_logprobs": _pl}]})\ntry:\n    _tf = L.teacher_forced(1, [1, 5, 6, 7], last=3)\nfinally:\n    L.U.urlopen = _orig_urlopen\ncheck("tf_parse_vllm_shape", _tf["positions"], [[-0.1, 1, 5], [-2.0, 2, 9], [-3.0, 4, None]])\n\n\n# ---- nvfp4_gate (rev 3e: BF16-KV reference; NVFP4 within 3x the FP8-KV drift, rank-1 within 0.02)\ndef band(r1, mean, p99):\n    return {"status": "MEASURED", "n": 256, "rank1_agreement": r1, "mean_abs_dlogprob": mean, "p99_abs_dlogprob": p99,\n            "max_abs_dlogprob": p99 * 1.5}\n\n\ndef both(b):                          # rev 3h: the gate needs 40k AND 90k on both TF arms\n    return {"40k": b, "90k": dict(b)}\n\n\ndef sec5(tgt="DONE", orc="NEAR_TIE", ref="DONE", f8="DONE", nv="DONE", nv_tf=None, f8_tf=None):\n    nv_tf = both(nv_tf["40k"]) if nv_tf and set(nv_tf) == {"40k"} else nv_tf\n    f8_tf = both(f8_tf["40k"]) if f8_tf and set(f8_tf) == {"40k"} else f8_tf\n    G["R"]["sections"]["5_nvfp4_kv"] = {\n        "nvfp4_target_only": {"status": tgt, "oracle": {"vs_fp8_mtp0": {"status": orc}}},\n        "tf_bf16kv_reference": {"status": ref},\n        "tf_fp8kv_vs_bf16": {"status": f8, "teacher_forced": f8_tf or both(band(0.97, 0.03, 0.4))},\n        "tf_nvfp4_vs_bf16": {"status": nv, "teacher_forced": nv_tf or both(band(0.96, 0.07, 0.9))}}\n    return G["nvfp4_gate"]()\n\n\ndef sec5_raw(nv_tf, f8_tf=None):            # no auto-completion of the 90k band\n    G["R"]["sections"]["5_nvfp4_kv"] = {\n        "nvfp4_target_only": {"status": "DONE", "oracle": {"vs_fp8_mtp0": {"status": "NEAR_TIE"}}},\n        "tf_bf16kv_reference": {"status": "DONE"},\n        "tf_fp8kv_vs_bf16": {"status": "DONE", "teacher_forced": f8_tf or both(band(0.97, 0.03, 0.4))},\n        "tf_nvfp4_vs_bf16": {"status": "DONE", "teacher_forced": nv_tf}}\n    return G["nvfp4_gate"]()\n\n\ncheck("nv_within_3x_fp8_allowed", sec5()[0], True)\ncheck("nv_tf_inconclusive_blocks", sec5(nv="INCONCLUSIVE_TF")[0], False)\ncheck("nv_missing_oracle_blocks", sec5(orc="INCONCLUSIVE_NO_ORACLE")[0], False)\ncheck("nv_fail_margin_blocks", sec5(orc="FAIL_HIGH_MARGIN")[0], False)\ncheck("nv_target_not_done_blocks", sec5(tgt="NO_GO_ORACLE_REQUEST_ERROR")[0], False)\ncheck("nv_no_fp8_calibration_blocks", sec5(f8="INCONCLUSIVE_TF")[0], False)\ncheck("nv_no_bf16_reference_blocks", sec5(ref="INCONCLUSIVE_TF")[0], False)\ncheck("nv_24nat_0pct_blocks", sec5(nv_tf={"40k": band(0.0, 24.95, 30.0)})[0], False)\ncheck("nv_rank1_drop_blocks", sec5(nv_tf={"40k": band(0.93, 0.03, 0.4)})[0], False)\ncheck("nv_p99_4x_blocks", sec5(nv_tf={"40k": band(0.97, 0.03, 1.7)})[0], False)\ncheck("nv_exact_fp8_floors", sec5(f8_tf={"40k": band(1.0, 0.0, 0.0)}, nv_tf={"40k": band(0.985, 0.015, 0.15)})[0], True)\ncheck("nv_none_metric_no_crash", sec5(nv_tf={"40k": {"status": "MEASURED", "n": 256, "rank1_agreement": None}})[0], False)\n_g = sec5_raw({"40k": band(0.96, 0.07, 0.9)})                      # rev 3h (GPT SMOKE audits): 40k only must STOP\ncheck("nv_40k_only_stops", (_g[0], _g[1].get("why")), (False, "MISSING_REQUIRED_TF_BANDS"))\n_g = sec5_raw(both(band(0.96, 0.07, 0.9)), f8_tf={"40k": band(0.97, 0.03, 0.4)})\ncheck("nv_fp8_calibration_40k_only_stops", (_g[0], _g[1].get("why")), (False, "MISSING_REQUIRED_TF_BANDS"))\n_g = sec5_raw({"40k": band(0.96, 0.07, 0.9), "90k": {**band(0.96, 0.07, 0.9), "n": 200}})\ncheck("nv_90k_short_band_stops", (_g[0], _g[1].get("why")), (False, "INVALID_TF_BAND"))\n_g = sec5_raw({"40k": band(0.96, 0.07, 0.9), "90k": {"status": "INCONCLUSIVE_TOP1_MISSING"}})\ncheck("nv_90k_top1_missing_stops", (_g[0], _g[1].get("why")), (False, "INVALID_TF_BAND"))\ncheck("nv_both_bands_pass", sec5_raw(both(band(0.96, 0.07, 0.9)))[0], True)\n\n# ---- graph_proof + trace_summary (#6)\ngp = G["graph_proof"]\ncheck("gp_unsettled", gp({"profile": {"status": "INCONCLUSIVE_PREFILL_NOT_SETTLED"}})["status"], "INCONCLUSIVE_PREFILL_NOT_SETTLED")\ncheck("gp_unproven", gp({"profile": {"trace": {"graph_verdict": "UNPROVEN"}}})["status"], "INCONCLUSIVE_GRAPH")\ncheck("gp_downgrade", gp({"profile": {"trace": {"graph_verdict": "FULL"}},\n                          "servers": {"a": {"graph_downgrade_lines": ["setting cudagraph_mode=PIECEWISE"]}}}, expect="FULL")["status"],\n      "DOWNGRADED")\ncheck("gp_downgrade_to_full_decode_only_not_counted", gp({"profile": {"trace": {"graph_verdict": "FULL"}},\n      "servers": {"a": {"graph_downgrade_lines": ["Overriding cudagraph_mode to FULL_DECODE_ONLY"]}}}, expect="FULL")["status"],\n      "PROVEN_FULL")\ncheck("gp_downgrade_032_wording", gp({"profile": {"trace": {"graph_verdict": "FULL"}},\n      "servers": {"a": {"graph_downgrade_lines": ["Overriding cudagraph_mode to PIECEWISE"]}}}, expect="FULL")["status"], "DOWNGRADED")\ncheck("gp_g0_mismatch", gp({"profile": {"trace": {"graph_verdict": "FULL"}}}, expect="PIECEWISE")["status"], "MISMATCH_EXPECTED")\ncheck("gp_full_proven", gp({"profile": {"trace": {"graph_verdict": "FULL"}}}, expect="FULL")["status"], "PROVEN_FULL")\ntdir = Path(NB).parent / "gate_trace"\ntdir.mkdir(exist_ok=True)\nev = [{"ph": "X", "cat": "cuda_runtime", "name": "cudaGraphLaunch", "ts": 10, "dur": 2, "tid": 1, "args": {"correlation": 7}},\n      {"ph": "X", "cat": "kernel", "name": "flashinfer_attn", "ts": 12, "dur": 5, "tid": 2, "args": {"correlation": 7}}]\n(tdir / "x_rank0.1.pt.trace.json").write_text(json.dumps({"traceEvents": ev}))\ncheck("trace_no_annotation_unproven", L.trace_summary(tdir)["graph_verdict"], "UNPROVEN")\nev2 = ev + [{"ph": "X", "cat": "user_annotation", "name": "gpu_model_runner: forward", "ts": 5, "dur": 20, "tid": 1},\n            {"ph": "X", "cat": "gpu_user_annotation", "name": "gpu_model_runner: forward", "ts": 5, "dur": 20, "tid": 9},\n            {"ph": "X", "cat": "gpu_user_annotation", "name": "gpu_model_runner: forward", "ts": 6, "dur": 20, "tid": 9}]\n(tdir / "x_rank0.1.pt.trace.json").write_text(json.dumps({"traceEvents": ev2}))\ncheck("trace_cpu_annotation_full", L.trace_summary(tdir)["graph_verdict"], "FULL")\n\n# ---- s2 (P5) release + result gate (#7) and s4_thread (P4) synthetic-weight gate (#10): stub bench scripts\nstub = Path(NB).parent / "gate_stub"\nstub.mkdir(exist_ok=True)\nG["OUT"] = stub\nG["STOCK"] = {"py": sys.executable, "env": dict(os.environ)}\nG["FP8"] = stub\n\n\ndef kit_with(name, payload):\n    (stub / name).write_text("import json, sys\\na = sys.argv\\nout = a[a.index(\'--out\') + 1]\\n"\n                             f"json.dump({payload!r}, open(out, \'w\'))\\n")\n    return str(stub / name)\n\n\nclass _L:\n    rel = True\n\n    @staticmethod\n    def wait_gpu_release(*a, **k):\n        return _L.rel\n\n\nG["L"] = _L\nG["kit"] = lambda n: kit_with(n, {"summary": {"all_checks_pass": True}}) if n.endswith("host_ar.py") else str(stub / n)\n_L.rel = False\nG["ABORT"]["why"] = None\ncheck("p5_release_false_fails", (G["s2"]()["status"], bool(G["ABORT"]["why"])), ("FAIL_GPU_RELEASE", True))\n_L.rel, G["ABORT"]["why"] = None, None\ncheck("p5_release_none_fails", G["s2"]()["status"], "FAIL_GPU_RELEASE")\n_L.rel, G["ABORT"]["why"] = True, None\ncheck("p5_clean_done", G["s2"]()["status"], "DONE")\nG["kit"] = lambda n: kit_with(n, {"summary": {"all_checks_pass": False}}) if n.endswith("host_ar.py") else str(stub / n)\ncheck("p5_checks_failed", G["s2"]()["status"], "CHECK_FAILED")\nG["kit"] = lambda n: kit_with(n, {"all_ok": True, "shapes": {"gate_up_proj": {"weights_source": {"kind": "synthetic"}}}}) \\\n    if n.endswith("fp8_gemm.py") else str(stub / n)\nG["P4_PROC"] = {"p": None}\nr4 = G["s4_thread"]()\ncheck("p4_synthetic_rejected", (r4["status"], r4.get("why")), ("CHECK_FAILED", "SYNTHETIC_WEIGHTS_NOT_ALLOWED"))\nG["kit"] = lambda n: kit_with(n, {"all_ok": True, "shapes": {"gate_up_proj": {"weights_source": {"kind": "real"}}}}) \\\n    if n.endswith("fp8_gemm.py") else str(stub / n)\ncheck("p4_real_done", G["s4_thread"]()["status"], "DONE")\n\n# ---- rev 3e: garbage-under-load windows and failed probes are NO_GO (audits B-O1, C-F2)\ncheck("gate_garbage_window", og({"oracle": {"errors": [None], "garbage": [None]}, "steady": {"40k@16": {"status": "FAIL_GARBAGE_UNDER_LOAD"}}}),\n      "NO_GO_GARBAGE_UNDER_LOAD")\ncheck("gate_garbage_replica", og({"oracle": {"errors": [None], "garbage": [None]},\n                                  "40k": {"replicas": [{"status": "STEADY"}, {"status": "FAIL_GARBAGE_UNDER_LOAD"}]}}),\n      "NO_GO_GARBAGE_UNDER_LOAD")\ncheck("gate_probe_fail_real", og({"oracle": {"errors": [None], "garbage": [None]}, "probes": {"ok": False}}), "NO_GO_SHORT_PROBE")\nG["FAKE"] = "fake"\ncheck("gate_probe_fail_fake_ignored", og({"oracle": {"errors": [None], "garbage": [None]}, "probes": {"ok": False}}), None)\nG["FAKE"] = None\n\n# ---- rev 3f: per-forward graph verdict (GPT rev 3e audit: 2 FULL + 1 EAGER read FULL by the median)\ndef fw_trace(kinds):\n    evs, ts = [], 0\n    for i, k in enumerate(kinds):\n        lo = ts\n        evs.append({"ph": "X", "cat": "user_annotation", "name": "gpu_model_runner: forward", "ts": lo, "dur": 50, "tid": 1})\n        if k == "FULL":\n            evs.append({"ph": "X", "cat": "cuda_runtime", "name": "cudaGraphLaunch", "ts": lo + 5, "dur": 1, "tid": 1, "args": {}})\n        elif k == "PIECEWISE":\n            for j in range(3):\n                evs.append({"ph": "X", "cat": "cuda_runtime", "name": "cudaGraphLaunch", "ts": lo + 5 + j, "dur": 1, "tid": 1, "args": {}})\n            evs.append({"ph": "X", "cat": "cuda_runtime", "name": "cudaLaunchKernel", "ts": lo + 20, "dur": 1, "tid": 1, "args": {}})\n        else:\n            for j in range(4):\n                evs.append({"ph": "X", "cat": "cuda_runtime", "name": "cudaLaunchKernel", "ts": lo + 5 + j, "dur": 1, "tid": 1, "args": {}})\n        ts += 100\n    (tdir / "x_rank0.1.pt.trace.json").write_text(json.dumps({"traceEvents": evs}))\n    return L.trace_summary(tdir)["graph_verdict"]\n\n\ncheck("trace_all_full", fw_trace(["FULL"] * 3), "FULL")\ncheck("trace_2full_1eager_mixed", fw_trace(["FULL", "FULL", "EAGER"]), "MIXED")\ncheck("trace_all_piecewise", fw_trace(["PIECEWISE"] * 3), "PIECEWISE")\n\n\n# ---- rev 3h (GPT SMOKE audits): vLLM 0.32 scopes are execute_context_N(N)_generation_N(N); eager input-prep kernels\n# inside a FULL scope are allowed, eager model compute (attention / GDN / GEMM / NCCL) is PIECEWISE, unresolved is not FULL\ndef scope_trace(kinds, name="execute_context_0(0)_generation_16(64)"):\n    evs, ts, c = [], 0, 1000\n    for k in kinds:\n        lo = ts\n        evs.append({"ph": "X", "cat": "user_annotation", "name": name, "ts": lo, "dur": 80, "tid": 1})\n        ng = {"FULL": 1, "FULL_PREP": 1, "PIECEWISE_REAL": 5, "ONE_GRAPH_UNRESOLVED": 1}[k]\n        for j in range(ng):\n            c += 1\n            evs.append({"ph": "X", "cat": "cuda_runtime", "name": "cudaGraphLaunch", "ts": lo + 2 + j, "dur": 1, "tid": 1,\n                        "args": {"correlation": c}})\n            evs.append({"ph": "X", "cat": "kernel", "name": "void marlin::Marlin<1>(int4 const*)", "ts": lo + 2 + j, "dur": 3,\n                        "tid": 7, "args": {"correlation": c}})\n        eager = {"FULL": [], "FULL_PREP": ["_prepare_rope_positions_kernel", "void at::native::vectorized_elementwise_kernel<4>()"],\n                 "PIECEWISE_REAL": ["void flashinfer::BatchPrefillWithPagedKVCacheKernel<true>()",\n                                    "void (anonymous namespace)::gdn_decode_post_conv_mtp_kernel<float, 3, false>()"],\n                 "ONE_GRAPH_UNRESOLVED": [None]}[k]\n        for j, kn in enumerate(eager):\n            c += 1\n            evs.append({"ph": "X", "cat": "cuda_runtime", "name": "cudaLaunchKernel", "ts": lo + 20 + j, "dur": 1, "tid": 1,\n                        "args": {"correlation": c}})\n            if kn:\n                evs.append({"ph": "X", "cat": "kernel", "name": kn, "ts": lo + 20 + j, "dur": 2, "tid": 7, "args": {"correlation": c}})\n        ts += 100\n    (tdir / "x_rank0.1.pt.trace.json").write_text(json.dumps({"traceEvents": evs}))\n    return L.trace_summary(tdir)\n\n\ncheck("scope_full_with_prep_kernels", scope_trace(["FULL_PREP"] * 3)["graph_verdict"], "FULL")\ncheck("scope_piecewise_attention_gdn_eager", scope_trace(["PIECEWISE_REAL"] * 3)["graph_verdict"], "PIECEWISE")\ncheck("scope_one_graph_unresolved_mixed", scope_trace(["ONE_GRAPH_UNRESOLVED"] * 3)["graph_verdict"], "MIXED")\ncheck("scope_full_and_piecewise_mixed", scope_trace(["FULL", "PIECEWISE_REAL"])["graph_verdict"], "MIXED")\n_st = scope_trace(["FULL_PREP"] * 2)\ncheck("scope_counts", (_st["forwards"], _st["per_forward_median_compute_eager"], _st["per_forward_unresolved_eager"]), (2, 0, 0))\ncheck("cat_anonymous_namespace_gdn", L._cat("void (anonymous namespace)::gdn_decode_post_conv_mtp_kernel<float, 3, false>(x)"),\n      "gdn_linear_attn")\ncheck("cat_reshape_and_cache_attention", L._cat("void vllm::reshape_and_cache_flash_kernel<x>(y)"), "attention")\ncheck("scope_other_names_ignored", scope_trace(["FULL"], name="execute_model_misc")["graph_verdict"], "UNPROVEN")\n\n# ---- rev 3h: P5 int8_eval pass rule (elementwise all within the bound, finite worst <= 1), when the kit bench is given\nif len(sys.argv) > 3:\n    _sb = importlib.util.spec_from_file_location("bench_host_ar", sys.argv[3])\n    B = importlib.util.module_from_spec(_sb)\n    _sb.loader.exec_module(B)\n    import torch\n    n = 1024\n    xs = [torch.randn(n, dtype=torch.float32) for _ in range(4)]\n    ex = torch.stack([x.double() for x in xs]).sum(0)\n    check("p5_eval_exact_ok", B.int8_eval(torch, xs, ex.float(), n, 4, torch.float32)["ok"], True)\n    bnd = B.int8_bound(torch, xs, ex, n, 4, torch.float32)\n    near = (ex + 0.99 * bnd).float()\n    ev = B.int8_eval(torch, xs, near, n, 4, torch.float32)\n    check("p5_eval_099_bound_ok", (ev["ok"], ev["worst_err_over_bound"] <= 1.0), (True, True))\n    over = ex.clone()\n    over[5] += 1.5 * bnd[5]\n    check("p5_eval_over_bound_fails", B.int8_eval(torch, xs, over.float(), n, 4, torch.float32)["ok"], False)\n    xs_nan = [x.clone() for x in xs]\n    xs_nan[0][40] = float("nan")\n    ok_nan = ex.clone()\n    ok_nan[32:64] = float("nan")\n    check("p5_eval_nan_group_all_nan_ok", B.int8_eval(torch, xs_nan, ok_nan.float(), n, 4, torch.float32)["ok"], True)\n    bad_nan = ex.clone()\n    bad_nan[40] = float("nan")\n    check("p5_eval_nan_group_partial_fails", B.int8_eval(torch, xs_nan, bad_nan.float(), n, 4, torch.float32)["ok"], False)\n    big = 245761                                    # the SMOKE fp32 size: mean of all-True reads 0.9999999999999999\n    xb = [torch.randn(big, dtype=torch.float32) for _ in range(4)]\n    eb = torch.stack([x.double() for x in xb]).sum(0)\n    evb = B.int8_eval(torch, xb, eb.float(), big, 4, torch.float32)\n    check("p5_eval_big_all_within_ok", (evb["ok"], evb["all_within_bound"]), (True, True))\nelse:\n    RES["p5_eval_tests"] = "not run (no bench path)"\n    BAD.append("p5_eval_tests_not_run")\n\n# ---- rev 3f: Xid recovery classes + GPU health probe\nG.update(FAKE=None, ABORT={"why": None}, STOCK={"py": sys.executable, "env": dict(os.environ)})\nG["R"]["setup"] = {}\nG["gpu_health"] = lambda: {"ok": True}\ncheck("xid_reset_class_aborts", (G["fault_gate"](["NVRM: Xid (PCI:0000:00:04): 79, GPU has fallen off the bus"], "armX"),\n                                 bool(G["ABORT"]["why"])), ("ABORT", True))\nG["ABORT"]["why"] = None\ncheck("app_fault_healthy_continues", (G["fault_gate"](["CUDA error: an illegal memory access was encountered"], "armY"),\n                                      G["ABORT"]["why"]), ("CONTINUE", None))\nG["gpu_health"] = lambda: {"ok": False}\ncheck("app_fault_unhealthy_aborts", (G["fault_gate"](["CUDA error: an illegal memory access was encountered"], "armZ"),\n                                     bool(G["ABORT"]["why"])), ("ABORT", True))\nG["ABORT"]["why"] = None\ncheck("no_fault_line_none", G["fault_gate"](["all good"], "armW"), None)\nG["gpu_health"] = lambda: {"ok": True}\n\n# ---- rev 3e: a GPU fault marks only its arm; later arms are labelled, not skipped; _done needs DONE + CLEAN\nG["R"]["setup"] = {}\na1 = G["taint_check"]({"status": "DONE", "correctness": "CLEAN", "servers": {"pp4c": {"xid": True, "xid_lines": ["CUDA error: an illegal memory access was encountered"]}}})\na2 = G["taint_check"]({"status": "DONE", "correctness": "CLEAN", "servers": {"next": {"xid": False}}})\ncheck("taint_arm_no_go", (a1["status"], a1["correctness"]), ("NO_GO_GPU_FAULT", "NO_GO_GPU_FAULT"))\ncheck("taint_later_arm_labelled_not_blocked", (a2["status"], a2.get("after_gpu_fault")), ("DONE", ["pp4c"]))\nexec(code.get("_done", "def _done(path):\\n    raise RuntimeError(\'missing\')"), G)\nG["R"]["sections"] = {"s": {"ok": {"status": "DONE", "correctness": "CLEAN", "after_gpu_fault": ["pp4c"]},\n                            "nogo": {"status": "NO_GO_ORACLE", "correctness": "NO_GO_ORACLE"},\n                            "dirty": {"status": "DONE", "correctness": "NO_GO_GARBAGE_UNDER_LOAD"}}}\ncheck("aa_picks_clean_after_fault", G["_done"]("s/ok"), True)\ncheck("aa_skips_no_go", G["_done"]("s/nogo"), False)\ncheck("aa_skips_unclean", G["_done"]("s/dirty"), False)\n\n# ---- rev 3e: PP4 staged fail-fast, counted requests (stubbed servers; the real s1_pp4)\nclass _Srv:\n    port, name = 1, "pp4"\n\n    def alive(self):\n        return True\n\n\nCALLS = []\n\n\ndef mk_greedy(bad_at=None):\n    def greedy(port, prompts, n_tok=0, timeout=0):\n        CALLS.append(len(prompts[0]))\n        tok = [1, 2, 3] if len(prompts[0]) != bad_at else []\n        return [{"tokens": tok, "text": "alpha beta gamma", "error": None, "margins": [2.0] * 3}]\n    return greedy\n\n\ndef run_pp4(probe_ok=True, bad_at=None, ref_clean=True):\n    CALLS.clear()\n    LV = []\n    Lp = types.SimpleNamespace(chat_probes=lambda port: {"ok": probe_ok}, greedy=mk_greedy(bad_at), compare=L.compare,\n                               degenerate=L.degenerate)\n    ref = [{"tokens": [1, 2, 3], "text": "alpha beta gamma", "error": None, "margins": [2.0] * 3}] * 4\n    P = [[0] * n for n in (2048, 2049, 8192, 40000)]\n    G.update(L=Lp, INT4="int4", ROWS={"int4_mtp0": ref} if ref_clean else {}, ORACLE_P=P[:3], LONG_P=P[3:], NTOK=3, SM=True,\n             left=lambda: 9e9, stock_flags=lambda *a, **k: [], stock_server=lambda *a, **k: _Srv(),\n             with_servers=lambda ss, t, body: body([_Srv()]), levels=lambda *a, **k: LV.append(a[1]) or {"40k@8": {"status": "STEADY"}})\n    exec(code["s1_pp4"], G)\n    G["taint_check"] = lambda o: o\n    out = G["s1_pp4"](True)()\n    return out.get("status"), list(CALLS), LV\n\n\ncheck("pp4_probe_fail_no_requests", run_pp4(probe_ok=False)[:2], ("NO_GO_SHORT_PROBE", []))\ncheck("pp4_2k_fail_stops", run_pp4(bad_at=2048)[:2], ("NO_GO_2k", [2048]))\ncheck("pp4_40k_fail_no_throughput", (lambda r: (r[0], r[2]))(run_pp4(bad_at=40000)), ("NO_GO_40k", []))\ncheck("pp4_all_pass_throughput", (lambda r: (r[0], r[1], r[2]))(run_pp4()), (None, [2048, 8192, 40000], ["40k"]))\ncheck("pp4_no_reference_skips", run_pp4(ref_clean=False)[0], "SKIPPED_NO_ORACLE")\nG["L"] = L\n\n# ---- rev 3e: EAGLE falls through on LAUNCH_FAILED / SERVER_DIED / NO_GO to the next variant (the real s7_eagle)\nfor node in ast.walk(tree):\n    if isinstance(node, ast.FunctionDef) and node.name == "s7_eagle":\n        code["s7_eagle"] = ast.get_source_segment(src, node)\n        break\n\n\ndef run_eagle(statuses):\n    seq = list(statuses)\n    names = []\n\n    def ws(ss, t, body):\n        names.append(ss[0])\n        return {"status": seq.pop(0), "servers": {}}\n    G.update(eagle_install=lambda: {"vllm": ["x"]}, EAGLE_DRAFT="d", FP8="f", ABORT={"why": None}, left=lambda: 9e9,\n             stock_flags=lambda *a, **k: [], stock_server=lambda name, *a, **k: name, with_servers=ws, taint_check=lambda o: o)\n    exec(code.get("s7_eagle", "def s7_eagle():\\n    raise RuntimeError(\'missing\')"), G)\n    out = G["s7_eagle"]()\n    return out.get("status"), out.get("variant"), names\n\n\ncheck("eagle_server_died_falls_through", run_eagle(["SERVER_DIED", "DONE"]),\n      ("DONE", "eagle4_fp8", ["eagle4_fp8_cutedsl", "eagle4_fp8"]))\ncheck("eagle_no_go_falls_through", run_eagle(["LAUNCH_FAILED", "NO_GO_ORACLE", "DONE"])[1], "eagle4_fp8_noprefix")\ncheck("eagle_all_fail_label", run_eagle(["LAUNCH_FAILED", "SERVER_DIED", "LAUNCH_FAILED"])[0], "NO_GO_ALL_EAGLE_VARIANTS")\ncheck("eagle_cutedsl_ok_first", run_eagle([None])[1], "eagle4_fp8_cutedsl")\n\n# ---- Merlin checkpoint full sha (#14): a good manifest passes, one changed byte fails\nif "_merlin_sha" not in code:\n    for node in ast.walk(tree):\n        if isinstance(node, ast.FunctionDef) and node.name == "_merlin_sha":\n            code["_merlin_sha"] = ast.get_source_segment(src, node)\n            break\nmdir = Path(NB).parent / "gate_merlin"\nmdir.mkdir(exist_ok=True)\n(mdir / "a.safetensors").write_bytes(b"abc" * 1000)\n(mdir / "q27_e23_p1_manifest.json").write_text(json.dumps({"files": {"a.safetensors": [3000, hashlib.sha256(b"abc" * 1000).hexdigest()]}}))\nG.update(MERLIN=mdir, MERLIN_SHA={"th": None, "ok": None, "bad": []}, save=lambda: None)\nG["R"]["inputs"] = {}\nexec(code.get("_merlin_sha", "def _merlin_sha():\\n    raise RuntimeError(\'missing\')"), G)\nG["_merlin_sha"]()\ncheck("merlin_sha_good", G["MERLIN_SHA"]["ok"], True)\n(mdir / "a.safetensors").write_bytes(b"abd" + b"abc" * 999)\nG["_merlin_sha"]()\ncheck("merlin_sha_changed_byte", (G["MERLIN_SHA"]["ok"], G["MERLIN_SHA"]["bad"]), (False, ["a.safetensors"]))\n\n# ---- static checks on the generated code (#4, #5, #8, #9)\ncheck("pp4_eager_first", src.index(\'"e_pp4_int4_eager"\') < src.index(\'"f_pp4_int4_compiled"\'), True)\ncheck("pp4_vs_int4_mtp0", \'ref = ROWS["int4_mtp0"]\' in src and \'refs=("int4_mtp3",)\' not in src, True)\ncheck("int4_ref_in_smoke_and_full", \'step("4_sm89_fp8_gemm", "int4_mtp0_target_only_ref", 12, s4_int4_ref)\' in src\n      and src.index(\'"int4_mtp0_target_only_ref", 12\') < src.index(\'"int4_mtp3_s20_control", 35\'), True)\ncheck("gpu_preflight_raises", \'raise RuntimeError("GPU_PREFLIGHT_FAILED\' in src, True)\ncheck("kit_hash_raises", \'raise RuntimeError("KERNEL_KIT_HASH_FAILED\' in src, True)\ncheck("p4_weights_real", \'"--weights", "real"\' in src, True)\ncheck("eagle_cutedsl_first", src.index(\'"eagle4_fp8_cutedsl"\') < src.index(\'("eagle4_fp8", [], True)\'), True)\n# rev 3h (three GPT SMOKE audits)\ncheck("pp4_after_aa_fill", src.index(\'step("8_aa_repeats", n, e, fn)\') < src.rindex("    pp4()"), True)\ncheck("merlin_xhigh_template", \'"--default-chat-template-kwargs", json.dumps({"enable_thinking": True, "reasoning_effort": "xhigh"})\'\n      in src and \'"--chat-template", str(CHAT_TEMPLATE_FILE)\' in src, True)\ncheck("prompts_production_template_xhigh", \'reasoning_effort=effort)\' in src and \'_render("xhigh")\' in src\n      and \'assert "Reasoning effort is set to xhigh" in _head_txt\' in src, True)\ncheck("merlin_tp4_long_levels", \'levels(port, "90k", [] if SM else [8, 12, 16, 20]\' in src\n      and \'levels(port, "120k", [8] if SM else [8, 12, 16]\' in src and \'levels(port, "40k", [8] if SM else [8, 16, 20, 24]\' in src, True)\ncheck("target_runs_fp8_calibration_and_gate", \'step("5_nvfp4_kv", "tf_fp8kv_vs_bf16", TE(14)\' in src\n      and \'"nvfp4_gate_verdict"\' in src, True)\n\nRES["all_ok"] = not BAD and not missing\nRES["failed"] = BAD\nprint("GATES " + json.dumps(RES, default=str))\nsys.exit(0 if RES["all_ok"] else 1)\n')
REP = {"checks": {}, "runs": {}, "part": "p3"}
KP = "p3"                      # p1 unit + TARGET_A | p2 TARGET_B + rerun | p3 FULL | p4 Merlin GCC 13 | p5 Merlin GCC 11
DO_UNIT = KP == "p1"
SMOKES = {"p1": ("TARGET_A",), "p2": ("TARGET_B",)}.get(KP, ())
DO_RERUN = KP == "p2"
DO_FULL = KP == "p3"
DO_MERLIN = KP in ("p4", "p5")
FI_TAGS = {"p4": ("gcc13",), "p5": ("gcc11",)}.get(KP, ())


def walk(d, path=""):
    """Every measurement window in a results tree (a dict with status and tok_s)."""
    if isinstance(d, dict):
        if "status" in d and "tok_s" in d:
            yield path, d["status"]
        for k, v in d.items():
            yield from walk(v, f"{path}/{k}")
    elif isinstance(d, list):
        for i, v in enumerate(d):
            yield from walk(v, f"{path}[{i}]")


def run(name, code_or_cmd, env, timeout, is_code=True):
    t = time.time()
    if is_code:
        (T / f"{name}.py").write_text(code_or_cmd)
        cmd = [sys.executable, str(T / f"{name}.py")]
    else:
        cmd = code_or_cmd
    with open(T / f"{name}.log", "w") as f:
        try:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, timeout=timeout, env={**os.environ, **env}, cwd=str(T))
            rc = r.returncode
        except subprocess.TimeoutExpired:
            rc = -9
    txt = (T / f"{name}.log").read_text(errors="replace")
    REP["runs"][name] = {"rc": rc, "sec": round(time.time() - t, 1), "tracebacks": txt.count("Traceback"), "tail": txt[-1500:]}
    print(f"[{time.time() - T0:7.1f}s] {name}: rc {rc} tracebacks {txt.count('Traceback')}", flush=True)
    return rc, txt


# 1. lab unit tests on Linux (port reuse right after a kill: the TIME_WAIT case; compare(); steady; capacity; replicas)
UNIT = r"""
import sys, json, time, os
sys.path.insert(0, "/tmp/e23kit")
import e23_lab as L
FAKE = "/tmp/e23kit/fake_vllm_server.py"
env = dict(os.environ, FAKE_SPEED="300")
def srv(name, port, seqs=8):
    return L.Server(name, [sys.executable, FAKE, "fp8-model", "--port", str(port), "--max-num-seqs", str(seqs), "--tensor-parallel-size", "1",
                           "--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": 3})], env, port, "/tmp/e23kit").start()
out = {}
a = srv("u1", 18121); out["ready1"] = a.wait_ready(60)[0]
P = [[100 + i] * 50 for i in range(8)]
g = L.greedy(18121, P[:2], n_tok=20); out["compare_same"] = L.compare(g, L.greedy(18121, P[:2], n_tok=20))["status"]
bad = [dict(r, tokens=r["tokens"][:5] + [999] + r["tokens"][6:]) for r in g]
out["compare_fail"] = L.compare(g, bad)["status"]
s = L.steady(18121, P, 4, secs=10, max_new=200000); out["steady"] = s["status"]
a.kill(); t = time.time()
b = srv("u2", 18121); out["port_reuse_after_kill_ready"] = b.wait_ready(60)[0]; out["port_reuse_sec"] = round(time.time() - t, 1)
c = srv("u3", 18122, seqs=2); c.wait_ready(60)
out["capacity"] = L.steady(18122, P, 4, secs=10, max_new=200000)["status"]
m = L.steady_multi([(18121, P, 3), (18122, P, 2)], secs=10, max_new=200000); out["multi"] = m["status"]
out["rss_mb"] = b.rss_mb()
b.kill(); c.kill()
out["killed_dead"] = (not b.alive()) and (not c.alive())
print("UNIT " + json.dumps(out))
"""
rc, txt = run("unit", UNIT, {}, 900) if DO_UNIT else (0, "")
if DO_UNIT:                       # rev 3d: the real gate functions of the FULL notebook against injected faults
    (T / "nb_gates_FULL.py").write_text(NB["FULL"])
    # rev 3h: the top-level kit file next to the manifest (the dataset also unpacks an older copy under p5_src/)
    _kb = next(Path("/kaggle/input/datasets/foysalemonshanto/q27-e23-sm89-kernels").rglob("q27_e23_kernels_manifest.json")).parent         / "bench_host_ar.py"
    _rc, _txt = run("gates", [sys.executable, str(T / "gate_tests.py"), str(T / "nb_gates_FULL.py"), str(T / "e23_lab.py"),
                              str(_kb)], {}, 600, is_code=False)
    _m = re.search(r"^GATES (.*)$", _txt, re.M)
    REP["checks"]["gates"] = json.loads(_m.group(1)) if _m else {"all_ok": False, "tail": _txt[-1500:]}
m = re.search(r"^UNIT (.*)$", txt, re.M)
REP["checks"]["unit"] = u = json.loads(m.group(1)) if m else None
REP["checks"]["unit_ok"] = bool(u) and u.get("compare_same") == "EXACT" and u.get("compare_fail") == "FAIL_HIGH_MARGIN" and \
    u.get("steady") == "STEADY" and u.get("port_reuse_after_kill_ready") and u.get("capacity") == "CAPACITY_LIMITED" and \
    u.get("multi") == "AGGREGATE_STEADY" and u.get("killed_dead") and isinstance(u.get("rss_mb"), float)

# 2. TARGET A or TARGET B (ports 8100-8103 vs 8200 / 8300), real mounts, fake server
FENV = {"E23_FAKE_SERVER": str(T / "fake_vllm_server.py"), "FAKE_SPEED": "300"}
procs = {}
for v in SMOKES:
    (T / f"nb_{v}.py").write_text(NB[v])
    os.makedirs(f"/tmp/e23kit/tmp_{v}", exist_ok=True)
    lf = open(T / f"nb_{v}.log", "w")
    procs[v] = (subprocess.Popen([sys.executable, str(T / f"nb_{v}.py")], stdout=lf, stderr=subprocess.STDOUT, cwd=str(T),
                                 env={**os.environ, **FENV, "E23_OUT": f"/tmp/e23kit/out_{v}", "E23_TMP": f"/tmp/e23kit/tmp_{v}"}), lf)
for v, (p, lf) in procs.items():
    try:
        rc = p.wait(timeout=3600)
    except subprocess.TimeoutExpired:
        p.kill(); rc = -9
    lf.close()
    txt = (T / f"nb_{v}.log").read_text(errors="replace")
    REP["runs"][v] = {"rc": rc, "tracebacks": txt.count("Traceback"), "tail": txt[-2500:]}
    rj = Path(f"/tmp/e23kit/out_{v}/e23_results.json")
    R = json.loads(rj.read_text()) if rj.is_file() else {}
    REP["checks"][v] = {"inputs": R.get("inputs"), "identity": R.get("identity"), "prompts": R.get("prompts"),
                        "steps": {f"{s}/{n}": r.get("status") for s, st in (R.get("sections") or {}).items() for n, r in st.items()
                                  if isinstance(r, dict)},
                        "windows": dict(walk(R.get("sections") or {})), "errors": R.get("errors"),
                        "report_md": R.get("report_md")}
    print(v, "rc", rc, "tracebacks", txt.count("Traceback"), flush=True)

# 3. competition RERUN: fallback submission only, no exception, no server
rc, txt = run("rerun_FULL", NB["FULL"], {"KAGGLE_IS_COMPETITION_RERUN": "1", "E23_OUT": "/tmp/e23kit/out_rerun"}, 600) \
    if DO_RERUN else (0, "")
sub = Path("/kaggle/working/submission.json")
ok_sub = False
if sub.is_file():
    s = json.loads(sub.read_text())
    tasks = json.load(open(next(Path("/kaggle/input").rglob("arc-agi_test_challenges.json"))))
    ok_sub = set(s) == set(tasks) and all(len(s[k]) == len(tasks[k]["test"]) and all(set(a) == {"attempt_1", "attempt_2"} for a in s[k]) for k in s)
    sub.unlink()                  # a CPU kit leaves no submission behind
REP["checks"]["rerun"] = {"rc": rc, "tracebacks": txt.count("Traceback"), "submission_format_ok": ok_sub}

# 4. FULL with a 30-minute budget: order and budget guard on Linux (the 90-minute run is the local harness)
os.makedirs("/tmp/e23kit/tmp_FULL", exist_ok=True)
rc, txt = run("nb_FULL", NB["FULL"], {**FENV, "E23_OUT": "/tmp/e23kit/out_FULL", "E23_TMP": "/tmp/e23kit/tmp_FULL",
                                      "E23_BUDGET_SEC": "1800"}, 3000) if DO_FULL else (0, "")
rj = Path("/tmp/e23kit/out_FULL/e23_results.json")
R = json.loads(rj.read_text()) if rj.is_file() else {}
REP["checks"]["FULL"] = {"order": R.get("order"), "steps": {f"{s}/{n}": r.get("status") for s, st in (R.get("sections") or {}).items()
                                                             for n, r in st.items() if isinstance(r, dict)}, "minutes": R.get("minutes")}

# 5. Merlin, for real (rev 3c: fake mode skips it): the offline install on this image, the serve-time compiler probe
# (GCC 13 from the payload vs the system GCC 11: C++20 CUDA compile + dlopen, and a FlashInfer JIT build inside the
# Merlin venv, which is Merlin's first-start compile path), and launch.build() with the notebook's arguments
MP = {}
try:
    if not DO_MERLIN:
        raise RuntimeError("SKIP_PART")
    pay = next(Path("/kaggle/input/models/foysalemonshanto/e23-merlin-env").rglob("merlin_offline_tar.bin"))
    t = time.time()
    shutil.rmtree("/tmp/mpay", ignore_errors=True)
    os.makedirs("/tmp/mpay")
    with tarfile.open(pay) as tf_:
        tf_.extractall("/tmp/mpay", filter="data")
    MP["extract_sec"] = round(time.time() - t)
    rc, txt = run("merlin_install", ["bash", "/tmp/mpay/merlin_offline/install_merlin_offline.sh", "/tmp/mpay/merlin_offline",
                                     "/tmp/merlin_off"], {}, 5400, is_code=False)
    kv = dict(re.findall(r"^(MERLIN_[A-Z]+)=(.*)$", txt, re.M))
    MP.update(install_rc=rc, install_ok="INSTALL OK" in txt, kv=kv)
    shutil.rmtree("/tmp/mpay", ignore_errors=True)
    venv = kv["MERLIN_VENV"]
    py = venv + "/bin/python"
    site = subprocess.run([py, "-c", "import sysconfig;print(sysconfig.get_paths()['purelib'])"], capture_output=True,
                          text=True).stdout.strip()
    ch = site + "/nvidia/cu13"
    nvcc = ch + "/bin/nvcc"
    lib13 = Path(kv["MERLIN_CC"]).parents[1] / "lib"
    MP["nvcc"] = subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout[-160:]
    MP["lib13_libstdcxx"] = sorted(p.name for p in lib13.glob("libstdc++.so*"))
    (T / "p20.cu").write_text('#include <concepts>\ntemplate <std::integral I> __global__ void k(I* x) { x[0] += 1; }\n'
                              'template __global__ void k<int>(int*);\nextern "C" int probe() { return 20; }\n')
    for tag, cxx, ldp in (("gcc13", kv["MERLIN_CXX"], str(lib13)), ("gcc11", "/usr/bin/g++", "")):
        so = T / f"p20_{tag}.so"
        r = subprocess.run([nvcc, "-std=c++20", "-arch=sm_89", "-shared", "-Xcompiler", "-fPIC", "-ccbin", cxx, str(T / "p20.cu"),
                            "-o", str(so)], capture_output=True, text=True, timeout=600)
        ld = subprocess.run([sys.executable, "-c", f"import ctypes;print(ctypes.CDLL('{so}').probe())"], capture_output=True,
                            text=True, env={**os.environ, "LD_LIBRARY_PATH": ldp}) if r.returncode == 0 else None
        MP[f"cxx20_{tag}"] = {"rc": r.returncode, "err": (r.stderr or "")[-600:], "dlopen": ld.stdout.strip() if ld else None,
                              "dlopen_err": ld.stderr[-300:] if ld else None}
    (T / "fi_probe.py").write_text(
        "import inspect, time, torch\n"
        "from flashinfer.jit import gen_single_decode_module as g\n"
        "sig = inspect.signature(g)\n"
        "kw = dict(dtype_q=torch.float16, dtype_kv=torch.float16, dtype_o=torch.float16, head_dim_qk=256, head_dim_vo=256,\n"
        "          head_dim=256, pos_encoding_mode=0, use_sliding_window=False, use_logits_soft_cap=False)\n"
        "print('SIG', list(sig.parameters), flush=True)\n"
        "spec = g(**{k: v for k, v in kw.items() if k in sig.parameters})\n"
        "t = time.time(); spec.build(); print('FI_BUILD_OK', round(time.time() - t), flush=True)\n")
    for tag, cc, cxx, ldp in (("gcc13", kv["MERLIN_CC"], kv["MERLIN_CXX"], str(lib13)), ("gcc11", "/usr/bin/gcc", "/usr/bin/g++", "")):
        if tag not in FI_TAGS:
            continue
        env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
        env.update(CC=cc, CXX=cxx, CUDAHOSTCXX=cxx, CUDA_HOME=ch, PATH=f"{venv}/bin:{ch}/bin:" + env.get("PATH", ""),
                   FLASHINFER_CUDA_ARCH_LIST="8.9", FLASHINFER_WORKSPACE_BASE=f"/tmp/fi_{tag}", LD_PRELOAD=kv.get("MERLIN_SHIM", ""),
                   LIBRARY_PATH="/usr/local/cuda-12.8/lib64/stubs", MAX_JOBS="4")
        if ldp:
            env["LD_LIBRARY_PATH"] = ldp
        try:
            r = subprocess.run([py, str(T / "fi_probe.py")], capture_output=True, text=True, timeout=2400, env=env)
            o_ = r.stdout + r.stderr
            MP[f"flashinfer_jit_{tag}"] = {"rc": r.returncode, "ok": "FI_BUILD_OK" in o_, "tail": o_[-1500:]}
        except subprocess.TimeoutExpired:
            MP[f"flashinfer_jit_{tag}"] = {"rc": -9, "ok": False, "tail": "timeout 2400 s"}
    sys.path.insert(0, kv["MERLIN_TREE"] + "/tools/merlin")
    import launch as ML
    ck = next(Path("/kaggle/input/models/foysalemonshanto/qwen3-8-27b-w4a16-merlin").rglob("q27_e23_p1_manifest.json")).parent
    plan = ML.build("27b", "27b-tp1-sm89-24g", str(ck), {**os.environ, "GPUS": "0", "PORT": "8100", "KV_BYTES": "8055029760",
                                                          "P2P": "off", "VENV": venv},
                    extra=["--served-model-name", "q27", "--max-model-len", "131072", "--port", "8100", "--max-num-seqs", "4"],
                    checkpoint="27b", status="untested")
    MP["launch_argv_tail"] = plan.argv()[-30:]
    dp = getattr(ML, "distributed_ports", None)
    MP["distributed_ports_8101"] = repr(dp(8101)) if dp else None
except Exception as e:
    MP["error"] = f"{type(e).__name__}: {e}"[:800]
    MP["tb"] = traceback.format_exc()[-1500:]
REP["checks"]["merlin_real"] = MP
print("MERLIN", json.dumps({k: v for k, v in MP.items() if k != "kv"}, default=str)[:4000], flush=True)

# verdict: no tracebacks in any notebook run, inputs found on the real mounts, unit tests, rerun format, no step that
# errored (CPU-expected: P4 / P5 benches without CUDA, PP4 NO_GO and EAGLE SKIPPED on the fake server), the real
# Merlin install, and a GCC 13 C++20 CUDA build that loads
inp = (REP["checks"].get("TARGET_A") or {}).get("inputs") or {}
BAD = ("ERROR", "BODY_ERROR", "SERVER_DIED", "FAIL_GPU_RELEASE", "LAUNCH_FAILED", "PATCH_FAILED", "NO_GO_GPU_FAULT", "FAIL_CHECKPOINT_SHA")
REP["checks"]["bad_steps"] = {v: {k: st for k, st in ((REP["checks"].get(v) or {}).get("steps") or {}).items() if st in BAD}
                              for v in SMOKES}
inp = inp or (REP["checks"].get("TARGET_B") or {}).get("inputs") or {}
ok = []                                                      # per-part verdict: only what this part ran
if DO_UNIT:
    ok.append(bool(REP["checks"]["unit_ok"]))
    ok.append((REP["checks"].get("gates") or {}).get("all_ok") is True)
for v in list(SMOKES) + (["rerun_FULL"] if DO_RERUN else []) + (["nb_FULL"] if DO_FULL else []):
    ok.append(REP["runs"].get(v, {}).get("rc") == 0 and REP["runs"].get(v, {}).get("tracebacks") == 0)
if SMOKES:
    ok += [bool(inp.get("merlin_files_ok")), all((inp.get("kits_sha_ok") or {}).values()), not any(REP["checks"]["bad_steps"].values()),
           all(inp.get(k) not in (None, "None") for k in ("FP8", "INT4", "MERLIN", "MERLIN_PAY", "EAGLE_PAY", "EAGLE_DRAFT", "KITS", "ARC"))]
if DO_RERUN:
    ok.append(ok_sub)
if DO_MERLIN:
    ok += [MP.get("install_ok") is True, (MP.get("cxx20_gcc13") or {}).get("dlopen") == "20"]
REP["all_ok"] = bool(ok) and all(ok)
REP["sec"] = round(time.time() - T0)
(W / "e23_cpu_kit_report.json").write_text(json.dumps(REP, indent=1, default=str))
with tarfile.open(W / "e23_cpu_kit_logs_tar.bin", "w") as tf:
    for p in sorted(T.rglob("*")):
        if p.is_file() and (p.suffix in (".log", ".json") and p.stat().st_size < 50e6):
            tf.add(p, str(p.relative_to(T)), recursive=False)
print(json.dumps({k: REP["checks"].get(k) for k in ("unit", "unit_ok", "rerun")}, default=str)[:3000])
print("inputs:", json.dumps(inp)[:1500])
print("ALL OK" if REP["all_ok"] else "NOT OK", "|", REP["sec"], "s", flush=True)
